In [ ]:
#| export
#|default_exp paper2solveit
from dialoghelper import *
from IPython.display import Markdown


## Reading Academic Papers in SolveIt via Marker OCR

SolveIt is a marvelous environment for active reading, especially of dense texts like academic papers in which working with code *while reading* accelerates comprehension. 
Here's a utility to automate the transformation of an academic paper (e.g. an Arxiv URL) into a series of cells of SolveIt, with nice handling of references and inline citations.

To use, just make sure `DATALAB_KEY` is in your env. You can get $5 in credit by signing up at datalab.to. Each paper conversion costs about 10 cents.

This builds off of Jeremy Howard's async wrapper around the Marker API and `dienhoa`'s convenience script for converting a markdown book into a collection of solveit dialogues.

In [ ]:
# import_gist('f3bcfa015660b031d0b92c2391d21921')

In [ ]:
import_gist??


```python
def import_gist(
    gist_id:str, # user/id or just id of gist to import as a module
    mod_name:str=None, # module name to create (taken from gist filename if not passed)
    add_global:bool=True, # add module to caller's globals?
    import_wildcard:bool=False, # import all exported symbols to caller's globals
    create_msg:bool=False # Add a message that lists usable tools
):
    "Import gist directly from string without saving to disk"
    fil = gist_file(gist_id)
    mod_name = mod_name or Path(fil['filename']).stem
    module = import_string(fil['content'], mod_name)
    glbs = currentframe().f_back.f_globals
    if add_global: glbs[mod_name] = module
    syms = getattr(module, '__all__', None)
    if syms is None: syms = [o for o in dir(module) if not o.startswith('_')]
    syms = [getattr(module, nm) for nm in syms]
    if import_wildcard:
        for sym in syms: glbs[sym.__name__] = sym
    if create_msg:
        pref = getattr(module, '__doc__', "Tools added to dialog:")
        asyncio.ensure_future(add_msg(f"{pref}\n\n{mk_toollist(syms)}"))
    return module
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

In [ ]:
#| export
# Jeremy Howard's pythonic wrapper around the Datalab Marker OCR API, from https://gist.github.com/jph00/f3bcfa015660b031d0b92c2391d21921 - captured inline for ease of reproducibility
from fastcore.utils import *
from IPython.display import Markdown
from httpx import get as xget,post as xpost
import time, asyncio, httpx
from base64 import b64decode
from fastcore.meta import use_kwargs_dict,delegates
dlab_params = dict(output_format='markdown', force_ocr=False, format_lines=False, paginate=False, use_llm=False,
    strip_existing_ocr=False, disable_image_extraction=False, max_pages=None, page_range=None)
dlab_url="https://www.datalab.to/api/v1/marker"

@use_kwargs_dict(**dlab_params)
async def submit_marker(fname=None, file=None, file_url=None, key=None, url=dlab_url, **kwargs):
    "Submit PDF to Datalab Marker API for conversion"
    key = key or os.environ.get("DATALAB_KEY")
    if fname: file = open(fname, "rb")
    try:
        files = {'file': (file.name, file, 'application/pdf')} if file else None
        if file_url: kwargs['file_url'] = file_url
        async with httpx.AsyncClient() as c:
            res = await c.post(url, files=files, data=kwargs, headers={"X-Api-Key": key})
            return res.json()
    finally:
        if fname and file: file.close()
@delegates(submit_marker)
async def submit_markers(files=None, fnames=None, file_urls=None, **kwargs):
    "Submit multiple PDFs concurrently, return list of response dicts"
    tasks = [submit_marker(file=f, **kwargs) for f in L(files)
        ] + [submit_marker(file_url=u, **kwargs) for u in L(file_urls)
        ] + [submit_marker(fname=u, **kwargs) for u in L(fnames)]
    return await asyncio.gather(*tasks)
async def poll_marker(d, key=None, max_polls=300, delay=2, verbose=False):
    "Poll Marker API until conversion complete"
    if not d.get('success', True): raise ValueError(f"Submit failed: {d.get('error')}")
    check_url = d['request_check_url']
    key = key or os.environ.get("DATALAB_KEY")
    async with httpx.AsyncClient() as c:
        for _ in range(max_polls):
            res = await c.get(check_url, headers={"X-Api-Key": key})
            data = res.json()
            if verbose: print(data["status"], end='; ')
            if data["status"] == "complete": return data
            if data["status"] == "failed": raise RuntimeError(f"Conversion failed: {data.get('error')}")
            await asyncio.sleep(delay)
    raise TimeoutError(f"Polling timed out after {max_polls * delay}s")
async def poll_markers(ds, key=None, max_polls=300, delay=2, verbose=False):
    "Poll multiple Marker API requests concurrently"
    return await asyncio.gather(*[poll_marker(d, key, max_polls, delay, verbose) for d in ds])
@delegates(submit_marker)
async def convert_pdf(fname=None, file=None, file_url=None, key=None, max_polls=300, delay=2, verbose=False, **kwargs):
    "Submit and poll until complete, return result"
    r = await submit_marker(fname=fname, file=file, file_url=file_url, key=key, **kwargs)
    return await poll_marker(r, key=key, max_polls=max_polls, delay=delay, verbose=verbose)
@delegates(submit_markers)
async def convert_pdfs(files=None, fnames=None, file_urls=None, key=None, max_polls=300, delay=2, verbose=False, **kwargs):
    "Submit multiple PDFs and poll all until complete"
    rs = await submit_markers(files=files, fnames=fnames, file_urls=file_urls, key=key, **kwargs)
    return await poll_markers(rs, key=key, max_polls=max_polls, delay=delay, verbose=verbose)
def _save_md(r, stem, path):
    path = Path(path)
    figures = path / "figures"
    figures.mkdir(exist_ok=True, parents=True)

    markdown = r["markdown"]
    for name, data in r["images"].items():
        (figures / name).write_bytes(b64decode(data))
        markdown = markdown.replace(f"]({name})", f"](figures/{name})")

    r["markdown"] = markdown
    (path / f"{stem}.md").write_text(markdown)

@delegates(convert_pdf)
async def pdf2md(fname, path='.', **kwargs):
    "Convert PDF to markdown and save with images"
    path = Path(path)
    path.mkdir(exist_ok=True, parents=True)
    r = await convert_pdf(fname=fname, **kwargs)
    _save_md(r, Path(fname).stem, path)
    return r

@delegates(convert_pdfs)
async def pdfs2md(fnames, path='.', **kwargs):
    "Convert multiple PDFs to markdown and save with images"
    path = Path(path)
    path.mkdir(exist_ok=True, parents=True)
    rs = await convert_pdfs(fnames=fnames, **kwargs)
    for fname,r in zip(fnames, rs): _save_md(r, Path(fname).stem, path)
    return rs

In [ ]:
pdfs2md?

```python
def pdfs2md(
    fnames, path:str='.', files:NoneType=None, file_urls:NoneType=None, key:NoneType=None, max_polls:int=300,
    delay:int=2, verbose:bool=False, fname:NoneType=None, file:NoneType=None, file_url:NoneType=None,
    url:str='https://www.datalab.to/api/v1/marker', output_format:str='markdown', force_ocr:bool=False,
    format_lines:bool=False, paginate:bool=False, use_llm:bool=False, strip_existing_ocr:bool=False,
    disable_image_extraction:bool=False, max_pages:NoneType=None, page_range:NoneType=None
):

```



```
Convert multiple PDFs to markdown and save with images
```



**File:** `/tmp/ipykernel_418/2174028609.py`

**Type:** function

In [ ]:
#| export
import httpx
from pathlib import Path
from tempfile import NamedTemporaryFile
def download_to_temp(url):
    if 'arxiv.org/abs/' in url: url = url.replace('/abs/', '/pdf/') + '.pdf'
    tmp = NamedTemporaryFile(delete=False, suffix=Path(url).suffix)
    content = httpx.get(url, follow_redirects=True).content
    tmp.write(content)
    tmp.flush()
    tmp.close()
    return Path(tmp.name)


In [ ]:
f = download_to_temp("http://arxiv.org/abs/2207.02093")

In [ ]:
f

Path('/tmp/tmpbk79v93b.pdf')

In [ ]:
o = await pdf2md(str(f))

In [ ]:
Markdown(o["markdown"])



# Predicting Out-of-Domain Generalization with Neighborhood Invariance

**Nathan Ng**

University of Toronto

Vector Institute

Massachusetts Institute of Technology

[nathanng@mit.edu](mailto:nathanng@mit.edu)

**Neha Hulkund**

Massachusetts Institute of Technology

[nhulkund@mit.edu](mailto:nhulkund@mit.edu)

**Kyungyun Cho**

New York University

President Design, Genentech

CIFAR Fellow

[kyungyun.cho@nyu.edu](mailto:kyungyun.cho@nyu.edu)

**Marzyeh Ghassemi**

Massachusetts Institute of Technology

CIFAR AI Chair

Vector Institute

[mghassem@mit.edu](mailto:mghassem@mit.edu)

Reviewed on OpenReview: <https://openreview.net/forum?id=jYkWdJzTwm>

## Abstract

Developing and deploying machine learning models safely depends on the ability to characterize and compare their abilities to generalize to new environments. Although recent work has proposed a variety of methods that can directly predict or theoretically bound the generalization capacity of a model, they rely on strong assumptions such as matching train/test distributions and access to model gradients. In order to characterize generalization when these assumptions are not satisfied, we propose neighborhood invariance, a measure of a classifier’s output invariance in a local transformation neighborhood. Specifically, we sample a set of transformations and given an input test point, calculate the invariance as the largest fraction of transformed points classified into the same class. Crucially, our measure is simple to calculate, does not depend on the test point’s true label, makes no assumptions about the data distribution or model, and can be applied even in out-of-domain (OOD) settings where existing methods cannot, requiring only selecting a set of appropriate data transformations. In experiments on robustness benchmarks in image classification, sentiment analysis, and natural language inference, we demonstrate a strong and robust correlation between our neighborhood invariance measure and actual OOD generalization on over 4,600 models evaluated on over 100 unique train/test domain pairs.

## 1 Introduction

As deep neural networks find increasing use in safety-critical domains such as autonomous driving (Gupta et al., 2021) and healthcare (Wiens et al., 2019), it is important to develop methods to understand and compare how these models generalize to new environments. Although empirically these models generalize in many settings (Hendrycks et al., 2020a; Allen-Zhu et al., 2018; Neyshabur et al., 2017a), they also exhibit numerous failure cases. For example, models have been shown to overfit to a dataset’s meta characteristics (Recht et al., 2019) or arbitrarily corrupted labels (Zhang et al., 2016), learn spurious correlations (Liang

![Figure 1: Comparison of neighborhood invariance for two classifiers. On the left, three boxes show the digit '3' rotated by 0, 30, and 60 degrees. Lines connect these boxes to two circular plots. The left plot, labeled 'Less Invariant Classifier', shows the digit '3' at the center of a circle with several overlapping, irregularly shaped regions of different colors (green, yellow, red, blue). The right plot, labeled 'More Invariant Classifier', shows the digit '3' at the center of a circle with a single, large, green region that encompasses most of the circle, with other colors only appearing at the very edges. The label 'G_r(x)' is at the bottom of each plot.](49ad3a646d84bcfeac02bdf2b3792a3e_img.jpg)

Figure 1: Comparison of neighborhood invariance for two classifiers. On the left, three boxes show the digit '3' rotated by 0, 30, and 60 degrees. Lines connect these boxes to two circular plots. The left plot, labeled 'Less Invariant Classifier', shows the digit '3' at the center of a circle with several overlapping, irregularly shaped regions of different colors (green, yellow, red, blue). The right plot, labeled 'More Invariant Classifier', shows the digit '3' at the center of a circle with a single, large, green region that encompasses most of the circle, with other colors only appearing at the very edges. The label 'G\_r(x)' is at the bottom of each plot.

Figure 1: The transformation neighborhood  $G_r(x)$  around  $x$  contains the set of points reachable from a transformation in  $G_r$  (pictured here as a rotation of  $r$  degrees). As  $r$  increases, the transformation neighborhood is partitioned into a distinct set of decision regions. More invariant classifiers (right) classify most points in the neighborhood into the same class and should generalize better compared to less invariant classifiers (left). Our measure of invariance is independent of what specific region  $x$  lies in.

& Zou, 2022), and change their predictions even with small adversarial perturbations (Goodfellow et al., 2014; Papernot et al., 2017). Many methods have been proposed to mitigate these issues, but precisely characterizing the generalization properties of a model in diverse settings remains an open problem.

One line of work aims to theoretically bound generalization capacity (Vapnik & Chervonenkis, 1971; Bartlett & Mendelson, 2003; McAllester, 1999; Neyshabur et al., 2017a; Dziugaite & Roy, 2017; Neyshabur et al., 2015b) or directly predict generalization (Keskar et al., 2016; Liang et al., 2019; Neyshabur et al., 2015a; Schiff et al., 2021; Jiang et al., 2019), and are useful in reasoning about a model beyond its performance on a specific known test set. However, these methods work only when train and test distributions are the same, and often rely on a strong set of assumptions such as access to labelled test data (Schiff et al., 2021), model weights (Neyshabur et al., 2015b; Bartlett et al., 2017; Neyshabur et al., 2017b), model gradients (Jiang et al., 2019), and training data (Keskar et al., 2016). More recent work aims to estimate the generalization of a trained model on unlabelled test data directly (Deng & Zheng, 2021; Jiang et al., 2021; Deng et al., 2021; Garg et al., 2022). However, these metrics are typically calculated based on the output logits of a model on individual examples, which can become poorly calibrated in out-of-domain (OOD) settings (Morteza & Li, 2022). In real world settings, we require a robust measure of generalization that can be applied across a wide range of test distributions and where we are often given access only to a black box model.

In this paper we propose *neighborhood invariance*, a complexity measure that correlates well with generalization and that only assumes access to a set of suitable data transformations. Given a test data point, we define the transformation neighborhood as the set of points that can be generated from a set of transformations with a given maximum magnitude. A classifier’s neighborhood invariance is then the proportion of points that are classified into the most commonly predicted class in this neighborhood. Intuitively, a classifier that is more invariant in this neighborhood should have the ability to represent examples with lower dimensionality and thus lower complexity, leading to stronger generalization. Different from other similar methods (Aithal K et al., 2021), we define invariance with respect to the neighborhood itself rather than relative to the prediction at the test point and do not require manually tuning weights, meaning our measure can be applied even when test distributions vary. In addition, since our measure makes so few assumptions it is applicable in a wide range of experimental settings and can be used to compare the generalization properties of multiple models even when labeled data is unavailable.

We investigate the correlation of a model’s neighborhood invariance with its capacity to generalize, focusing on experimental settings with OOD dataset shifts (Taori et al., 2020) where test data is sampled from a

distribution different from the training distribution. We select common OOD benchmark datasets in image classification (Krizhevsky, 2009; Lu et al., 2020; Recht et al., 2018; Deng, 2012; Darlow et al., 2018; Netzer et al., 2011; Arjovsky et al., 2019; Taori et al., 2020), sentiment analysis (Ni et al., 2019), and natural language inference (Williams et al., 2018), which totals over 100 pairs of training/test domains. We consider a large pool of over 4,600 models trained on these datasets with varying architectures and generalization properties, and sample sets of transformations commonly used for data augmentation (Ng et al., 2020; Cubuk et al., 2020; Wei & Zou, 2019; Xie et al., 2019). Across a wide set of correlation metrics, we find that neighborhood invariance measures outperform or match baselines in almost all experimental settings.

## 2 Related Work

**Characterizing Model Invariance** Ensuring various kinds of model invariance is a well studied aspect of learning generalizable models and has been analyzed extensively from a causality perspective (Bühlmann, 2018; Peters et al., 2015; Haavelmo, 1943). At the largest scale, models trained on a wide support of training data and domains have demonstrated robust zero-shot and few-shot abilities (Radford et al., 2021; Brown et al., 2020; Wortsman et al., 2022). At a smaller scale, models that are invariant across data domains or interventions (Arjovsky et al., 2019; Gulrajani & Lopez-Paz, 2020; Bühlmann, 2018) are able to learn representations that do not depend on spurious correlations. Finally, at the smallest scale, local invariance to data augmentations (Cubuk et al., 2020), local changes (Rifai et al., 2011), augmentation graphs (HaoChen et al., 2021), similar neighbors (Luo et al., 2018), or interpolation between points (Verma et al., 2019; Zhang et al., 2018) have demonstrated improvements in model generalization. In our paper, we consider model invariance at this local scale.

Recent work has shown that models that are invariant to local transformations factorize the input space into a base space and the set of transformations (Sokolić et al., 2017; Sannai et al., 2021), effectively reducing the input dimensionality and thus model complexity (Anselmi et al., 2016; Anselmi et al., 2015). Measuring this decrease in complexity can be performed by analyzing the sample cover (Zhu et al., 2021). A similar line of work derives estimation error bounds based on the intrinsic dimensionality of deep ReLU networks in Hölder (Schmidt-Hieber, 2019; Nakada & Imaizumi, 2020; Chen et al., 2019), Besov, mixed smooth Besov (Suzuki, 2018), and anisotropic Besov (Suzuki & Nitanda, 2021) function spaces.

Most similar to our work, Aithal K et al. (2021) measures a model’s robustness to perturbations as a proxy for generalization. Our method generalizes theirs and differs in a few key ways. We calculate our measure on the test set relative to a transformation neighborhood and can thus adapt to any specific domain for which we measure complexity and predict generalization. In contrast, Aithal K et al. (2021) calculate their measure on the training set, use the models’ prediction as a ground truth, and require manually tuning the weights of augmentations, meaning it is relatively brittle and can only be applied to in-domain data. In addition we analyze the correlation of our neighborhood invariance measure on a wide range of OOD benchmarks on image classification, sentiment analysis, and natural language inference, while Aithal K et al. (2021) consider only image classification tasks with matching train/test distributions.

**Measures of Complexity and Predicting Generalization** Traditional methods of analyzing the generalization bounds of neural networks use theoretical measures of complexity. VC dimension (Vapnik & Chervonenkis, 1971) and Rademacher complexity (Bartlett & Mendelson, 2003) can be used to bound the generalization of particular function classes, although they are often vacuous at the scale of deep neural networks (Dziugaite & Roy, 2017). The PAC-Bayes framework (McAllester, 1999; Neyshabur et al., 2017a; Dziugaite & Roy, 2017; Garg et al., 2021) can be used to build tighter generalization bounds by considering the “sharpness” of the local minima. Norm-based measures (Neyshabur et al., 2015b; Bartlett et al., 2017; Neyshabur et al., 2017b) bound generalization by considering different norms of the weights of learned networks. More recent analyses have focused on empirically motivated measures that do not provide theoretical bounds. These include the sharpness of minima in parameter space Keskar et al. (2016), Fisher-Rao norm Liang et al. (2019), distance from initialization (Nagarajan & Kolter, 2019), path norm (Neyshabur et al., 2015a), layer margin distributions (Jiang et al., 2019), and perturbation response curves Schiff et al. (2021).

However, these measures are only applicable when train and test distributions match. Although some generalization bounds have been derived for these OOD settings (Garg et al., 2021; Ben-David et al., 2007;

![Figure 2: A diagram illustrating the neighborhood invariance estimation process. An OOD Test Example x is sampled from its Neighborhood, which consists of transformed examples {g_i(x)}_{i=1}^N. These examples are then evaluated by two classifiers, f_1 (More Invariant Classifier) and f_2 (Less Invariant Classifier). The classifier f_1 shows a higher agreement between its outputs for the neighborhood examples and the test example x compared to f_2, indicated by the larger blue shaded area in the bar chart for f_1.](2fa4a1bf91d0f34e87c689fbc1211fe3_img.jpg)

Figure 2: A diagram illustrating the neighborhood invariance estimation process. An OOD Test Example x is sampled from its Neighborhood, which consists of transformed examples {g\_i(x)}\_{i=1}^N. These examples are then evaluated by two classifiers, f\_1 (More Invariant Classifier) and f\_2 (Less Invariant Classifier). The classifier f\_1 shows a higher agreement between its outputs for the neighborhood examples and the test example x compared to f\_2, indicated by the larger blue shaded area in the bar chart for f\_1.

Figure 2: To estimate neighborhood invariance we sample a set of transformations  $\{g_i\}_{i=1}^N \sim \mathcal{G}_r$  and generate a set of nearby examples  $\{g_i(x)\}_{i=1}^N$  for every test example  $x$ . This set of examples is then evaluated using each classifier. We expect classifiers with more points classified in the most common class to generalize better to the given test set.

Zhang et al., 2019), they rely on access to the test data distribution. In addition, many testbeds examine only synthetic shifts, whereas natural shifts such as WILDS (Koh et al., 2021) are much more difficult. In real world settings where test distributions are often unknown, a separate line of work aims to directly predict generalization from unlabelled test data. These methods either predict the correctness on individual examples (Deng & Zheng, 2021; Jiang et al., 2021; Deng et al., 2021), directly estimate the total error (Garg et al., 2022; Guillory et al., 2021; Chen\* et al., 2021; Chuang et al., 2020; Vedantam et al., 2021), or learn linear models relating ID and OOD accuracy (Miller et al., 2021) or agreement (Back et al., 2022).

## 3 Neighborhood Invariance Measure

In this section we introduce our neighborhood invariance measure. We start by defining the transformation neighborhood of a point, then motivate our formulation of invariance in this neighborhood, and finally show how to estimate it in practice.

### 3.1 Motivation

Consider a classification task from an input space  $\mathcal{X}$  to an output space  $\mathcal{Y}$  with  $k$  classes. We are given a model  $f: \mathcal{X} \to \mathcal{Y}$  trained on an in-domain training dataset  $\mathcal{D}_i = \{(x_i^1, y_i^1), \dots, (x_i^n, y_i^n)\}$  sampled from a distribution  $P_i(\mathcal{X}, \mathcal{Y})$ , and an out-of-domain test dataset  $\mathcal{D}_o = \{(x_o^1, y_o^1), \dots, (x_o^m, y_o^m)\}$  sampled from a distribution  $P_o(\mathcal{X}, \mathcal{Y})$ . We assume further that domains are covariate shifted such that  $P_o(\mathcal{Y}|\mathcal{X})$  does not change between domains

We consider a set of data transformations  $\mathcal{G}_r = \{g: \mathcal{X} \to \mathcal{X} \mid m(g) < r\}$  where  $g$  is a particular data transformation with an associated measure of the magnitude of the transformation  $m(g)$ . For example, a set of image rotation transformations with a maximum angle of 30 degrees might be denoted  $\mathcal{G}_{30}$  where  $m(g_i) = \alpha$  is the angle of rotation for a specific  $g_i$ . For a given test point  $x \in \mathcal{D}_o$ , we define the transformation neighborhood  $\mathcal{G}_r(x) = \{g(x) \in \mathcal{X} \mid g \in \mathcal{G}_r\}$  as the set of outputs after applying all transformations in  $\mathcal{G}_r$ . Defining the neighborhood this way allows us to consider a wide range of nearby points in a controllable way without needing access to the underlying data distribution. As shown in Figure 1, for a given  $r$ , we can define a neighborhood decision distribution as

$$p_r(x) = \frac{|\{f(x') = j \mid x' \in \mathcal{G}_r(x)\}|}{|\mathcal{G}_r(x)|}. \quad (1)$$

We then define our **neighborhood invariance measure** as

$$\mu(f, x) = \max_{j \in \mathcal{Y}} p_j(x). \quad (2)$$

We assume that data transformations are selected such that the label for the transformed point  $g(x)$  is still well defined. For example, flipping MNIST digits horizontally would cause most examples to have an undefined label. If the label is undefined, then  $f(g(x))$  should produce close to random outputs and thus

constant invariance values of  $1/k$  regardless of the generalization properties of  $f$ , causing our measure to fail. We empirically measure this phenomenon across a wide range of transformations in Section 5.4.

Intuitively, a classifier that is more invariant in the neighborhood of  $x$  should be able to represent it with a lower input dimensionality and thus be less complex, leading to stronger generalization capabilities. In contrast, a less invariant classifier will need a higher input dimensionality to represent the neighborhood of  $x$  and thus be more complex, leading to weaker generalization. Crucially, since our invariance is measured with respect to the neighborhood around the test point rather than the test point itself, it does not rely on the ground truth label. In addition, it makes no assumptions about the model or the distribution from which test data was sampled, making it applicable in many settings where existing complexity measures cannot be calculated, including common OOD robustness settings.

### 3.2 Estimating Neighborhood Invariance

Calculating neighborhood invariance exactly is typically intractable since evaluating all possible transformations is impossible. Instead, we perform Monte Carlo estimation by sampling a set of  $N$  transformations  $\{g_t\}_{t=1}^N$  from  $\mathcal{G}_r$  (including the identity transformation  $f(x) = x$ ) and calculating

$$\mu(f, x) = \frac{1}{N} \sum_{i=1}^{N} \mathbb{1}(f(g_i(x)) = \hat{y}(f, x)), \quad (3)$$

where

$$\hat{y}(f, x) = \arg \max_{j \in Y} \sum_{i=1}^{N} \mathbb{1}(f(g_i(x)) = j) \quad (4)$$

is the most commonly output label. The average neighborhood invariance across the entire dataset  $\mathcal{D}_o$  is then  $\frac{1}{m} \sum_{j=1}^{m} \mu(f, x_o^j)$ .

## 4 Experimental Setup

Empirically evaluating the quality of a complexity measure is difficult and requires careful experimental design. Typically, evaluation is done by generating a large pool of models with sufficiently varied generalization properties, but if we generate these models by varying only a few hyperparameters, our observed correlation may be an artifact of these factors affecting both generalization and our measure. To this end, we follow a similar experimental setup to Jiang et al. (2019).

### 4.1 Data

For our experiments we focus on three tasks: large and small scale image classification, sentiment analysis on single sentences and natural language inference on sentence pairs. For each task we construct a set of datasets sampled from different data domains.

**Image Classification** For image classification we begin by considering 7 datasets domain shifted from ImageNet (Deng et al., 2009; Russakovsky et al., 2015). These include ImageNetV2 (Recht et al., 2019) and Imagenet-Sketch (Wang et al., 2019) with the same output classes, as well as ObjectNet (Barbu et al., 2018), ImageNetVid, YTBb anchors (Gu et al., 2019; Recht et al., 2019), ImageNet-A (Hendrycks et al., 2021b), and ImageNet-R (Hendrycks et al., 2021a) with a smaller subset of output classes.

In addition to ImageNet datasets we construct two sets of smaller scale datasets. The first we call **CI10** and consists of CIFAR10 (Krizhevsky, 2009), CINIC10 (Darlow et al., 2018), CIFAR10.1 (Recht et al., 2018), and CIFAR10.2 (Lu et al., 2020). The second we call **Numbers** and consists of SVHN (Netzer et al., 2011), MNIST (Deng, 2012), and Colored MNIST (Arjovsky et al., 2019). Domains in each set share the same set of output classes.

**Sentiment Analysis (SA)** We use the datasets subsampled from the Amazon reviews dataset (Ni et al., 2019) which contains product reviews from Amazon. Following Hendrycks et al. (2020b) and Ng et al. (2020),

| Model   | Datset   | Training Domain | Batch Size | Depth | Width | Dropout | Weight Decay | Label Noise | Learning Rate | Batch Norm | Seed | Data Avg | # Converged | # Evaluations |
|---------|----------|-----------------|------------|-------|-------|---------|--------------|-------------|---------------|------------|------|----------|-------------|---------------|
| CNN     | Amazon   | 10              | 3          | 3     | 3     | 3       | 3            | —           | —             | —          | —    | —        | 2,418       | 24,180        |
|         | MNLI     | 5               | 3          | 3     | 3     | 2       | 3            | 3           | —             | —          | —    | —        | 786         | 29,800        |
|         | Amazon   | 10              | 3          | —     | —     | —       | 3            | 3           | —             | —          | —    | —        | 332         | 33,200        |
| RoBERTa | MNLI     | 5               | 3          | —     | —     | 2       | 3            | 3           | —             | —          | —    | —        | 213         | 10,650        |
| Various | ImageNet | —               | —          | —     | —     | —       | —            | —           | —             | —          | —    | —        | 401         | 2,406         |
| NIN     | SVHN     | —               | 3          | 3     | —     | 3       | 2            | —           | —             | —          | —    | —        | 54          | 162           |
|         | CIFAR10  | —               | 2          | 2     | 2     | 2       | 2            | —           | —             | —          | —    | —        | 32          | 128           |
|         | CIFAR10  | —               | 3          | 3     | —     | 3       | 2            | —           | —             | —          | —    | —        | 54          | 216           |
|         | CIFAR10  | —               | —          | —     | 3     | —       | 3            | —           | 2             | 2          | 3    | 2        | 216         | 864           |
| ResNet  | CIFAR10  | —               | 2          | 2     | 4     | —       | 2            | —           | 2             | 2          | —    | —        | 128         | 512           |
| CNN     | CINIC10  | —               | 2          | 2     | 4     | —       | 2            | —           | 2             | 2          | —    | —        | 4,644       | 112,118       |

Table 1: Number of possible hyperparameter values for each architecture and task. Fields denoted with a — indicate that this hyperparameter is fixed or not applicable. We also list the total number of models converged and evaluations run in each model pool. In total we consider 4,644 models and 112,118 evaluations.

we split the dataset into 10 different domains based on review category. For all domains and datasets, models are trained to predict a review’s star rating from 1 to 5.

**Natural Language Inference (NLI)** We use the MNLI (Williams et al., 2018) dataset, a corpus of NLI data from 10 distinct genres of written and spoken English. We train on the 5 genres with training data and evaluate on all 10 genres. Models are given two sentences, a premise and hypothesis, and predict whether the hypothesis is entailed by, is neutral to, or contradicts the premise.

### 4.2 Model and Hyperparameter Space

For large scale image classification on ImageNet, we use pretrained models from the ImageNet Testbed (Taori et al., 2020) which covers a wide range of architectures including ResNext (Xie et al., 2016), EfficientNet (Tan & Le, 2019), BiT (Beyer et al., 2021), Vision Transformers (Dosovitskiy et al., 2020), CLIP (Radford et al., 2021), and many more models. We provide a full list of models evaluated in Appendix A.2. For smaller scale image classification tasks, we use models trained for the tasks 1, 2, 4, 5, and 9 from the Predicting Generalization in Deep Learning competition (PGDL) (Jiang et al., 2020) as well as models from Jiang et al. (2019), which covers Network in Network (NIN) (Lin et al., 2013), VGG (Simonyan & Zisserman, 2015), ResNet (He et al., 2015), and CNN models trained on CIFAR10, CINIC10, and SVHN. On natural language tasks we consider CNN (Kim, 2014; Mou et al., 2016) and RoBERTa (Liu et al., 2019) based models. On natural language models we apply label noise by randomly replacing a fraction of training labels with uniform samples from the label space. We argue that label noise is not an artificial training setting as stated in Jiang et al. (2019) but rather a method of entropy regularization (Pereyra et al., 2017; Xie et al., 2016) which prevents models from becoming overconfident.

In order to control for the varying convergence rates and learning capacities of our different models, we follow Jiang et al. (2019) and early stop the training of models when they reach a given training cross entropy loss (usually around 99% training accuracy), or if they reach the max number of training epochs. We discard all models which do not converge within this time. The total number of models trained and converged in each pool as well as details on hyperparameter variations for each task and model provided in Table 1. We include further details on model training, the hyperparameter space, and specific choices in hyperparameters in Appendix A.4, A.2, and A.3.

### 4.3 Evaluation Metrics

Given a set of domains defined by distributions  $\{P_1, P_2, \dots, P_n\}$  and a set of datasets  $\{\mathcal{D}_i \sim P_i\}_{i=1}^n$  sampled from these domains, we train a set of models  $F_i = \{f_i^1, f_i^2, \dots, f_i^m\}$  on each dataset  $\mathcal{D}_i$ . We evaluate all models  $f_i^k \in F_i$  on all OOD test datasets  $\mathcal{D}_o : o \neq i$ , generating a set of invariance and generalization values  $(\mu_{io}^k, g_{io}^k)$ . We define generalization as the top-1 accuracy of  $f_i^k$  on  $\mathcal{D}_o$ .

We evaluate our measure first by predicting the generalization of a given model to an OOD test set. Specifically, we select an OOD test set  $\mathcal{D}_o$  and an in-domain training set  $\mathcal{D}_i : i \neq o$  and predict the OOD generalization  $g_{io}^k$

of a model  $f_i^k$  trained on  $\mathcal{D}_i$  and evaluated on  $\mathcal{D}_o$  from its invariance value  $\mu_{io}^k$ . To generate these predictions we use a linear model  $\hat{g} = a\mu + b$  with parameters  $a, b \in \mathbb{R}$ . To estimate our parameters  $a$  and  $b$ , we select a pool of models  $\{f_j^k \in F_j : j \neq i, o\}$  that are trained on *all remaining datasets*. Each model  $f_j^k$  in this pool is evaluated on the OOD dataset  $\mathcal{D}_o$  to give us a set of pairs  $\{(\mu_{jo}^k, g_{jo}^k)\}$ . We then find  $a, b$  by minimizing the mean squared error  $(a^*, b^*) = \arg \min_{a,b} \sum_{j,k} (a\mu_{jo}^k + b - g_{jo}^k)^2$  on all models in the pool.

We use the learned parameters to make generalization predictions  $\hat{g}_{io}^k = a\mu_{io}^k + b$  for every model  $f_i^k \in F_i$  on  $\mathcal{D}_o$  and measure the coefficient of determination  $R^2$  (Glantz et al., 1990). We also measure the residuals of our linear model by calculating the mean absolute error (MAE) between our predictions and the actual generalization. For every pair of training domain  $i$  and OOD test domain  $o$ , we evaluate  $R^2$  and MAE then average each metric across all pairs. We report MAE values as percentage points.

We also consider the rank correlation between neighborhood invariance and actual generalization. Specifically, for a pair of models  $f_i, f_j$  with measure and generalization pairs  $(\mu_i, g_i)$  and  $(\mu_j, g_j)$ , we want  $g_i > g_j$  if  $\mu_i > \mu_j$ . We use Kendall’s rank  $\tau$  coefficient (Kendall, 1938) to measure how consistent these sets of rankings are. We measure four different  $\tau$  values:

**ID  $\tau$**  This metric evaluates the correlation of our measure with in-domain generalization. We select a training dataset  $\mathcal{D}_i$  and consider pairs  $\{(\mu_{ii}^k, g_{ii}^k)\}$  generated from the set of models  $F_i$  trained on  $\mathcal{D}_i$ .  $\tau$  values are averaged across all training domains.

**Macro  $\tau$**  This metric evaluates the correlation of our measure individually on each training/OOD test domain pair. We select a training dataset  $\mathcal{D}_i$  and a OOD test dataset  $\mathcal{D}_o$  and consider pairs  $\{(\mu_{io}^k, g_{io}^k)\}$  generated from the set of models  $F_i$  trained on  $\mathcal{D}_i$ .  $\tau$  values are averaged across all pairs of training and OOD test domains.

**Micro  $\tau$**  This metric evaluates the correlation of our measure on a given OOD test domain across models trained on all other domains. We select a single OOD test domain  $\mathcal{D}_o$  and consider pairs  $\{(\mu_{io}^k, g_{io}^k)\}$  generated from the set of models  $\{f_i^k \in F_i : i \neq o\}$  trained on all other datasets  $\{\mathcal{D}_i : i \neq o\}$ .  $\tau$  values are averaged across all test domains. We use this metric only when different models are trained on different training sets.

**Arch  $\tau$**  This metric evaluates the correlation of our measure on models trained with different architectures. Arch  $\tau$  is calculated similar to Micro  $\tau$ , except  $F_i$  now includes models from all architectures.  $\tau$  values are averaged across all test domains.

### 4.4 Data Transformations

Defining the transformation neighborhood requires defining a set of data transformations with associated magnitudes. For image classification, we consider four transformations: RandAugment (Cubuk et al., 2020) which randomly combines various transformations, random translation in the X- and Y-axes, random patch erasing (Zhong et al., 2020) which removes randomly sized patches from the image, and horizontal flips and crops. We call neighborhood invariance measures based on these transformations **NI-RandAug**, **NI-Translate**, **NI-Erase**, and **NI-FC** respectively. For natural language tasks, we also consider four transformations: SSMBA (Ng et al., 2020) which generates examples in a manifold neighborhood using a denoising autoencoder, EDA (Wei & Zou, 2019) which applies random word level operations, backtranslation (BT) (Rico Sennrich, 2016; Xie et al., 2019) which translates back and forth from a pivot language, and a transformation that randomly replaces a percentage of tokens. We call neighborhood invariance measures based on these transformations **NI-SSMBA**, **NI-EDA**, **NI-BT**, and **NI-RandRep** respectively. For all experiments we sample  $n = 10$  transformations in addition to the identity transformation, although ablations in section 5.4 show that our method is relatively robust to the specific number of transformations sampled. We provide further details on specific transformation magnitude values and implementations for all methods in Appendix A.5.

### 4.5 Baselines

Since our experimental setting makes so few assumptions, there are very few complexity measures that we can compare against. This includes Aithal K et al. (2021), which requires matching train/test distributions.

We thus consider complexity measures that require only model weights, specifically the **Spectral** (Yoshida & Miyato, 2017; Neyshabur et al., 2017b) and **Frobenius** (Neyshabur et al., 2015b) norms. However, in our experiments we find close to 0 or negative correlation for these measures, so we do not report their performance. We also compare our method against output based methods that directly predict OOD generalization. We use **ATC-MC** and **ATC-NE** (Garg et al. (2022) as our two baselines, which calculate a threshold on in-domain validation data based on max confidence and negative entropy scores respectively. To calculate metrics on these methods we treat the generated accuracy predictions as a score. To calculate ID  $\tau$  values we select a threshold value based on validation data then calculate predicted accuracy values on test data from the same domain. For ImageNet domain shift datasets where the output classes are a subset of the original 1,000 ImageNet classes, we do not recompute a subclassed ATC threshold as we do not assume prior knowledge of the OOD output classes.

## 5 Results

We now present the results of our experiments evaluating the quality of our neighborhood invariance measure. We begin by analyzing the effect of dataset distance on the correlation of neighborhood invariance with generalization in a toy setting. Our main set of results evaluate neighborhood invariance on OOD benchmarks in image classification, sentiment analysis, and natural language inference. Finally, we examine the correlation of our measure in extreme OOD settings and analyze the factors that affect the quality of our neighborhood invariance estimates.

### 5.1 Dataset Distance: Toy Analysis

In general, as with any complexity measure or generalization predictor, we expect neighborhood invariance to perform more poorly as we move farther from the training domain. In the worst case, if a classifier becomes a degenerate constant classifier in a far enough OOD domain, then neighborhood invariance reaches a constant maximum value of 1 while generalization becomes random. In order for our measure to work well, we assume that test domains are sufficiently close to training domains so that model predictions are non-constant. In this section we present an analysis of the effect of dataset distance on the quality of our measure in a toy setting.

We consider a binary classification task of points inside and outside a unit hypersphere in  $\mathcal{X} = \mathbb{R}^n$ , as shown in Figure 2. We define different data domains as univariate gaussian distributions  $P(\mathcal{X})$ , centered at a point  $\mu$  on the hypersphere. The distance between two datasets  $\mathcal{D}_1 \sim P_1(\mathcal{X})$  and  $\mathcal{D}_2 \sim P_2(\mathcal{X})$  can then be measured as the distance along the hypersphere between  $\mu_1$  and  $\mu_2$  in radians. We generate a training dataset  $\mathcal{D}_0$  by sampling points around the north pole of the hypersphere, then generate out-of-domain datasets  $\mathcal{D}_j$  at varying distances from  $\mathcal{D}_0$ . Given a model trained on  $\mathcal{D}_0$ , we can calculate its generalization to  $\mathcal{D}_j$ , as well as its neighborhood invariance.

In our experiments, we consider a 16 dimensional hypersphere and sample 1000 points per data distribution for each dataset. The univariate gaussian distributions that we sample data points from have fixed variance 0.005, ensuring models cannot generalize fully across the entire hypersphere. We train 200 single hidden-layer MLPs with hidden dimension of 16 on the training dataset  $\mathcal{D}_0$ , each with a random level of label noise between 0-30% to ensure a wide range of generalization properties. We consider 40 different dataset distances, equally spaced along the hypersphere between opposite poles. For a given dataset distance, we select 5 points at random from the corresponding circumference and generate 5 datasets from univariate gaussians centered at these points. To measure neighborhood invariance for a given  $x$ , we sample 10 transformations from the set of transformations defined as a perturbation along the hypersphere of radius  $\|x\|$  with a maximum distance  $m(g) = \|g(x) - x\| \le 0.01$ . For each dataset we measure the neighborhood invariance and generalization for each of the 200 trained models and calculate the Kendall  $\tau$  correlation between them. For a given dataset distance, the  $\tau$  values are then averaged across all datasets. Results are presented in Figure 3b.

For datasets closest to the training dataset, the correlation between generalization and neighborhood invariance is high. However, as dataset distance increases, correlation decreases. At a distance of around  $\pi/4$ , correlation becomes nearly 0, and continues decreasing until the two values are negatively correlated on data sampled

![Figure 3(a): A 3D diagram of a sphere representing a hypersphere. A central point is labeled D0. Six other points are labeled D1 through D6, distributed across the sphere's surface at various distances from D0. D1 is closest to D0, while D6 is on the opposite side of the sphere. Dashed lines represent the paths of increasing distance from D0 to D6.](b93cbfb52e37619e688175a6aad9edd9_img.jpg)

Figure 3(a): A 3D diagram of a sphere representing a hypersphere. A central point is labeled D0. Six other points are labeled D1 through D6, distributed across the sphere's surface at various distances from D0. D1 is closest to D0, while D6 is on the opposite side of the sphere. Dashed lines represent the paths of increasing distance from D0 to D6.

(a) We generate data from univariate gaussians whose means lie on a hypersphere. We train models on a dataset  $\mathcal{D}_0$  to classify points as inside or outside the hypersphere then test them on out of domain datasets  $\mathcal{D}_j$  that lie at various distances from  $\mathcal{D}_0$  as measured by radian distance along the hypersphere.

![Figure 3(b): A line graph showing the correlation between neighborhood invariance (NI) and OOD generalization (ATC) as a function of dataset distance. The x-axis is 'Dataset Distance' ranging from 0 to π. The y-axis is 'Kendall τ' ranging from -1 to 1. Two data series are plotted: NI (blue line with circles) and ATC (red line with squares). Both series start at approximately 0.6 at distance 0 and decrease as distance increases, reaching approximately -0.8 at distance π. The two lines are nearly identical.](bedcca5cdf168e3508ef511d94ec514c_img.jpg)

Figure 3(b): A line graph showing the correlation between neighborhood invariance (NI) and OOD generalization (ATC) as a function of dataset distance. The x-axis is 'Dataset Distance' ranging from 0 to π. The y-axis is 'Kendall τ' ranging from -1 to 1. Two data series are plotted: NI (blue line with circles) and ATC (red line with squares). Both series start at approximately 0.6 at distance 0 and decrease as distance increases, reaching approximately -0.8 at distance π. The two lines are nearly identical.

(b) The correlation between neighborhood invariance and OOD generalization remains high near the training domain but quickly decreases as dataset distance increases, becoming negatively correlated for datasets further than  $\pi/4$  away on the hypersphere. We observe almost identical results for baseline ATC methods.

Figure 3: Toy analysis of the effect of dataset distance on the correlation between neighborhood invariance and OOD generalization.

from the opposite side of the hypersphere from the training domain. This behavior almost identical for both neighborhood invariance and ATC methods (ATC-MC and ATC-NE perform the same so we report only one). These results demonstrate that the correlation of neighborhood invariance with generalization should decrease as dataset distance increases. To investigate the degree to which this happens in practice, we consider a set of extreme OOD experiments (Section 5.3) where we observe a surprisingly small decrease in correlation, indicating a closer dataset distance than might initially be assumed.

### 5.2 Correlation with OOD Generalization

We first present results in Table 2 analyzing the correlation of our proposed neighborhood invariance measure with OOD generalization. We report  $R^2$ , MAE, Macro  $\tau$ , Micro  $\tau$ , ID  $\tau$ , and Arch  $\tau$  as detailed in Section 4.3. We omit results on Spectral and Frobenius norm measures as they are close to 0 or negative for all metrics. We do not report Micro  $\tau$  values for image classification models since each model type is trained on only one domain. Additional experiments and results are presented in Appendix B.

**ImageNet-Scale Image Classification** Results on ImageNet-scale image classification datasets are presented in Table 2a and are averaged across all architectures. On standard domain shifts, NI-RandAug performs slightly better than ATC methods on  $R^2$  and Macro  $\tau$  with similar MAE. On the adversarial ImageNet-A dataset, ATC methods fail completely whereas NI methods maintain strong performance and still correlate well with accuracy. However, both methods exhibit large MAE and fail to accurately predict actual OOD accuracy. On ID  $\tau$  NI methods perform slightly worse than ATC methods although they still show very strong correlations.

**CIFAR10-Scale Image Classification** Results on smaller scale image classification datasets are presented in Table 2b and are averaged across all architectures. On CI10 datasets, NI-RandAug significantly outperforms ATC baselines and all other measures on all metrics. NI-RandAug also exhibits only a small decrease in correlation when moving from in-domain (ID  $\tau$ ) to OOD datasets (Macro  $\tau$ ), compared to ATC methods which suffer a much larger drop. On Numbers datasets, NI-RandAug outperforms all other methods on  $R^2$  and MAE, although it performs slightly worse on Macro  $\tau$  compared to NI-Translate and on ID  $\tau$  compared to ATC baselines.

For both CI10 and Numbers, using patch erasing and flip and crop transformations cause our method to perform worse or fail entirely, in contrast to ImageNet where they perform similarly or better. For these

| Measure      | Domain Shifts |              |              | ImageNet-A   |              |              |
|--------------|---------------|--------------|--------------|--------------|--------------|--------------|
|              | $R^2$         | MAE          | Macro $\tau$ | $R^2$        | MAE          | Macro $\tau$ |
| NI-RandAug   | <b>0.709</b>  | 11.87        | <b>0.724</b> | 0.577        | <b>31.17</b> | 0.586        |
| NI-Translate | 0.587         | 13.58        | 0.604        | 0.468        | 29.54        | 0.439        |
| NI-Erase     | 0.492         | 11.86        | 0.555        | 0.446        | 33.06        | 0.517        |
| NI-FC        | 0.603         | 12.28        | 0.691        | <b>0.679</b> | 31.84        | <b>0.589</b> |
| ATC-NE       | 0.607         | <b>11.40</b> | 0.703        | 0.209        | 32.00        | 0.248        |
| ATC-MC       | 0.622         | 11.97        | 0.691        | 0.159        | 32.85        | 0.190        |

(a) Results on ImageNet scale models and datasets. On standard domain shift datasets NI-RandAug performs slightly better than ATC methods, and maintains strong performance on adversarial data where ATC methods fail completely.

| Measure      | CI10         |             |              |              | Numbers      |              |             |              |              |
|--------------|--------------|-------------|--------------|--------------|--------------|--------------|-------------|--------------|--------------|
|              | $R^2$        | MAE         | Macro $\tau$ | ID $\tau$    | Arch $\tau$  | $R^2$        | MAE         | Macro $\tau$ | ID $\tau$    |
| NI-RandAug   | <b>0.899</b> | <b>3.11</b> | <b>0.768</b> | <b>0.793</b> | <b>0.837</b> | <b>0.764</b> | <b>5.33</b> | 0.642        | 0.733        |
| NI-Translate | 0.732        | 3.56        | 0.607        | 0.661        | 0.786        | 0.685        | 6.12        | 0.667        | <b>0.881</b> |
| NI-Erase     | 0.518        | 4.41        | 0.411        | 0.406        | 0.299        | 0.153        | 10.17       | -0.135       | 0.324        |
| NI-FC        | 0.417        | 4.47        | 0.371        | 0.344        | 0.683        | 0.208        | 10.42       | -0.316       | -0.033       |
| ATC-NE       | 0.655        | 3.61        | 0.548        | 0.693        | 0.689        | 0.616        | 6.74        | 0.637        | 0.859        |
| ATC-MC       | 0.640        | 3.65        | 0.544        | 0.682        | 0.685        | 0.692        | 6.19        | <b>0.682</b> | 0.844        |

(b) Results on small scale image classification, averaged across all model architectures. No Micro  $\tau$  is reported since models are trained on a single domain and no Arch  $\tau$  is reported for the Numbers dataset since we only consider a single architecture. NI-RandAug beats all other methods on almost all metrics.

| Measure    | CNN          |             |              |              | RoBERTa      |              |             |              |              |              |              |
|------------|--------------|-------------|--------------|--------------|--------------|--------------|-------------|--------------|--------------|--------------|--------------|
|            | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ | ID $\tau$    | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ | ID $\tau$    | Arch $\tau$  |
| NI-SSMBA   | 0.662        | <b>1.93</b> | <b>0.677</b> | <b>0.689</b> | <b>0.629</b> | <b>0.972</b> | <b>1.29</b> | <b>0.832</b> | <b>0.829</b> | <b>0.838</b> | 0.588        |
| NI-EDA     | 0.641        | 2.04        | 0.664        | 0.649        | 0.611        | 0.968        | 1.45        | 0.830        | 0.810        | 0.830        | 0.512        |
| NI-BT      | 0.550        | 2.99        | 0.592        | 0.501        | 0.538        | 0.961        | 1.47        | 0.813        | 0.801        | 0.801        | 0.523        |
| NI-RandRep | 0.409        | 2.64        | 0.544        | 0.554        | 0.439        | 0.967        | 1.27        | 0.821        | 0.816        | 0.822        | 0.537        |
| ATC-NE     | 0.760        | 2.47        | 0.514        | 0.633        | 0.467        | 0.852        | 2.38        | 0.707        | 0.691        | 0.749        | 0.660        |
| ATC-MC     | <b>0.761</b> | 2.46        | 0.517        | 0.634        | 0.467        | 0.869        | 2.26        | 0.722        | 0.705        | 0.749        | <b>0.663</b> |

(c) Results on sentiment analysis (SA) datasets. NI-SSMBA beats all other methods on almost all metrics.

| Measure    | CNN          |             |              |              | RoBERTa      |              |             |              |              |              |              |
|------------|--------------|-------------|--------------|--------------|--------------|--------------|-------------|--------------|--------------|--------------|--------------|
|            | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ | ID $\tau$    | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ | ID $\tau$    | Arch $\tau$  |
| NI-SSMBA   | 0.575        | 2.09        | 0.570        | <b>0.534</b> | 0.704        | 0.933        | 1.19        | 0.750        | 0.730        | 0.771        | 0.301        |
| NI-EDA     | <b>0.577</b> | <b>2.04</b> | <b>0.581</b> | 0.511        | <b>0.709</b> | 0.941        | 1.26        | <b>0.789</b> | <b>0.757</b> | <b>0.799</b> | 0.572        |
| NI-BT      | 0.509        | 2.11        | 0.470        | 0.449        | 0.584        | <b>0.944</b> | <b>1.07</b> | 0.759        | 0.740        | 0.778        | 0.563        |
| NI-RandRep | 0.451        | 2.20        | 0.452        | 0.428        | 0.570        | 0.890        | 1.70        | 0.688        | 0.647        | 0.710        | 0.401        |
| ATC-NE     | 0.576        | 2.52        | 0.568        | 0.446        | 0.705        | 0.737        | 2.22        | 0.557        | 0.541        | 0.739        | 0.635        |
| ATC-MC     | 0.576        | 2.52        | 0.568        | 0.446        | 0.706        | 0.769        | 2.10        | 0.581        | 0.567        | 0.748        | <b>0.636</b> |

(d) Results on natural language inference (NLI) datasets. NI measures beat baselines on all metrics except Arch  $\tau$ .Table 2: Evaluation metrics measuring the correlation of our neighborhood invariance measure with ID/OOD generalization. The best performing measures for each metric are bolded. On all tasks, neighborhood invariance achieves strong generalization and beats baseline methods on almost all metrics. Full tables for  $R^2$ , Macro  $\tau$ , Micro  $\tau$ , and ID  $\tau$  on individual train/test domains are in Appendix C

| Measure    | CNN          |              | RoBERTa      |              |
|------------|--------------|--------------|--------------|--------------|
|            | $R^2$        | Micro $\tau$ | $R^2$        | Micro $\tau$ |
| NI-SSMBA   | <b>0.584</b> | <b>0.566</b> | <b>0.941</b> | <b>0.816</b> |
| NI-EDA     | 0.575        | <b>0.567</b> | 0.884        | 0.715        |
| NI-BT      | 0.538        | 0.470        | 0.906        | 0.766        |
| NI-RandRep | 0.277        | 0.373        | 0.918        | 0.776        |
| ATC-NE     | 0.271        | 0.405        | 0.329        | 0.356        |
| ATC-MC     | 0.295        | 0.506        | 0.437        | 0.436        |

(a) Results on the `Drugs`.com dataset.

| Measure    | CNN          |              | RoBERTa      |              |
|------------|--------------|--------------|--------------|--------------|
|            | $R^2$        | Micro $\tau$ | $R^2$        | Micro $\tau$ |
| NI-SSMBA   | 0.083        | 0.080        | 0.691        | 0.463        |
| NI-EDA     | 0.202        | 0.110        | <b>0.739</b> | <b>0.540</b> |
| NI-BT      | <b>0.213</b> | <b>0.247</b> | <b>0.730</b> | <b>0.527</b> |
| NI-RandRep | 0.096        | 0.102        | 0.030        | 0.012        |
| ATC-NE     | 0.077        | -0.107       | 0.719        | 0.345        |
| ATC-MC     | 0.076        | -0.106       | 0.734        | 0.354        |

(b) Results on the MedNLI dataset.

Table 3: Evaluation metrics measuring the correlation of our neighborhood invariance measure with generalization on extreme OOD datasets. NI- $\star$  methods beat baselines on both tasks, with RoBERTa models exhibiting only a slight degradation in correlation compared to more typical OOD settings.

smaller datasets, since these transformations are more likely to generate images that cannot be classified (e.g. images without an object in frame), they produce more similar invariance values between classifiers that cannot be used to rank them properly. This effect is more pronounced for Numbers datasets because flipping the image or removing even small portions of the number to be classified can render the task impossible. In contrast, random image translation which almost always preserves label information performs similarly well across both datasets and almost matches NI-RandAug. We provide a larger set of ablations on the Numbers dataset exploring this phenomenon in Section 5.4. The strong performance of NI-RandAug indicates that combining multiple transformations is helpful for mitigating dataset-specific transformation sensitivities, as in the case of Numbers.

**Sentiment Analysis (SA)** Results on Sentiment Analysis datasets are presented in Table 2c. In experiments on both architectures, our neighborhood invariance measures achieves strong correlation with OOD generalization and beats all baselines on almost all metrics. Of the transformations considered, NI-SSMBA performs the best across both architectures. NI-EDA, NI-BT, and even NI-RandRep achieve strong results as well, often beating ATC baselines. We observe particularly strong correlation on RoBERTa models, with a nearly perfectly linear  $R^2$  value of 0.972 and large Micro  $\tau$  of 0.829. We hypothesize that this is due to the pretrained initialization of RoBERTa models, which gives a strong inductive bias towards learning a space invariant to transformations that preserve meaning. In contrast, CNN models are trained from random initializations and may not learn as closely aligned a space. On cross architecture analysis, we observe strong Arch  $\tau$  for our neighborhood measures, although they are outperformed by both ATC methods. Compared to image classification results, our results on sentiment analysis tasks are an overall less sensitive to the data transformations selected because they are less likely to destroy information necessary for classification.

**Natural Language Inference (NLI)** Results on Natural Language Inference tasks are presented in Table 2d. Similar to our sentiment analysis results, our neighborhood invariance measures achieve strong correlation with OOD generalization on both architectures and beat all baselines. Correlations in general on NLI are lower than those of sentiment analysis because it is more difficult to maintain the complex relationship between the two sentences during a transformation. For example, changing a single word can easily change a sentence pair from entailment to contradiction, whereas many words must be changed to modify a 5 star review to a 1 star review. We observe exceptionally high correlation on RoBERTa models, for which we offer a similar hypothesis as in our sentiment analysis experiments. On cross architecture analysis, we observe strong correlation for our NI-EDA and NI-BT although they are outperformed by both ATC methods.

### 5.3 Extreme OOD Generalization

We now consider more extreme generalization to data domains with specialized and knowledge intensive data. We consider only natural language tasks as it is difficult to find a sufficiently specialized image classification dataset that maintains the same output classes. For sentiment analysis we use the `Drugs`.com review dataset (Gräßer et al., 2018), and for natural language inference we use MedNLI (Romanov & Shivade, 2018), an

![Figure 4: Three line plots showing Micro τ for different ablations on CNN models. (a) Correlation vs. # Test Dataset Examples (log scale). (b) Correlation vs. # of Neighborhood Samples. (c) Correlation vs. Corruption %.](398674b42e3466add6d47f420c136494_img.jpg)

Figure 4 consists of three line plots showing the Micro  $\tau$  (Y-axis, ranging from 0.2 to 0.6) for three methods: NI-SSMBA (blue circles), NI-EDA (red squares), and NI-BT (green triangles). The X-axis for all plots is the Micro  $\tau$ .

(a) Correlation improves as the size of the test dataset that we evaluate on increases. The X-axis is '# Test Dataset Examples' on a log scale (10, 100, 1,000). All three methods show an upward trend, with NI-SSMBA consistently having the highest correlation.

| # Test Dataset Examples | NI-SSMBA | NI-EDA | NI-BT |
|-------------------------|----------|--------|-------|
| 10                      | 0.30     | 0.28   | 0.25  |
| 100                     | 0.40     | 0.38   | 0.30  |
| 1,000                   | 0.60     | 0.55   | 0.45  |

(b) Correlation is constant as the number of transformations decreases, even when we sample only a single transformation in addition to the identity transformation. The X-axis is '# of Neighborhood Samples' (10, 100). NI-SSMBA and NI-EDA remain constant at approximately 0.65, while NI-BT is constant at approximately 0.45.

| # of Neighborhood Samples | NI-SSMBA | NI-EDA | NI-BT |
|---------------------------|----------|--------|-------|
| 10                        | 0.65     | 0.65   | 0.45  |
| 100                       | 0.65     | 0.65   | 0.45  |

(c) Correlation increases until an optimal corruption percentage is reached, then decreases as the corruption continues to increase. The X-axis is 'Corruption %' (0, 20, 40, 60, 80). NI-SSMBA shows a peak correlation of approximately 0.65 at 20% corruption, then decreases to approximately 0.40 at 80% corruption.

| Corruption % | NI-SSMBA |
|--------------|----------|
| 0            | 0.60     |
| 20           | 0.65     |
| 40           | 0.62     |
| 60           | 0.55     |
| 80           | 0.40     |

Figure 4: Three line plots showing Micro τ for different ablations on CNN models. (a) Correlation vs. # Test Dataset Examples (log scale). (b) Correlation vs. # of Neighborhood Samples. (c) Correlation vs. Corruption %.

Figure 4: Micro  $\tau$  for our neighborhood invariance measure calculated with varying ablations on CNN models evaluated on Amazon toy reviews.

NI dataset generated from clinical notes and patient history. Both datasets contain highly specific medical language not seen in any of our training domains. All models from all original training domains are evaluated on each of these extreme OOD domains, and we report  $R^2$  and Micro  $\tau$ . Results are shown in Table 3

On the **Drugs.com** dataset, we observe a small decrease in correlation for neighborhood invariance methods compared to results on AWS datasets. However, ATC methods begin to fail, with Micro  $\tau$  on RoBERTa models dropping significantly from 0.706 to 0.356. This suggests that models become poorly calibrated in extreme OOD settings, making ATC methods fragile. On MedNLI we observe a much larger disparity in performance. For CNN models, most of our measures fail to correlate at all, and ATC methods degrade so much they became anti-correlated with generalization. For RoBERTa models we observe only minor drops in correlation for all measures. For both tasks NI-RandRep exhibits almost no correlation with OOD generalization. This suggests that the choice of transformation becomes much more important as we move farther from our training domain.

### 5.4 Ablations

In this section we examine factors that may affect the quality of neighborhood invariance estimation and its correlation with actual generalization. Since rerunning all of our experiments is too costly, we evaluate on **toys** Amazon reviews using a pool of CNN models trained on all other domains for the first three ablations, and on the Numbers datasets using NiN models trained on SVHN for the final two.

**Test Dataset Size:** Does our neighborhood invariance measure still correlate well when the test dataset is small? We randomly and iteratively subsample our test dataset of 2000 examples to reduce our dataset size down to 10 examples. We then measure our models' neighborhood invariance on each subsampled dataset and calculate the Micro  $\tau$  on all models. Results are shown in Figure 4a. We find that for all neighborhoods, smaller datasets lead to noisier invariance estimates and lower correlation. As dataset size increases, correlation increases as well.

**Number of Transformations:** How many transformations do we need to sample in order to generate a reliable neighborhood invariance estimate? We sample a varying number of transformations for each test example, from a minimum of two transformations to a maximum of 100 transformations, then estimate our neighborhood invariance measure with each set of transformations on the entire test dataset and calculate the micro  $\tau$ . By default we always include the identity transformation. Results are shown in Figure 4b. We find that our measure is surprisingly robust to the number of samples, with only a small difference between 100 and 2 transformations sampled. For all measures, correlation slightly increases as the number of samples increases and we achieve a better estimation of the true invariance value.

| Training Regime           | $R^2$        | MAE          | Macro $\tau$ | ID $\tau$    |
|---------------------------|--------------|--------------|--------------|--------------|
| Standard Training         | 0.684        | 12.37        | <b>0.763</b> | 0.852        |
| + Augmentation/Robustness | 0.707        | 12.23        | 0.650        | <b>0.855</b> |
| + Pretraining/Extra Data  | <b>0.799</b> | <b>11.73</b> | 0.707        | 0.836        |
| All Models                | 0.709        | 11.87        | 0.724        | 0.845        |

Table 4: NI-RandAug results on different subsets of the ImageNet Testbed models trained with different training regimes. Training with augmentation and robustness interventions decreases macro  $\tau$  compared to standard training but increases all other metrics.  $R^2$  and MAE are higher for models trained/pretrained with large amounts of data.

**Transformation Magnitude:** How sensitive is our measure to the maximum magnitude of transformation considered? We use the SSMBa transformation for which the magnitude of a transformation is defined by the percentage of tokens corrupted, which we vary from a minimum of 5% to a maximum of 85%. After sampling a set of transformations from each corruption level, we estimate neighborhood invariance on the test dataset using each set and calculate the micro  $\tau$  on all models. Results are shown in Figure 4c. We find that as we begin to increase the corruption percentage, correlation begins to increase as well. Correlation reaches a maximum, then decreases as we continue to increase our corruption percentage. However, even at 85% corruption, our method is quite robust and achieves a micro  $\tau$  of 0.416.

**Selecting Transformations:** How do we ensure that transformations are suitable for a given dataset and do not destroy label information? We consider the set of NiN models trained on SVHN and a set of image transformations including RandAugment (Cubuk et al., 2020), rotations, translations, shears, brightness jittering, contrast jittering, color jittering, patch erasing, and flips and crops. For each transformation and both OOD datasets ColoredMNIST and MNIST we calculate the Macro  $\tau$  correlation between accuracy and neighborhood invariance, as well as the average entropy difference between a model’s output on a transformed image and on the original image. Results are shown in Figure 5.

Neighborhood invariance is relatively insensitive to the transformation selected, with most transformations performing similarly up to a certain entropy difference threshold around 0.1, after which it fails. The transformations that fail, erase and flip crop, both tend to destroy label information and lead to much higher entropy outputs. We propose this method of examining entropy differences as a simple way to diagnose whether a given transform is appropriate for a specific dataset.

**Pretraining and Training with Augmentation:** Does self-supervised pretraining or training with data augmentations, which should make models more invariant to certain transformations, make our neighborhood invariance measure ineffective? We begin by examining our metrics on subsets of models from the ImageNet Testbed (Taori et al., 2020) split by models trained on standard ImageNet (81 models), models trained on ImageNet with augmentations and robustness interventions (74 models), and finally models trained with extra data or pretrained with self-supervised objectives (41 models). Results are shown in Table 4. We find that compared to evaluating only on standard training models, the addition of augmentations or pretraining slightly degrades Macro  $\tau$ , but improves  $R^2$  and MAE. Compared to the overall results on all models, we do not observe any large decreases in performance.

Since the augmentations considered in the set of ImageNet Testbed models are not the same across models, we also consider models from PGDL (Jiang et al., 2020) trained with and without a single type of augmentation: flip crop. Calculating our evaluation metrics on each set of models allows us to isolate the effect of data augmentation. Results are shown in Table 5. We find that measuring neighborhood invariance using the same transformation (NI-FC) that models are trained with causes only a slight degradation compared to models trained without. When measuring invariance using other transformations (NI-RandAug, NI-Erase), evaluation metrics actually improve slightly.

![Figure 5: A scatter plot showing the correlation between Average Entropy Δ (x-axis, ranging from -0.1 to 0.3) and Macro τ (y-axis, ranging from -0.5 to 1). Data points are colored according to the transformation they represent: RandAugment (blue), Affine 10 (green), Erase (red), FlipCrop (purple), Rotate 20 (orange), Rotate 30 (yellow), Translate 10 (brown), Translate 30 (grey), Shear 15 (pink), Color (dark blue), Contrast (light blue), and Brightness (dark green). The points are clustered around the origin, indicating a correlation between entropy and neighborhood invariance.](0b8b087a7baa471015d3ffeaa43d9a6c_img.jpg)

Figure 5: A scatter plot showing the correlation between Average Entropy Δ (x-axis, ranging from -0.1 to 0.3) and Macro τ (y-axis, ranging from -0.5 to 1). Data points are colored according to the transformation they represent: RandAugment (blue), Affine 10 (green), Erase (red), FlipCrop (purple), Rotate 20 (orange), Rotate 30 (yellow), Translate 10 (brown), Translate 30 (grey), Shear 15 (pink), Color (dark blue), Contrast (light blue), and Brightness (dark green). The points are clustered around the origin, indicating a correlation between entropy and neighborhood invariance.

Figure 5: Transformations whose neighborhood invariance measures correlate well with OOD generalization exhibit smaller differences in output entropy.

| Measure      | Flip Crop | $R^2$        | MAE         | Macro $\tau$ | ID $\tau$    |
|--------------|-----------|--------------|-------------|--------------|--------------|
| NI-RandAug   | ✗         | 0.833        | 3.32        | 0.719        | <b>0.753</b> |
|              | ✓         | <b>0.844</b> | <b>3.16</b> | <b>0.732</b> | 0.744        |
| NI-Translate | ✗         | 0.874        | 2.61        | 0.768        | 0.831        |
|              | ✓         | <b>0.877</b> | <b>2.52</b> | <b>0.794</b> | <b>0.845</b> |
| NI-Erase     | ✗         | 0.790        | 3.28        | 0.716        | 0.702        |
|              | ✓         | <b>0.812</b> | <b>3.17</b> | <b>0.722</b> | <b>0.719</b> |
| NI-FC        | ✗         | <b>0.681</b> | 4.34        | <b>0.620</b> | <b>0.587</b> |
|              | ✓         | 0.663        | <b>4.23</b> | 0.608        | 0.554        |

Table 5: Training with and without flip crop augmentation has a minimal effect on the effectiveness of our method, even when the transformation neighborhood aligns with those used to train the model (NI-FC).

## 6 Discussion

In this paper, motivated by the limited settings in which existing complexity measures can be applied, we propose a simple to calculate neighborhood invariance measure that can be applied even when test distributions are unknown and model training data, weights, and gradients are unavailable. We evaluate our method on image classification, sentiment analysis, and natural language inference datasets, calculating a variety of correlation metrics with both in-domain and out-of-domain (OOD) generalization. Across almost all tasks and experimental settings, we find that our neighborhood invariance measure consistently outperforms baseline methods and correlates strongly with actual generalization. However, our method has several limitations. Data transformations must be selected such that labels for transformed points are still well defined, although examining entropy differences can diagnose poor transformation choices. In settings where such transformations are difficult to define, our method may not be applicable or provide inappropriately high estimates, so practitioners must be careful to verify their estimates with a labelled test set. In addition, our neighborhood invariance measure may fail in sufficiently OOD settings where a model may become poorly calibrated or degenerate, although we find in practice on our tasks that even extreme OOD settings are similar enough for our measure to perform well. In future work we plan to explore using similar measures calculated over transformation neighborhoods as a method for OOD detection.

## 7 Acknowledgments

We would like to thank Taylor Killian, Tom Hartvigsen, and Swami Sankaranarayanan for their helpful discussion and comments. Resources used in preparing this research were provided, in part, by the Province of Ontario, the Government of Canada through CIFAR, and companies sponsoring the Vector Institute ([www.vectorinstitute.ai/partners](http://www.vectorinstitute.ai/partners)).

## References

- Sumukh Aithal K, Dhruva Kashyap, and Natarajan Subramanyam. Robustness to Augmentations as a Generalization metric. *arXiv e-prints*, art. arXiv:2101.06459, January 2021.
- Zeyuan Allen-Zhu, Yuanzhi Li, and Yingyu Liang. Learning and Generalization in Overparameterized Neural Networks, Going Beyond Two Layers. *arXiv e-prints*, art. arXiv:1811.04918, November 2018.
- Fabio Anselmi, Lorenzo Rosasco, and Tomaso Poggio. On Invariance and Selectivity in Representation Learning. *arXiv e-prints*, art. arXiv:1503.05938, March 2015.
- Fabio Anselmi, Joel Z. Leibo, Lorenzo Rosasco, Jim Mutch, Andrea Tacchetti, and Tomaso Poggio. Unsupervised learning of invariant representations. *Theoretical Computer Science*, 633:112–121, 2016. ISSN 0304-

3975. doi: <https://doi.org/10.1016/j.tcs.2015.06.048>. URL <https://www.sciencedirect.com/science/article/pii/S0304397515005587>. Biologically Inspired Processes in Neural Computation.
- Martin Arjovsky, Léon Bottou, Ishaaan Gulrajani, and David Lopez-Paz. Invariant risk minimization. *arXiv*, 2019.
- Christina Baek, Yiding Jiang, Aditi Raghunathan, and Zico Kolter. Agreement-on-the-line: Predicting the performance of neural networks under distribution shift, 2022.
- Andrei Barbu, David Mayo, Julian Alverio, William Luo, Christopher Wang, Dan Gutfreund, Josh Tenenbaum, and Boris Katz. Objectnet: A large-scale bias-controlled dataset for pushing the limits of object recognition models. In H. Wallach, H. Larochelle, A. Beygelzimer, F. d’Alché-Buc, E. Fox, and R. Garnett (eds.), *Advances in Neural Information Processing Systems*, volume 32. Curran Associates, Inc., 2019. URL [https://proceedings.neurips.cc/paper\\_files/paper/2019/file/97af07a14acba681feacf3012730892-Paper.pdf](https://proceedings.neurips.cc/paper_files/paper/2019/file/97af07a14acba681feacf3012730892-Paper.pdf).
- Peter L. Bartlett and Shahar Mendelson. Rademacher and gaussian complexities: Risk bounds and structural results. *J. Mach. Learn. Res.*, 3(null):463–482, mar 2003. ISSN 1532-4435.
- Peter L Bartlett, Dylan J Foster, and Matus J Telgarsky. Spectrally-normalized margin bounds for neural networks. In I. Guyon, U. V. Luxburg, S. Bengio, H. Wallach, R. Fergus, S. Vishwanathan, and R. Garnett (eds.), *Advances in Neural Information Processing Systems*, volume 30. Curran Associates, Inc., 2017. URL <https://proceedings.neurips.cc/paper/2017/file/b22b257ad0519d4500539da3c8bcf4dd-Paper.pdf>.
- Shai Ben-David, John Blitzer, Koby Crammer, and Fernando Pereira. Analysis of representations for domain adaptation. In B. Schölkopf, J. Platt, and T. Hoffman (eds.), *Advances in Neural Information Processing Systems*, volume 19. MIT Press, 2007. URL <https://proceedings.neurips.cc/paper/2006/file/b1b0432ceaf0ce714426e9114852ac7-Paper.pdf>.
- Lucas Beyer, Xiaohua Zhai, Amélie Royer, Larisa Markova, Rohan Anil, and Alexander Kolesnikov. Knowledge distillation: A good teacher is patient and consistent. *CoRR*, abs/2106.05237, 2021. URL <https://arxiv.org/abs/2106.05237>.
- Tom Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared D Kaplan, Prafulla Dhariwal, Arvind Neelakantan, Pranav Shyam, Girish Sastry, Amanda Askell, Sandhini Agarwal, Ariel Herbert-Voss, Gretchen Krueger, Tom Henighan, Rewon Child, Aditya Ramesh, Daniel Ziegler, Jeffrey Wu, Clemens Winter, Chris Hesse, Mark Chen, Eric Sigler, Mateusz Litwin, Scott Gray, Benjamin Chess, Jack Clark, Christopher Berner, Sam McCandlish, Alec Radford, Ilya Sutskever, and Dario Amodei. Language models are few-shot learners. In H. Larochelle, M. Ranzato, R. Hadsell, M.F. Balcan, and H. Lin (eds.), *Advances in Neural Information Processing Systems*, volume 33, pp. 1877–1901. Curran Associates, Inc., 2020. URL [https://proceedings.neurips.cc/paper\\_files/paper/2020/file/1457c0dbfcb4967418bf8ac142f64a-Paper.pdf](https://proceedings.neurips.cc/paper_files/paper/2020/file/1457c0dbfcb4967418bf8ac142f64a-Paper.pdf).
- Peter Bühlmann. Invariance, causality and robustness, 2018.
- Mayee Chen\*, Karan Goel\*, Nimit Sohoni\*, Fait Poms, Kayvon Fatahalian, and Christopher Re. Mandoline: Model evaluation under distribution shift. *International Conference of Machine Learning (ICML)*, 2021.
- Minshuo Chen, Haoming Jiang, Wenjing Liao, and Tuo Zhao. Nonparametric Regression on Low-Dimensional Manifolds using Deep ReLU Networks : Function Approximation and Statistical Recovery. *arXiv e-prints*, art. arXiv:1908.01842, August 2019.
- Ching-Yao Chuang, Antonio Torralba, and Stefanie Jegelka. Estimating generalization under distribution shifts via domain-invariant representations. *International conference on machine learning*, 2020.
- Ekin Dogus Cubuk, Barret Zoph, Jon Shlens, and Quoc Le. Randaugment: Practical automated data augmentation with a reduced search space. In H. Larochelle, M. Ranzato, R. Hadsell, M.F. Balcan, and H. Lin (eds.), *Advances in Neural Information Processing Systems*, volume 33, pp. 18613–18624. Curran Associates, Inc., 2020. URL <https://proceedings.neurips.cc/paper/2020/file/d85b63ef0cc114d0a3bb7b7d808028f-Paper.pdf>.

- Luke N. Darlow, Elliot J. Crowley, Andreas Antoniou, and Amos J. Storkey. CINIC-10 is not ImageNet or CIFAR-10. *arXiv e-prints*, art. arXiv:1810.03505, October 2018.
- Jia Deng, Wei Dong, Richard Socher, Li-Jia Li, Kai Li, and Li Fei-Fei. Imagenet: A large-scale hierarchical image database. In *2009 IEEE Conference on Computer Vision and Pattern Recognition*, pp. 248–255, 2009. doi: 10.1109/CVPR.2009.5206848.
- Li Deng. The mnist database of handwritten digit images for machine learning research. *IEEE Signal Processing Magazine*, 29(6):141–142, 2012.
- Weijian Deng and Liang Zheng. Are labels always necessary for classifier accuracy evaluation? In *Proc. CVPR*, 2021.
- Weijian Deng, Stephen Gould, and Liang Zheng. What does rotation prediction tell us about classifier accuracy under varying testing environments? In *ICML*, 2021.
- Alexey Dosovitskiy, Lucas Beyer, Alexander Kolesnikov, Dirk Weissenborn, Xiaohua Zhai, Thomas Unterthiner, Mostafa Dehghani, Matthias Minderer, Georg Heigold, Sylvain Gelly, Jakob Uszkoreit, and Neil Houlsby. An image is worth 16x16 words: Transformers for image recognition at scale. In *International Conference on Learning Representations*, 2020.
- Gintare Karolina Dziugaite and Daniel M. Roy. Computing nonvacuous generalization bounds for deep (stochastic) neural networks with many more parameters than training data. In *Proceedings of the 33rd Annual Conference on Uncertainty in Artificial Intelligence (UAI)*, 2017.
- Saurabh Garg, Sivaraman Balakrishnan, Zachary C. Lipton, Behnam Neyshabur, and Hanie Sedghi. Leveraging Unlabeled Data to Predict Out-of-Distribution Performance. *arXiv e-prints*, art. arXiv:2201.04234, January 2022.
- Vikas Garg, Adam Tauman Kalai, Katrina Liggett, and Steven Wu. Learn to expect the unexpected: Probably approximately correct domain generalization. In Arindam Banerjee and Kenji Fukumizu (eds.), *Proceedings of The 24th International Conference on Artificial Intelligence and Statistics*, volume 130 of *Proceedings of Machine Learning Research*, pp. 3574–3582. PMLR, 13–15 Apr 2021. URL <https://proceedings.mlr.press/v130/garg21a.html>.
- Stanton A Glantz, Bryan K Slinker, and Torsten B Neilands. *Primer of Applied Regression and Analysis of Variance*. Health Professions Division, McGraw-Hill, New York, 1990.
- Ian Goodfellow, Jonathon Shlens, and Christian Szegedy. Explaining and harnessing adversarial examples. *arXiv 1412.6572*, 12 2014.
- Felix Gräßer, Surya Kallumadi, Hagen Malberg, and Sebastian Zaunseder. Aspect-based sentiment analysis of drug reviews applying cross-domain and cross-data learning. In *Proceedings of the 2018 International Conference on Digital Health*, DH ’18, pp. 121–125, New York, NY, USA, 2018. Association for Computing Machinery. ISBN 9781450364935. doi: 10.1145/3194658.3194677. URL <https://doi.org/10.1145/3194658.3194677>.
- Keren Gu, Brandan Yang, Jiquan Ngiam, Quoc Le, and Jonathon Shlens. Using videos to evaluate image model robustness. In *SafeML Workshop at ICLR*, 2019.
- Devin Guillory, Vaishaal Shankar, Sayna Ebrahimi, Trevor Darrell, and Ludwig Schmidt. Predicting with Confidence on Unseen Distributions. In *International Conference on Computer Vision*, 2021.
- Ishaan Gulrajani and David Lopez-Paz. In Search of Lost Domain Generalization. *arXiv e-prints*, art. arXiv:2007.01434, July 2020.
- Abhishek Gupta, Alagan Anpalagan, Ling Guan, and Ahmed Shaharyar Khwaja. Deep learning for object detection and scene perception in self-driving cars: Survey, challenges, and open issues. *Array*, 10: 100057, 2021. ISSN 2590-0056. doi: <https://doi.org/10.1016/j.array.2021.100057>. URL <https://www.sciencedirect.com/science/article/pii/S259000562100059>.

- Trygve Haavelmo. The statistical implications of a system of simultaneous equations. *Econometrica*, 11(1): 1–12, 1943. ISSN 00129682, 14680262. URL <http://www.jstor.org/stable/1905714>.
- Jeff Z. HaoChen, Colin Wei, Adrien Gaidon, and Tengyu Ma. Provable Guarantees for Self-Supervised Deep Learning with Spectral Contrastive Loss. *arXiv e-prints*, art. arXiv:2106.04156, June 2021.
- Kaiming He, Xiangyu Zhang, Shaoqing Ren, and Jian Sun. Deep Residual Learning for Image Recognition. *arXiv e-prints*, art. arXiv:1512.03385, December 2015.
- Dan Hendrycks, Xiaoyuan Liu, Eric Wallace, Adam Dziedzic, Rishabh Krishnan, and Dawn Song. Pretrained transformers improve out-of-distribution robustness. In *Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics*, pp. 2744–2751, Online, July 2020a. Association for Computational Linguistics. doi: 10.18653/v1/2020.acl-main.244. URL <https://aclanthology.org/2020.acl-main.244>.
- Dan Hendrycks, Xiaoyuan Liu, Eric Wallace, Adam Dziedzic, Rishabh Krishnan, and Dawn Song. Pretrained transformers improve out-of-distribution robustness. In *Association for Computational Linguistics*, 2020b.
- Dan Hendrycks, Steven Basart, Norman Mu, Saurav Kadavath, Frank Wang, Evan Dorundo, Rahul Desai, Tyler Zhu, Samyak Parajuli, Mike Guo, Dawn Song, Jacob Steinhardt, and Justin Gilmer. The many faces of robustness: A critical analysis of out-of-distribution generalization. *ICCV*, 2021a.
- Dan Hendrycks, Kevin Zhao, Steven Basart, Jacob Steinhardt, and Dawn Song. Natural adversarial examples. *CVPR*, 2021b.
- Yiding Jiang, Dilip Krishnan, Hossein Mobahi, and Samy Bengio. Predicting the generalization gap in deep networks with margin distributions. In *International Conference on Learning Representations*, 2019. URL <https://openreview.net/forum?id=HJ1QfnCqKX>.
- Yiding Jiang, Behnam Neyshabur, Hossein Mobahi, Dilip Krishnan, and Samy Bengio. Fantastic Generalization Measures and Where to Find Them. *arXiv e-prints*, art. arXiv:1912.02178, December 2019.
- Yiding Jiang, Pierre Foret, Scott Yak, Daniel M. Roy, Hossein Mobahi, Gintare Karolina Dziugaite, Samy Bengio, Suriya Gunasekar, Isabelle Guyon, and Behnam Neyshabur. NeurIPS 2020 Competition: Predicting Generalization in Deep Learning. *arXiv e-prints*, art. arXiv:2012.07976, December 2020.
- Yiding Jiang, Vaishnavh Nagarajan, Christina Baek, and J. Zico Kolter. Assessing Generalization of SGD via Disagreement. *arXiv e-prints*, art. arXiv:2106.13799, June 2021.
- M. G. Kendall. A new measure of rank correlation. *Biometrika*, 30(1/2):81–93, 1938. ISSN 00063444. URL <http://www.jstor.org/stable/2332226>.
- Nitish Shirish Keskar, Dheevatsa Mudigere, Jorge Nocedal, Mikhail Smelyanskiy, and Ping Tak Peter Tang. On large-batch training for deep learning: Generalization gap and sharp minima. *arXiv preprint arXiv:1609.04836*, 2016.
- Yoon Kim. Convolutional neural networks for sentence classification. In *Proceedings of the 2014 Conference on Empirical Methods in Natural Language Processing (EMNLP)*, 2014.
- Diederik P. Kingma and Jimmy Ba. Adam: A method for stochastic optimization, 2014. URL <http://arxiv.org/abs/1412.6980>. cite arxiv:1412.6980Comment: Published as a conference paper at the 3rd International Conference for Learning Representations, San Diego, 2015.
- Pang Wei Koh, Shiori Sagawa, Henrik Marklund, Sang Michael Xie, Marvin Zhang, Akshay Balsubramani, Weihua Hu, Michihiro Yasunaga, Richard Lanas Phillips, Irena Gao, Tony Lee, Etienne David, Ian Stavness, Wei Guo, Berton A. Earnshaw, Imran S. Haque, Sara Beery, Jure Leskovec, Anshul Kundaje, Emma Pierson, Sergey Levine, Chelsea Finn, and Percy Liang. WILDS: A benchmark of in-the-wild distribution shifts. In *International Conference on Machine Learning (ICML)*, 2021.
- Alex Krizhevsky. Learning multiple layers of features from tiny images. Technical report, 2009.

- Tengyuan Liang, Tomaso Poggio, Alexander Rakhlin, and James Stokes. Fisher-rao metric, geometry, and complexity of neural networks. In Kamalika Chaudhuri and Masashi Sugiyama (eds.), *Proceedings of the Twenty-Second International Conference on Artificial Intelligence and Statistics*, volume 89 of *Proceedings of Machine Learning Research*, pp. 888–896. PMLR, 16–18 Apr 2019. URL <https://proceedings.mlr.press/v89/liang19a.html>.
- Weixin Liang and James Zou. Metashift: A dataset of datasets for evaluating contextual distribution shifts and training conflicts. In *International Conference on Learning Representations*, 2022. URL <https://openreview.net/forum?id=HTex8qKav0S>.
- Min Lin, Qiang Chen, and Shuicheng Yan. Network In Network. *arXiv e-prints*, art. arXiv:1312.4400, December 2013.
- Yinhan Liu, Myle Ott, Naman Goyal, Jingfei Du, Mandar Joshi, Danqi Chen, Omer Levy, Mike Lewis, Luke Zettlemoyer, and Veselin Stoyanov. Roberta: A robustly optimized bert pretraining approach. *arXiv preprint arXiv:1907.11692*, 2019.
- Shangyun Lu, Bradley Nott, Aaron Olson, Alberto Todeschini, Hossein Vahabi, Yair Carmon, and Ludwig Schmidt. Harder or different? a closer look at distribution shift in dataset reproduction. In *ICML Workshop on Uncertainty and Robustness in Deep Learning*, 2020.
- Yuchen Luo, Jun Zhu, Mengxi Li, Yong Ren, and Bo Zhang. Smooth neighbors on teacher graphs for semi-supervised learning. In *2018 IEEE/CVF Conference on Computer Vision and Pattern Recognition*, pp. 8896–8905, 2018. doi: 10.1109/CVPR.2018.00927.
- David A. McAllester. Pac-bayesian model averaging. In *Proceedings of the Twelfth Annual Conference on Computational Learning Theory*, COLT ’99, pp. 164–170, New York, NY, USA, 1999. Association for Computing Machinery. ISBN 1581131674. doi: 10.1145/307400.307435. URL <https://doi.org/10.1145/307400.307435>.
- John Miller, Rohan Taori, Aditi Raghunathan, Shiori Sagawa, Pang Wei Koh, Vaishaal Shankar, Percy Liang, Yair Carmon, and Ludwig Schmidt. Accuracy on the line: On the strong correlation between out-of-distribution and in-distribution generalization, 2021.
- Peymam Morteza and Yixuan Li. Provable guarantees for understanding out-of-distribution detection. *Proceedings of the AAAI Conference on Artificial Intelligence*, 36(7):7831–7840, Jun. 2022. doi: 10.1609/aaai.v36i7.20752. URL <https://ojs.aaai.org/index.php/AAAI/article/view/20752>.
- Lili Mou, Rui Men, Ge Li, Yan Xu, Lu Zhang, Rui Yan, and Zhi Jin. Natural language inference by tree-based convolution and heuristic matching. In *Proceedings of the 54th Annual Meeting of the Association for Computational Linguistics (Volume 2: Short Papers)*, pp. 130–136, Berlin, Germany, August 2016. Association for Computational Linguistics. doi: 10.18653/v1/P16-2022. URL <https://aclanthology.org/P16-2022>.
- Vaishnavh Nagarajan and J Zico Kolter. Generalization in deep networks: The role of distance from initialization. In *NeurIPS Workshop on Deep Learning: Bridging Theory and Practice*, 2019.
- Ryumei Nakada and Masaaki Imaizumi. Adaptive approximation and generalization of deep neural network with intrinsic dimensionality. *Journal of Machine Learning Research*, 21(174):1–38, 2020. URL <http://jmlr.org/papers/v21/20-002.html>.
- Yuval Netzer, Tao Wang, Adam Coates, A. Bissacco, Bo Wu, and A. Ng. Reading digits in natural images with unsupervised feature learning. In *NIPS Workshop on Deep Learning and Unsupervised Feature Learning*, 2011.
- Behnam Neyshabur, Russ R Salakhutdinov, and Nati Srebro. Path-sgd: Path-normalized optimization in deep neural networks. In C. Cortes, N. Lawrence, D. Lee, M. Sugiyama, and R. Garnett (eds.), *Advances in Neural Information Processing Systems*, volume 28. Curran Associates, Inc., 2015a. URL <https://proceedings.neurips.cc/paper/2015/file/ea32c96f620053cf442ad32258076b9-Paper.pdf>.

- Behnam Neyshabur, Ryota Tomioka, and Nathan Srebro. Norm-based capacity control in neural networks. In Peter Grünwald, Elad Hazan, and Satyen Kale (eds.), *Proceedings of The 28th Conference on Learning Theory*, volume 40 of *Proceedings of Machine Learning Research*, pp. 1376–1401, Paris, France, 03–06 Jul 2015b. PMLR. URL <https://proceedings.mlr.press/v40/Neyshabur15.html>.
- Behnam Neyshabur, Srinadh Bhojanapalli, David McAllester, and Nathan Srebro. Exploring generalization in deep learning. In *Proceedings of NeurIPS*, 2017a.
- Behnam Neyshabur, Srinadh Bhojanapalli, and Nathan Srebro. A pac-bayesian approach to spectrally-normalized margin bounds for neural networks. In *International Conference on Learning Representations*, 2017b.
- Nathan Ng, Kyunghyun Cho, and Marzyeh Ghassemi. Ssmba: Self-supervised manifold based data augmentation for improving out-of-domain robustness. In *Proc. of EMNLP*, 2020.
- Jianmo Ni, Jiacheng Li, and Julian McAuley. Justifying recommendations using distantly-labeled reviews and fined-grained aspects. In *Proceedings of EMNLP*, 2019.
- Myle Ott, Sergey Edunov, Alexei Baevski, Angela Fan, Sam Gross, Nathan Ng, David Grangier, and Michael Auli. fairseq: A fast, extensible toolkit for sequence modeling. In *Proceedings of NAACL-HLT 2019: Demonstrations*, 2019.
- Nicolas Papernot, Patrick McDaniel, Ian Goodfellow, Somesh Jha, Z. Berkay Celik, and Ananthram Swami. Practical black-box attacks against machine learning. In *Proceedings of the 2017 ACM on Asia Conference on Computer and Communications Security*, ASIA CCS ’17, pp. 506–519, New York, NY, USA, 2017. Association for Computing Machinery. ISBN 9781450349444. doi: 10.1145/3052973.3053009. URL <https://doi.org/10.1145/3052973.3053009>.
- Gabriel Pereyra, George Tucker, Jan Chorowski, Łukasz Kaiser, and Geoffrey Hinton. Regularizing neural networks by penalizing confident output distributions. In *ICLR*, 2017.
- Jonas Peters, Peter Bühlmann, and Nicolai Meinshausen. Causal inference using invariant prediction: identification and confidence intervals, 2015.
- Alec Radford, Jong Wook Kim, Chris Hallacy, Aditya Ramesh, Gabriel Goh, Sandhini Agarwal, Girish Sastry, Amanda Askell, Pamela Mishkin, Jack Clark, Gretchen Krueger, and Ilya Sutskever. Learning transferable visual models from natural language supervision. *CoRR*, abs/2103.00020, 2021. URL <https://arxiv.org/abs/2103.00020>.
- Benjamin Recht, Rebecca Roelofs, Ludwig Schmidt, and Vaishaal Shankar. Do cifar-10 classifiers generalize to cifar-10? 2018. <https://arxiv.org/abs/1806.00451>.
- Benjamin Recht, Rebecca Roelofs, Ludwig Schmidt, and Vaishaal Shankar. Do ImageNet classifiers generalize to ImageNet? In Kamalika Chaudhuri and Ruslan Salakhutdinov (eds.), *Proceedings of the 36th International Conference on Machine Learning*, volume 97 of *Proceedings of Machine Learning Research*, pp. 5389–5400. PMLR, 09–15 Jun 2019. URL <https://proceedings.mlr.press/v97/recht19a.html>.
- Alexandra Birch Rico Sennrich, Barry Haddow. Improving neural machine translation models with monolingual data. In *Proc. of ACL*, 2016.
- Salah Rifai, Yann N Dauphin, Pascal Vincent, Yoshua Bengio, and Xavier Muller. The manifold tangent classifier. In J. Shawe-Taylor, R. Zemel, P. Bartlett, F. Pereira, and K.Q. Weinberger (eds.), *Advances in Neural Information Processing Systems*, volume 24. Curran Associates, Inc., 2011. URL <https://proceedings.neurips.cc/paper/2011/file/d1f44e2f09dc172978a4d3151d11d63e-Paper.pdf>.
- Alexey Romanov and Chaitanya Shivade. Lessons from natural language inference in the clinical domain. 2018. URL <http://arxiv.org/abs/1808.06752>.

- Olga Russakovsky, Jia Deng, Hao Su, Jonathan Krause, Sanjeev Satheesh, Sean Ma, Zhiheng Huang, Andrej Karpathy, Aditya Khosla, Michael Bernstein, Alexander C. Berg, and Li Fei-Fei. Imagenet large scale visual recognition challenge. *International Journal of Computer Vision*, 115(3):211–252, Dec 2015. ISSN 1573-1405. doi: 10.1007/s11263-015-0816-y. URL <https://doi.org/10.1007/s11263-015-0816-y>.
- Akiyoshi Sannai, Masaki Imaizumi, and Makoto Kawano. Improved generalization bounds of group invariant equivariant deep networks via quotient feature spaces. In *Conference on Uncertainty in Artificial Intelligence*, 2021.
- Yair Schiff, Brian Quanz, Payel Das, and Pin-Yu Chen. Predicting deep neural network generalization with perturbation response curves. In *Neural Information Processing Systems*, 2021.
- Johannes Schmidt-Hieber. Deep ReLU network approximation of functions on a manifold. *arXiv e-prints*, art. arXiv:1908.00695, August 2019.
- Karen Simonyan and Andrew Zisserman. Very deep convolutional networks for large-scale image recognition. In *Proceedings of the International Conference on Learning Representations*, 2015.
- Jure Školič, Raja Giryes, Guillermo Sapiro, and Miguel R. D. Rodrigues. Generalization error of deep neural networks: Role of classification margin and data structure. In *2017 International Conference on Sampling Theory and Applications (SampTA)*, pp. 147–151, 2017. doi: 10.1109/SAMPTA.2017.8024476.
- Taiji Suzuki. Adaptivity of deep relu network for learning in besov and mixed smooth besov spaces: optimal rate and curse of dimensionality. In *International Conference on Learning Representations*, 2018.
- Taiji Suzuki and Atsushi Nitanda. Deep learning is adaptive to intrinsic dimensionality of model smoothness in anisotropic besov space. In M. Ranzato, A. Beygelzimer, Y. Dauphin, P.S. Liang, and J. Wortman Vaughan (eds.), *Advances in Neural Information Processing Systems*, volume 34, pp. 3609–3621. Curran Associates, Inc., 2021. URL <https://proceedings.neurips.cc/paper/2021/file/1dabc10f06236c67cb7dbb37587d8b38a-Paper.pdf>.
- Mingxing Tan and Quoc Le. EfficientNet: Rethinking model scaling for convolutional neural networks. In Kamalika Chaudhuri and Ruslan Salakhutdinov (eds.), *Proceedings of the 36th International Conference on Machine Learning*, volume 97 of *Proceedings of Machine Learning Research*, pp. 6105–6114. PMLR, 09–15 June 2019. URL <https://proceedings.mlr.press/v97/tan19a.html>.
- Rohan Taori, Achal Dave, Vaishaal Shankar, Nicholas Carlini, Benjamin Recht, and Ludwig Schmidt. Measuring robustness to natural distribution shifts in image classification. In H. Larochelle, M. Ranzato, R. Hadsell, M.F. Balcan, and H. Lin (eds.), *Advances in Neural Information Processing Systems*, volume 33, pp. 18583–18599. Curran Associates, Inc., 2020. URL <https://proceedings.neurips.cc/paper/2020/file/48330f857a17c53d217014ee776bf450-Paper.pdf>.
- V. N. Vapnik and A. Ya. Chervonenkis. On the uniform convergence of relative frequencies of events to their probabilities. *Theory of Probability & Its Applications*, 16(2):264–280, 1971. doi: 10.1137/1116025. URL <https://doi.org/10.1137/1116025>.
- Ramakrishna Vedantam, David Lopez-Paz, and David J. Schwab. An empirical investigation of domain generalization with empirical risk minimizers. In *Neural Information Processing Systems*, 2021.
- Vikas Verma, Alex Lamb, Christopher Beckham, Amir Najafi, Ioannis Mitliagkas, David Lopez-Paz, and Yoshua Bengio. Manifold mixup: Better representations by interpolating hidden states. In Kamalika Chaudhuri and Ruslan Salakhutdinov (eds.), *Proceedings of the 36th International Conference on Machine Learning*, volume 97 of *Proceedings of Machine Learning Research*, pp. 6438–6447. PMLR, 09–15 June 2019. URL <https://proceedings.mlr.press/v97/verma19a.html>.
- Haohan Wang, Songwei Ge, Zachary Lipton, and Eric P. Xing. Learning robust global representations by penalizing local predictive power. In H. Wallach, H. Larochelle, A. Beygelzimer, F. d’Alché-Buc, E. Fox, and R. Garnett (eds.), *Advances in Neural Information Processing Systems*, volume 32. Curran Associates, Inc., 2019. URL [https://proceedings.neurips.cc/paper\\_files/paper/2019/file/3eeefceb087e64f89c2d59e8a249915-Paper.pdf](https://proceedings.neurips.cc/paper_files/paper/2019/file/3eeefceb087e64f89c2d59e8a249915-Paper.pdf).

- Jason Wei and Kai Zou. EDA: Easy data augmentation techniques for boosting performance on text classification tasks. In *Proceedings of the 2019 Conference on Empirical Methods in Natural Language Processing and the 9th International Joint Conference on Natural Language Processing (EMNLP-IJCNLP)*, pp. 6383–6389, Hong Kong, China, November 2019. Association for Computational Linguistics. URL <https://www.aclweb.org/anthology/D19-1670>.
- J. Wiens, S. Saria, M. Sendak, M. Ghassemi, V. Liu, F. Doshi-Velez, K. Jung, K. Heller, D. Kale, M. Saeed, P. Ossorio, S. Thadaney-Israni, and A. Goldenberg. Do no harm: A roadmap for responsible machine learning for healthcare. *Nature Medicine*, 25(10):1337–1340, 2019.
- Adina Williams, Nikita Nangia, and Samuel Bowman. A broad-coverage challenge corpus for sentence understanding through inference. In *Proceedings of the 2018 Conference of the North American Chapter of the Association for Computational Linguistics: Human Language Technologies, Volume 1 (Long Papers)*, pp. 1112–1122. Association for Computational Linguistics, 2018. URL <http://aclweb.org/anthology/N18-1101>.
- Mitchell Wortsman, Gabriel Ilharco, Jong Wook Kim, Mike Li, Simon Kornblith, Rebecca Roelofs, Raphael Gontijo Lopes, Hannaneh Hajishirzi, Ali Farhadi, Hongseok Namkoong, and Ludwig Schmidt. Robust fine-tuning of zero-shot models. In *Proceedings of the IEEE/CVF Conference on Computer Vision and Pattern Recognition (CVPR)*, pp. 7959–7971, June 2022.
- Lingxi Xie, Jingdong Wang, Zhen Wei, Meng Wang, and Qi Tian. DisturbLabel: Regularizing CNN on the Loss Layer. *arXiv e-prints*, art. arXiv:1605.00055, April 2016.
- Qizhe Xie, Zihang Dai, Eduard Hovy, Minh-Thang Luong, and Quoc V Le. Unsupervised data augmentation for consistency training. *arXiv preprint arXiv:1904.12848*, 2019.
- Saining Xie, Ross Girshick, Piotr Dollár, Zhuowen Tu, and Kaiming He. Aggregated residual transformations for deep neural networks. *arXiv preprint arXiv:1611.05431*, 2016.
- Yuichi Yoshida and Takeru Miyato. Spectral Norm Regularization for Improving the Generalizability of Deep Learning. *arXiv e-prints*, art. arXiv:1705.10941, May 2017.
- Chiyuan Zhang, Samy Bengio, Moritz Hardt, Benjamin Recht, and Oriol Vinyals. Understanding deep learning requires rethinking generalization. *arXiv e-prints*, art. arXiv:1611.03530, November 2016.
- Hongyi Zhang, Moustapha Cisse, Yann N. Dauphin, and David Lopez-Paz. mixup: Beyond empirical risk minimization. In *International Conference on Learning Representations*, 2018. URL <https://openreview.net/forum?id=r1Ddp1-Rb>.
- Yuchen Zhang, Tianle Liu, Mingsheng Long, and Michael Jordan. Bridging theory and algorithm for domain adaptation. In Kamalika Chaudhuri and Ruslan Salakhutdinov (eds.), *Proceedings of the 36th International Conference on Machine Learning*, volume 97 of *Proceedings of Machine Learning Research*, pp. 7404–7413. PMLR, 09–15 Jun 2019. URL <https://proceedings.mlr.press/v97/zhang19i.html>.
- Zhun Zhong, Liang Zheng, Guoliang Kang, Shaozi Li, and Yi Yang. Random erasing data augmentation. *Proceedings of the AAAI Conference on Artificial Intelligence*, 34(07):13001–13008, Apr. 2020. doi: 10.1609/aaai.v34i07.7000. URL <https://ojs.aaai.org/index.php/AAAI/article/view/7000>.
- Sicheng Zhu, Bang An, and Furong Huang. Understanding the generalization benefit of model invariance from a data perspective. In *NeurIPS*, 2021.

## A Experimental Setup

In this section we present full details of our experimental setup, including data preprocessing and specifics on model architecture and hyperparameter space. All models are trained on a single RTX6000 GPU.

### A.1 Data Preprocessing

Large ImageNet scale datasets are preprocessed using the pipeline provided by Taori et al. (2020). Small scale image classification datasets are preprocessed by normalizing pixel values and resizing to  $32 \times 32$  if necessary. We use the same preprocessing steps for sentiment analysis and NLI experiments. All data is first tokenized using a GPT-2 style tokenizer and BPE vocabulary provided by `fairseq` (Ott et al., 2019). This BPE vocabulary consists of 50263 types. Corresponding labels are encoded using a label dictionary consisting of as many types as there are classes. Input text and labels are then binarized for model training.

### A.2 Model Architecture

The full list of the 196 models we evaluate from the Imagenet-Testbed (Taori et al., 2020) is provided below:

```
alexnet_lpf2, vit_large_patch32_384, wide_resnet101.2, resnet50_aws_baselin, resnet50_feature_cuitax, densenet169,
efficientnet-b2-autoaug, vgg11_bn, resnet50_with_ljpeg_compression_aws, BiT-M-R50x3d-nonfinetuned, efficientnet-b6-autoaug,
resnet50_lpf5, mobilenet_v2, resnet50_aws, densenet121_lpf5, BiT-M-R101x1-nonfinetuned, efficientnet-b2-advrp-autoaug,
resnet101_32x8d_aws, resnet50_with_fog_aws, resnet50_trained_on_S1N_and_IW, resnet101_32x4d, resnet50_with_
contrast_aws, PixResNet50OnCifar10, resnet50_imagenet_subsample_1_of_32_batch64_original_images, resnet101_32x4d_
aws1, squeezeunet1_1, resnet50_imagenet_subsample_1_of_16_batch64_original_images, resnet18-rotation-nocrop_40,
resnet101_cuitax, efficientnet-b5, resnet50_with_motion_blur_aws, vit_large_patch36_384, efficientnet-b4, resnet50_
lpf5, dmlp07, resnet101_32x8d_with_augmix, vgg16, resnet18_sml, vgg11_bn, vgg16, resnet50_with_pixscale_aws,
resnet154, squeezeunet1_lpf2, shufflenet_v2_x1.0, se_resnet101, alexnet_lpf5, densenet121, efficientnet-b3-advrp-autoaug,
resnet50_augmix, resnet50_simsim, efficientnet-b6-advrp-autoaug, resnet50_imagenet_subsample_5000_classes_batch64_
original_images, resnet50_imagenet_subsample_0.0, dmlp08, mobilenet_v2_lpf5, resnet101_lpf5, alexnet_vgg16_bn,
efficientnet-b0, inceptionv3, resnet18-rotation-vorstri0_30, resnet152_3x_simclr2_finetuned_100pt_tf_port, resnet50_
imagenet_subsample_1_of_2_batch64_original_images, wide_resnet50_2, polynet, efficientnet-b7-randaug, dmlp01,
bn_lpf2, instargan_resnet101_32x16d, vgg16_bn_lpf5, resnet50_linf_eps8_robot, efficientnet-b1-advrp-autoaug,
inceptionv3, vit_b_32_clip_zeroshot, resnet18-rotation-vorstri0_40, resnet50_imagenet_100percent_batch64_original_
images, resnet50_with_frost_aws, efficientnet-b3-advrp-autoaug, resnet50_imagenet_subsample_125_classes_batch64_original_
images, efficientnet-b7-advrp-autoaug, resnet50_sml, vgg16_lpf5, vit_base_patch16_224, resnet54_lpf5, resnet152,
32x4d, PixResNet50OnCifar10, resnet50_with_saturnate_aws, PixResNet50OnCifar10_v2, densenet121_lpf5, resnet50_imagenet_subsample_1_of_
4_batch64_original_images, resnet50-rotation-random_40, resnet50_adv-train-free, resnet18_lpf3, BiT-M-R50x3d-nonfinetuned,
efficientnet-b7-advrp-autoaug, resnet50_sml_with_spatter_aws, resnet50_trained_on_S1N, resnet50_simclr2_finetuned_100pt_tf_
port, panametlarge, BiT-M-R50x3d-ILSVRC2012, resnet50_imagenet_subsample_250_classes_batch64_original_images, resnet50_imagenet_
subsample_1_of_2_batch64_original_images, resnet50_v2, resnet50_32x4d, resnet50_clip_zeroshot, resnet50_32x4d_
aws1, BiT-M-R50x1-nonfinetuned, BiT-M-R101x1-ILSVRC2012, resnet50_imagenet_subsample_1_of_8_batch64_original_images, vit_
large_patch16_224, efficientnet-b1-autoaug, efficientnet-b6-advrp-autoaug, efficientnet-b6-autoaug, resnet50_with_zoom_
blur_aws, resnet50_32x4d_sml, PixResNet50_v2, resnet50_lpf5, resnet101_aws1, efficientnet-b5, squeezeunet1_0,
resnet50_imagenet_subsample_1_of_4_batch64_original_images, resnet50_lpf3, binincption, efficientnet-b6-advrp-autoaug,
resnet50_linf_eps8_robot, PixResNet50, mazenet0.5, resnet50_mimp, densenet121_lpf2, resnet18-rotation-standard_40, se_
resnet101_32x4d, resnet18-rotation-random_30, efficientnet-b6-autoaug, efficientnet-b4-autoaug, vgg11, resnet101_32x4d,
BiT-M-R50x1-ILSVRC2012, resnet50_aws, resnet50_aws, resnet50_aws, contrast_motion_blur, jpeg_compression_aws, vgg16_lpf5, resnet50_
dmlp02, inception, resnet152_3x_simclr2_linear_probe_tf_port, dmlp08, binincption-imagenet21k, efficientnet-b6-advrp-autoaug,
resnet101_32x16d_sml, vit_base_patch32_384, densenet101, inceptionresnet_v2, cafferesnet101, instargan_resnet101_32x8d,
resnet50, PixResNet50_on_adaptation, resnet101_32x8d_sml, resnet101_lpf5, mobilenet_v2_lpf5, instargan_resnet101_32x4d,
naznetmobile, mobilenet_v2_lpf2, resnet101_lpf2, se_resnet50, dmlp08, resnet50_with_brightness_aws, resnet101_64x4d,
resnet101_32x4d_sml, vgg16_bn, densenet152, resnet50_deeppaugment_augmix, se_resnet152, resnet50_cutout, resnet50_cuitax,
resnet50_12_eps8_robot, efficientnet-b1, resnet50_with_defocus_blur_aws, BiT-M-R101x1-ILSVRC2012, vgg16_bn_lpf5, resnet50_
trained_on_S1N_and_IW_then_finetuned_on_ILN, naznetlarge, resnet50_with_gaussian_noise_aws, vit_base_patch16_224,
resnet50_aws, resnet50_with_gryasscale_aws, vgg16, resnet54_lpf3, efficientnet-b4-advrp-autoaug, vgg16_lpf3, resnet101_61
```

Our small image classification models are Network in Network (NIN) (Lin et al., 2013), VGG (Simonyan & Zisserman, 2015), and CNN models. Training and hyperparameter details for these models are provided in Jiang et al. (2020).

For natural language tasks, our CNN models are based on the architecture in Kim (2014). Our input embeddings are 512 dimensional, which we treat as our channel dimension. Our base model applies a set of three one dimensional convolutions of kernel size 3, 4, and 5 with 256 output channels. We modulate the number of stacked convolutions (depth) as well as the channel size (width). Each convolution generates a separate representation that is max pooled across the sequence and concatenated together. We feed this representation into a MLP classifier with a single hidden layer of 512 dimensions. We apply dropout of 0.2 to our inputs and MLP classifier.

Our RoBERTa models use a pre-trained RoBERTaBase model provided by `fairseq`. Classification token embeddings are fed into an MLP classifier with a single hidden layer of 512 dimensions. All models are written within the `fairseq` framework (Ott et al., 2019) and trained on a single RTX6000 or T4 GPU.

| Hyperparameter | CNN                   |                  | RoBERTa               |                       |
|----------------|-----------------------|------------------|-----------------------|-----------------------|
|                | SA                    | NLI              | SA                    | NLI                   |
| Batch Size     | {32, 64, 128}         | {32, 64, 128 }   | {8, 16, 32}           | {8, 16, 32}           |
| Depth          | {1, 2, 3 }            | {1, 2, 3 }       | 1                     | 1                     |
| Width          | {128, 256, 512}       | {128, 256, 512 } | 768                   | 768                   |
| Dropout        | {0.0, 0.25, 0.5}      | {0.0, 0.25}      | {0.0, 0.1}            | {0.0, 0.1}            |
| Weight Decay   | {0.0, 0.0001, 0.0005} | 0.0              | {0.0, 0.0001, 0.0005} | {0.0, 0.0001, 0.0005} |
| Label Noise    | 0.0                   | {0.0, 0.2, 0.4}  | {0.0, 0.2, 0.4}       | {0.0, 0.2, 0.4}       |

Table 6: Possible hyperparameter values for each architecture and task.

### A.3 Model Hyperparameters

Hyperparameter values for image classification models are provided in Jiang et al. (2020). For natural language models we vary the following hyperparameters: training domain, batch size, depth, width, dropout, weight decay, and label noise. For training domains, on sentiment analysis we choose between `books`, `clothing`, `home`, `kindle`, `movies`, `pets`, `sports`, `tech`, `tools`, `toys`. For training domains on NLI, we choose between `slate`, `government`, `fiction`, `telephone`, `travel`. NLI datasets include additional test sets `oup`, `nineeleven`, `facetoface`, `verbatim`, `letters`. Possible values for all other hyperparameters are provided in Table 6

### A.4 Model Training

All models are trained with the Adam optimizer (Kingma & Ba, 2014) with  $\beta = (0.9, 0.98)$  and  $\epsilon = 1 \times 10^{-6}$ . CNN models are trained with learning rate  $1 \times 10^{-3}$  and RoBERTa models are trained with learning rate  $1 \times 10^{-5}$ . We use a inverse square root learning rate scheduler to anneal learning rate over training. We early stop CNN models on sentiment analysis at 0.04 cross entropy and on NLI at 0.03 cross entropy. We early stop RoBERTa models on sentiment analysis at 0.05 cross entropy and on NLI at 0.03 cross entropy. Training details for image classification models are provided in Jiang et al. (2020).

### A.5 Transformation Magnitudes

We define how to determine the magnitude of each data transformation below, and use a maximum value based on best practices provided in their respective papers.

- **RandAugment:** The magnitude of a transformation is determined by the magnitude parameter in the RandAugment algorithm as well as the number of augmentations applied. In our experiments we consider transformations with a maximum magnitude of 15, with 3 augmentations for larger ImageNet models and 1 augmentation for smaller models.
- **Translate:** The magnitude of a transformation is determined by the maximum percentage of the image the image will be translated in both the X- and Y-axes. In our experiments we consider translations of up to 10% of the size of the image in both axes.
- **Erase:** The magnitude of the erase transformation is determined by the percentage of the image erased. In our experiments we consider transformations that remove a maximum size of 33% of the total image area and an aspect ratio between 1/3 and 10/3.
- **Flip and Crop:** The magnitude of the flip and crop transformation is determined by the flip probability and the crop size. In our experiments we consider transformations that flip the image 50% of the time and crop the image with a lower bound of 8% of total image area and an aspect ratio between 3/4 and 4/3. Images are resized to  $32 \times 32$  after cropping.
- **SSMBA:** The magnitude of a SSMBA transformation is determined by the percentage of tokens corrupted, where of the tokens selected, 10% are unmasked, 10% are randomly replaced, and the

remaining 80% are masked. In our experiments we consider SSMBa transformations with a maximum of 15% of tokens corrupted.

- **EDA:** The magnitude of an EDA transformation is determined by the percentage of tokens noised. In our experiments we consider transformations with a maximum of 10% of the tokens.
- **Backtranslation:** The magnitude of a backtranslation operation is determined by the temperature of the softmax-ed distribution from which we sample tokens. In our experiments we consider transformations with a maximum temperature of 0.7.
- **Random Replacement:** The magnitude of a random replacement operation is determined by the percentage of tokens replaced. In our experiments we consider transformations with a maximum of 15% of tokens replaced.

## B Additional Experiments

### B.1 Norm-Based Complexity Measures

Following (Jiang et al., 2019), we calculate our spectral norm measure as  $\Pi_{i=1}^d \|\mathbf{W}_i\|_2^2$  and Frobenius norm measure  $\Pi_{i=1}^d \|\mathbf{W}_i\|_F^2$ . We do not list results on these measures as the correlations are often negative or 0.

### B.2 Cross-Domain Correlation

In this set of experiments we measure the correlation between neighborhood invariance and generalization values of a single model trained on a single training domain evaluated across different OOD test domains. For natural language experiments we average correlations across all CNN and RoBERTa models and training domains and call these the **CNN  $\tau$**  and **Roberta  $\tau$** . Since these results are rank correlations over only 9 values, they are quite noisy. Results are presented in Table 7.

Neighborhood invariance performs quite poorly on both models, although they still outperforms ATC baselines. We hypothesize different regions of the input space may have different optimal levels of smoothness that achieve the lowest generalization error. Our value of interest is then not the absolute smoothness, but the *relative* smoothness compared to this optimal value. These values are the same when comparing different models evaluated on the same domain, but are not the same for the same model evaluated on different domains, making correlating across domains difficult. On the natural image manifold where domains are more well behaved and uniform compared to the natural language manifold, the relative smoothness may not differ much between domains allowing us to correlate our measure across domains.

### B.3 Negative Entropy Results

As an alternative to defining the invariance as the maximum value of the neighborhood decision distribution in Eq. 1, we also consider defining it using the negative entropy of the same distribution:

$$\mu(f, x) = \sum_{j \in \mathcal{Y}} p_j(x) \log p_j(x)$$

A full table of results including metrics calculated on neighborhood invariance measured with negative entropy is provided in Table 8. We refer to measures calculated with entropy as **NE-SSMBa**, **NE-EDA**, **NE-BT**, and **NE-Random**. For most metrics, NE- $\ast$  methods perform similarly or slightly worse.

## C Full Results

We provide a full breakdown of results on the correlation metrics  $R^2$  (Tables 10, 11, 12, 13, 14, 15), macro  $\tau$  (Tables 17, 18, 19, 20, 21, 22), micro  $\tau$  (Tables 23, 24), and ID  $\tau$  (Tables 25, 26, 27) for each set of datasets and models. We also provide additional standard deviations for all main results in Table 2.

| Task | Measure    | CNN $\tau$   | RoBERTa $\tau$ |
|------|------------|--------------|----------------|
| SA   | NI-SSMBA   | 0.360        | 0.010          |
|      | NI-EDA     | 0.431        | <b>0.266</b>   |
|      | NI-BT      | 0.505        | 0.245          |
|      | NI-RandRep | <b>0.570</b> | 0.150          |
|      | ATC-NE     | 0.543        | 0.228          |
|      | ATC-MC     | 0.539        | 0.224          |
| NLI  | NI-SSMBA   | 0.022        | 0.260          |
|      | NI-EDA     | 0.102        | 0.335          |
|      | NI-BT      | 0.089        | 0.333          |
|      | NI-RandRep | <b>0.226</b> | <b>0.440</b>   |
|      | ATC-NE     | 0.219        | 0.231          |
|      | ATC-MC     | 0.223        | 0.239          |

Table 7: Correlation metrics evaluating the ability of our smoothness measure to predict OOD generalization across test datasets. Our smoothness measures achieves strong correlation in image classification tasks but fails in natural language tasks.

| Task | Measure    | CNN          |             |              |              | RoBERTa      |             |              |              |
|------|------------|--------------|-------------|--------------|--------------|--------------|-------------|--------------|--------------|
|      |            | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ | $R^2$        | MAE         | Macro $\tau$ | Micro $\tau$ |
| SA   | NI-SSMBA   | <b>0.662</b> | 1.93        | <b>0.677</b> | <b>0.689</b> | <b>0.972</b> | 1.29        | <b>0.832</b> | <b>0.829</b> |
|      | NI-EDA     | 0.641        | 2.04        | 0.664        | 0.649        | 0.968        | 1.45        | 0.830        | 0.810        |
|      | NI-BT      | 0.550        | 2.99        | 0.592        | 0.501        | 0.961        | 1.47        | 0.813        | 0.801        |
|      | NI-RandRep | 0.409        | 2.64        | 0.544        | 0.554        | 0.967        | <b>1.27</b> | 0.821        | 0.816        |
|      | NE-SSMBA   | 0.595        | 2.53        | 0.708        | 0.713        | 0.971        | 1.32        | 0.830        | 0.824        |
|      | NE-EDA     | 0.534        | 2.71        | 0.698        | 0.674        | 0.965        | 1.55        | 0.825        | 0.801        |
|      | NE-BT      | 0.471        | 3.59        | 0.618        | 0.541        | 0.961        | 1.46        | 0.813        | 0.799        |
|      | NE-RandRep | 0.283        | 3.37        | 0.570        | 0.552        | 0.964        | 1.34        | 0.818        | 0.809        |
|      | ATC-NE     | 0.530        | 3.80        | 0.506        | 0.642        | 0.849        | 3.59        | 0.684        | 0.706        |
|      | ATC-MC     | 0.528        | 3.76        | 0.507        | 0.642        | 0.863        | 3.54        | 0.698        | 0.716        |
| NLI  | NI-SSMBA   | 0.575        | 2.09        | 0.570        | <b>0.534</b> | 0.933        | 1.19        | 0.750        | 0.730        |
|      | NI-EDA     | 0.577        | <b>2.04</b> | 0.581        | 0.511        | 0.941        | 1.26        | <b>0.789</b> | <b>0.757</b> |
|      | NI-BT      | 0.509        | 2.11        | 0.470        | 0.449        | 0.944        | 1.07        | 0.759        | 0.740        |
|      | NI-RandRep | 0.451        | 2.20        | 0.452        | 0.428        | 0.890        | 1.70        | 0.688        | 0.647        |
|      | NE-SSMBA   | 0.588        | 2.20        | 0.579        | 0.520        | 0.941        | 1.39        | 0.738        | 0.711        |
|      | NE-EDA     | <b>0.606</b> | 2.11        | <b>0.597</b> | 0.512        | 0.937        | 1.48        | 0.767        | 0.732        |
|      | NE-BT      | 0.536        | 2.27        | 0.480        | 0.422        | <b>0.954</b> | <b>1.12</b> | 0.764        | 0.750        |
|      | NE-RandRep | 0.457        | 2.36        | 0.451        | 0.397        | 0.904        | 1.79        | 0.665        | 0.591        |
|      | ATC-NE     | 0.378        | 3.57        | 0.430        | 0.294        | 0.673        | 2.35        | 0.536        | 0.52         |
|      | ATC-MC     | 0.382        | 3.57        | 0.433        | 0.297        | 0.718        | 2.21        | 0.570        | 0.556        |

Table 8: Correlation metrics evaluating the quality of our neighborhood invariance measure on two tasks, sentiment analysis and natural language inference, and two architectures, CNN and RoBERTa. Details on metric calculations and baselines are provided in sections 4.3 and 4.5. This full table of results includes metrics calculated with neighborhood negative entropy measure as well.

| Measure      | Test Domain  |                 |              |              |              |              |              |
|--------------|--------------|-----------------|--------------|--------------|--------------|--------------|--------------|
|              | ImageNetV2   | Imagenet-Sketch | ObjectNet    | ImageNet-Vid | YTB-B        | ImageNet-R   | ImageNet-A   |
| NI-RandAug   | 0.810        | 0.641           | <b>0.613</b> | <b>0.767</b> | <b>0.570</b> | <b>0.763</b> | 0.577        |
| NI-Translate | 0.794        | 0.436           | 0.461        | 0.642        | 0.395        | 0.560        | 0.468        |
| NI-Erase     | 0.715        | 0.222           | 0.384        | 0.635        | 0.399        | 0.398        | 0.446        |
| NI-FC        | 0.767        | 0.406           | 0.516        | 0.706        | 0.446        | 0.618        | <b>0.679</b> |
| ATC-MC       | 0.980        | <b>0.710</b>    | 0.451        | 0.552        | 0.200        | 0.463        | 0.159        |
| ATC-NE       | <b>0.991</b> | 0.709           | 0.392        | 0.484        | 0.229        | 0.441        | 0.209        |

Table 9: Full  $R^2$  metrics for all ImageNet test domains. We average values across models.

| Model | Train Domain | Measure      | Test Domain |               |              |
|-------|--------------|--------------|-------------|---------------|--------------|
|       |              |              | SVHN        | Colored MNIST | MNIST        |
| NiN   | SVHN         | NI-RandAug   | —           | <b>0.785</b>  | 0.744        |
|       |              | NI-Translate | —           | 0.728         | 0.642        |
|       |              | NI-Erase     | —           | 0.088         | 0.218        |
|       |              | NI-FC        | —           | 0.134         | 0.283        |
|       |              | ATC-NE       | —           | 0.506         | 0.725        |
|       |              | ATC-MC       | —           | 0.615         | <b>0.769</b> |
|       |              | ATC-NE       | —           | 0.506         | 0.725        |
|       |              | ATC-MC       | —           | 0.615         | <b>0.769</b> |

Table 10: Full  $R^2$  metrics for all test domains for image classification models on SVHN, Colored MNIST, and MNIST.

| Model  | Train Domain | Measure      | Test Domain  |              |              |              |
|--------|--------------|--------------|--------------|--------------|--------------|--------------|
|        |              |              | CIFAR10      | CINIC10      | CIFAR10.1    | CIFAR10.2    |
| NiN    | CIFAR10      | NI-RandAug   | —            | <b>0.898</b> | <b>0.927</b> | <b>0.876</b> |
|        |              | NI-Translate | —            | 0.688        | 0.864        | 0.730        |
|        |              | NI-Erase     | —            | 0.404        | 0.526        | 0.547        |
|        |              | NI-FC        | —            | 0.109        | 0.028        | 0.000        |
|        |              | ATC-NE       | —            | 0.237        | 0.575        | 0.435        |
|        |              | ATC-MC       | —            | 0.252        | 0.538        | 0.390        |
|        |              | ATC-NE       | —            | 0.816        | 0.844        | <b>0.853</b> |
|        |              | ATC-MC       | —            | 0.702        | 0.760        | 0.680        |
| ResNet | CIFAR10      | NI-RandAug   | —            | <b>0.889</b> | <b>0.899</b> | 0.836        |
|        |              | NI-Translate | —            | 0.807        | 0.812        | 0.783        |
|        |              | NI-Erase     | —            | 0.726        | 0.628        | 0.660        |
|        |              | NI-FC        | —            | 0.736        | 0.782        | 0.693        |
|        |              | ATC-NE       | —            | 0.702        | 0.760        | 0.680        |
|        |              | ATC-MC       | —            | 0.736        | 0.782        | 0.693        |
|        |              | ATC-NE       | —            | <b>0.969</b> | <b>0.950</b> | <b>0.929</b> |
|        |              | ATC-MC       | —            | 0.849        | 0.844        | 0.809        |
| VGG    | CIFAR10      | NI-RandAug   | —            | <b>0.969</b> | <b>0.950</b> | <b>0.929</b> |
|        |              | NI-Translate | —            | 0.096        | 0.100        | 0.104        |
|        |              | NI-Erase     | —            | 0.637        | 0.524        | 0.487        |
|        |              | NI-FC        | —            | 0.557        | 0.774        | 0.724        |
|        |              | ATC-NE       | —            | 0.559        | 0.764        | 0.709        |
|        |              | ATC-MC       | —            | 0.559        | 0.764        | 0.709        |
|        |              | ATC-NE       | —            | <b>0.922</b> | <b>0.876</b> | <b>0.865</b> |
|        |              | ATC-MC       | —            | 0.516        | 0.449        | 0.407        |
| CNN    | CINIC10      | NI-RandAug   | <b>0.922</b> | —            | <b>0.876</b> | <b>0.865</b> |
|        |              | NI-Translate | 0.516        | —            | 0.449        | 0.407        |
|        |              | NI-Erase     | 0.603        | —            | 0.504        | 0.548        |
|        |              | NI-FC        | 0.416        | —            | 0.397        | 0.395        |
|        |              | ATC-NE       | 0.869        | —            | 0.750        | 0.724        |
|        |              | ATC-MC       | 0.868        | —            | 0.741        | 0.716        |
|        |              | ATC-NE       | 0.869        | —            | 0.750        | 0.724        |
|        |              | ATC-MC       | 0.868        | —            | 0.741        | 0.716        |

Table 11: Full  $R^2$  metrics for all test domains for image classification models on CIFAR10, CINIC10, CIFAR10.1, and CIFAR10.2.

| Train Domain | Measure   | Test Domain |          |       |        |        |       |        |       |       |       |
|--------------|-----------|-------------|----------|-------|--------|--------|-------|--------|-------|-------|-------|
|              |           | books       | clothing | home  | kindle | movies | pets  | sports | tech  | tools | toys  |
| books        | NS-SSMBA  | 0.688       | 0.771    | 0.752 | 0.683  | 0.823  | —     | 0.762  | 0.697 | 0.711 | 0.729 |
|              | N-EIDA    | —           | 0.720    | 0.725 | 0.699  | 0.689  | 0.810 | 0.693  | 0.663 | 0.727 | 0.805 |
|              | N-BT      | —           | 0.560    | 0.663 | 0.639  | 0.593  | 0.675 | 0.602  | 0.591 | 0.592 | 0.658 |
|              | N-RandRep | —           | 0.413    | 0.418 | 0.331  | 0.343  | 0.479 | 0.373  | 0.304 | 0.401 | 0.496 |
| clothing     | ATC-NC    | 0.765       | 0.859    | 0.828 | 0.777  | 0.834  | 0.844 | 0.791  | 0.814 | 0.872 | —     |
|              | ATC-MC    | —           | 0.759    | 0.854 | 0.821  | 0.771  | 0.828 | 0.830  | 0.790 | 0.808 | 0.871 |
|              | NS-SSMBA  | 0.517       | —        | 0.601 | 0.531  | 0.525  | 0.535 | 0.566  | 0.484 | 0.549 | 0.631 |
|              | N-EIDA    | 0.409       | —        | 0.419 | 0.452  | 0.408  | 0.395 | 0.457  | 0.388 | 0.486 | 0.534 |
| home         | N-BT      | 0.234       | —        | 0.164 | 0.267  | 0.273  | 0.121 | 0.391  | 0.164 | 0.193 | 0.355 |
|              | N-RandRep | 0.236       | —        | 0.237 | 0.223  | 0.250  | 0.212 | 0.213  | 0.156 | 0.246 | 0.335 |
|              | ATC-NC    | 0.480       | —        | 0.673 | 0.478  | 0.467  | 0.466 | 0.739  | 0.299 | 0.772 | 0.847 |
|              | ATC-MC    | 0.477       | —        | 0.675 | 0.477  | 0.466  | 0.474 | 0.742  | 0.300 | 0.772 | 0.848 |
| kindle       | NS-SSMBA  | 0.533       | 0.545    | —     | 0.524  | 0.511  | 0.665 | 0.648  | 0.576 | 0.604 | 0.592 |
|              | N-EIDA    | 0.412       | 0.554    | —     | 0.451  | 0.433  | 0.586 | 0.496  | 0.417 | 0.525 | 0.657 |
|              | N-BT      | 0.313       | 0.390    | —     | 0.327  | 0.316  | 0.346 | 0.366  | 0.304 | 0.385 | 0.408 |
|              | N-RandRep | 0.273       | 0.260    | —     | 0.236  | 0.325  | 0.286 | 0.299  | 0.270 | 0.282 | 0.370 |
| movies       | ATC-NC    | 0.608       | 0.861    | —     | 0.675  | 0.618  | 0.838 | 0.838  | 0.720 | 0.863 | 0.894 |
|              | ATC-MC    | 0.612       | 0.861    | —     | 0.679  | 0.626  | 0.838 | 0.839  | 0.719 | 0.863 | 0.895 |
|              | NS-SSMBA  | 0.645       | 0.680    | 0.650 | —      | 0.626  | 0.765 | 0.659  | 0.553 | 0.556 | 0.634 |
|              | N-EIDA    | 0.722       | 0.688    | 0.695 | —      | 0.662  | 0.785 | 0.690  | 0.658 | 0.623 | 0.783 |
| pets         | N-BT      | 0.655       | 0.674    | 0.684 | —      | 0.625  | 0.745 | 0.704  | 0.695 | 0.649 | 0.735 |
|              | N-RandRep | 0.325       | 0.364    | 0.253 | —      | 0.369  | 0.442 | 0.392  | 0.244 | 0.274 | 0.445 |
|              | ATC-NC    | 0.747       | 0.659    | 0.701 | —      | 0.690  | 0.792 | 0.717  | 0.765 | 0.610 | 0.776 |
|              | ATC-MC    | 0.759       | 0.642    | 0.699 | —      | 0.687  | 0.784 | 0.708  | 0.507 | 0.594 | 0.765 |
| sports       | NS-SSMBA  | 0.541       | 0.640    | 0.658 | 0.615  | —      | 0.633 | 0.656  | 0.583 | 0.653 | 0.741 |
|              | N-EIDA    | 0.542       | 0.662    | 0.571 | 0.676  | —      | 0.629 | 0.594  | 0.574 | 0.636 | 0.789 |
|              | N-BT      | 0.692       | 0.679    | 0.667 | 0.653  | —      | 0.723 | 0.739  | 0.700 | 0.709 | 0.754 |
|              | N-RandRep | 0.194       | 0.276    | 0.281 | 0.279  | —      | 0.229 | 0.316  | 0.216 | 0.316 | 0.369 |
| tech         | ATC-NC    | 0.698       | 0.719    | 0.708 | 0.668  | —      | 0.718 | 0.707  | 0.611 | 0.752 | 0.818 |
|              | ATC-MC    | 0.707       | 0.725    | 0.713 | 0.664  | —      | 0.732 | 0.711  | 0.624 | 0.755 | 0.820 |
|              | NS-SSMBA  | 0.458       | 0.543    | 0.578 | 0.529  | 0.548  | —     | 0.606  | 0.528 | 0.590 | 0.721 |
|              | N-EIDA    | 0.426       | 0.552    | 0.554 | 0.467  | 0.513  | —     | 0.546  | 0.463 | 0.549 | 0.677 |
| tools        | N-BT      | 0.485       | 0.523    | 0.477 | 0.549  | 0.563  | —     | 0.581  | 0.550 | 0.580 | 0.587 |
|              | N-RandRep | 0.260       | 0.333    | 0.284 | 0.302  | 0.320  | —     | 0.304  | 0.271 | 0.345 | 0.432 |
|              | ATC-NC    | 0.611       | 0.851    | 0.849 | 0.651  | 0.680  | —     | 0.825  | 0.591 | 0.812 | 0.883 |
|              | ATC-MC    | 0.614       | 0.846    | 0.848 | 0.648  | 0.683  | —     | 0.824  | 0.589 | 0.810 | 0.882 |
| toys         | NS-SSMBA  | 0.499       | 0.520    | 0.569 | 0.524  | 0.463  | 0.573 | —      | 0.517 | 0.590 | 0.721 |
|              | N-EIDA    | 0.464       | 0.514    | 0.460 | 0.489  | 0.381  | 0.508 | —      | 0.448 | 0.541 | 0.594 |
|              | N-BT      | 0.406       | 0.295    | 0.334 | 0.383  | 0.381  | 0.325 | —      | 0.381 | 0.393 | 0.411 |
|              | N-RandRep | 0.283       | 0.173    | 0.212 | 0.256  | 0.219  | 0.239 | —      | 0.204 | 0.231 | 0.331 |
| tech         | ATC-NC    | 0.712       | 0.900    | 0.926 | 0.741  | 0.956  | 0.981 | —      | 0.828 | 0.931 | 0.953 |
|              | ATC-MC    | 0.723       | 0.897    | 0.926 | 0.754  | 0.965  | 0.896 | —      | 0.833 | 0.923 | 0.952 |
|              | NS-SSMBA  | 0.610       | 0.598    | 0.635 | 0.620  | 0.596  | 0.604 | 0.636  | —     | 0.545 | 0.848 |
|              | N-EIDA    | 0.550       | 0.475    | 0.519 | 0.603  | 0.608  | 0.581 | 0.587  | —     | 0.532 | 0.649 |
| toys         | N-BT      | 0.588       | 0.437    | 0.572 | 0.632  | 0.597  | 0.558 | 0.603  | —     | 0.523 | 0.648 |
|              | N-RandRep | 0.340       | 0.241    | 0.262 | 0.307  | 0.337  | 0.285 | 0.291  | —     | 0.200 | 0.432 |
|              | ATC-NC    | 0.766       | 0.820    | 0.904 | 0.791  | 0.817  | 0.866 | 0.874  | —     | 0.882 | 0.893 |
|              | ATC-MC    | 0.771       | 0.823    | 0.904 | 0.794  | 0.817  | 0.864 | 0.875  | —     | 0.882 | 0.893 |
| tech         | NS-SSMBA  | 0.554       | 0.505    | 0.548 | 0.556  | 0.605  | 0.651 | 0.606  | 0.577 | —     | 0.660 |
|              | N-EIDA    | 0.466       | 0.443    | 0.469 | 0.459  | 0.368  | 0.545 | 0.512  | 0.405 | —     | 0.293 |
|              | N-BT      | 0.451       | 0.385    | 0.447 | 0.476  | 0.482  | 0.501 | 0.483  | 0.484 | —     | 0.462 |
|              | N-RandRep | 0.323       | 0.241    | 0.204 | 0.306  | 0.399  | 0.316 | 0.253  | 0.227 | —     | 0.297 |
| toys         | ATC-NC    | 0.661       | 0.875    | 0.901 | 0.679  | 0.719  | 0.867 | 0.968  | 0.795 | —     | 0.916 |
|              | ATC-MC    | 0.670       | 0.874    | 0.903 | 0.684  | 0.729  | 0.866 | 0.969  | 0.801 | —     | 0.916 |
|              | NS-SSMBA  | 0.625       | 0.693    | 0.656 | 0.620  | 0.643  | 0.661 | 0.708  | 0.618 | 0.650 | —     |
|              | N-EIDA    | 0.473       | 0.582    | 0.487 | 0.481  | 0.498  | 0.552 | 0.535  | 0.429 | 0.528 | —     |
| tech         | N-BT      | 0.311       | 0.408    | 0.331 | 0.351  | 0.324  | 0.355 | 0.420  | 0.334 | 0.357 | —     |
|              | N-RandRep | 0.272       | 0.197    | 0.234 | 0.254  | 0.286  | 0.257 | 0.301  | 0.215 | 0.218 | —     |
|              | ATC-NC    | 0.686       | 0.936    | 0.935 | 0.742  | 0.713  | 0.838 | 0.885  | 0.561 | 0.906 | —     |
|              | ATC-MC    | 0.691       | 0.935    | 0.858 | 0.742  | 0.718  | 0.843 | 0.886  | 0.574 | 0.869 | —     |

Table 12: Full  $R^2$  metrics for all pairs of training and test domains for CNN models trained on AWS.

| Train Domain | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|              |            | books        | clothing     | home         | kindle       | movies       | pets         | sports       | tech         | tools        | toys         |
| books        | NL-SSMBA   | —            | <b>0.973</b> | <b>0.983</b> | <b>0.987</b> | <b>0.963</b> | <b>0.973</b> | <b>0.985</b> | <b>0.977</b> | <b>0.968</b> |              |
|              | NL-EDA     | —            | 0.961        | 0.970        | 0.982        | <b>0.966</b> | 0.966        | 0.970        | 0.957        | <b>0.978</b> | <b>0.981</b> |
|              | NL-BT      | —            | 0.946        | 0.979        | 0.985        | 0.961        | 0.972        | 0.972        | 0.975        | 0.970        | 0.955        |
|              | NL-RandRep | —            | 0.956        | 0.970        | 0.981        | 0.958        | 0.967        | 0.970        | 0.973        | 0.952        | 0.961        |
|              | ATC-NE     | —            | 0.845        | 0.861        | 0.957        | 0.927        | 0.940        | 0.852        | 0.925        | 0.853        | 0.876        |
|              | ATC-MC     | —            | 0.838        | 0.842        | 0.947        | 0.924        | 0.929        | 0.840        | 0.927        | 0.845        | 0.876        |
| clothing     | NL-SSMBA   | 0.944        | —            | 0.974        | <b>0.942</b> | 0.980        | 0.981        | 0.984        | 0.973        | 0.969        | 0.978        |
|              | NL-EDA     | <b>0.974</b> | —            | 0.970        | 0.922        | 0.955        | 0.978        | 0.961        | 0.971        | 0.953        | 0.972        |
|              | NL-BT      | 0.944        | —            | <b>0.977</b> | 0.921        | 0.971        | <b>0.983</b> | 0.982        | 0.980        | 0.970        | <b>0.993</b> |
|              | NL-RandRep | 0.950        | —            | 0.965        | 0.933        | <b>0.982</b> | 0.977        | <b>0.985</b> | <b>0.981</b> | <b>0.974</b> | <b>0.973</b> |
|              | ATC-NE     | 0.792        | —            | 0.764        | 0.764        | 0.913        | 0.939        | 0.933        | 0.846        | 0.914        | 0.973        |
|              | ATC-MC     | 0.801        | —            | 0.945        | 0.789        | 0.917        | 0.948        | 0.933        | 0.887        | 0.926        | 0.971        |
| home         | NL-SSMBA   | 0.973        | 0.972        | —            | <b>0.976</b> | <b>0.972</b> | <b>0.979</b> | 0.976        | <b>0.980</b> | 0.972        | <b>0.984</b> |
|              | NL-EDA     | 0.961        | 0.975        | —            | 0.967        | 0.959        | 0.960        | 0.949        | 0.917        | 0.963        | 0.979        |
|              | NL-BT      | 0.947        | 0.958        | —            | 0.944        | 0.947        | 0.948        | <b>0.977</b> | 0.972        | <b>0.971</b> | 0.974        |
|              | NL-RandRep | <b>0.978</b> | <b>0.976</b> | —            | 0.970        | 0.967        | 0.975        | <b>0.973</b> | 0.964        | <b>0.975</b> | 0.983        |
|              | ATC-NE     | 0.878        | 0.959        | —            | 0.958        | 0.908        | 0.930        | 0.962        | 0.860        | 0.928        | 0.951        |
|              | ATC-MC     | 0.885        | 0.960        | —            | 0.963        | 0.913        | 0.945        | 0.964        | 0.872        | 0.938        | 0.952        |
| kindle       | NL-SSMBA   | <b>0.992</b> | 0.963        | 0.966        | —            | 0.961        | 0.957        | <b>0.963</b> | 0.971        | 0.943        | <b>0.984</b> |
|              | NL-EDA     | 0.983        | <b>0.968</b> | <b>0.951</b> | —            | 0.966        | <b>0.962</b> | 0.953        | <b>0.974</b> | <b>0.963</b> | 0.969        |
|              | NL-BT      | 0.973        | 0.947        | <b>0.975</b> | —            | 0.954        | 0.949        | 0.954        | 0.973        | 0.923        | 0.973        |
|              | NL-RandRep | 0.979        | 0.946        | 0.956        | —            | <b>0.967</b> | 0.961        | 0.946        | 0.970        | 0.934        | 0.975        |
|              | ATC-NE     | 0.931        | 0.380        | 0.510        | —            | 0.861        | 0.162        | 0.260        | 0.424        | 0.179        | 0.608        |
|              | ATC-MC     | 0.930        | 0.534        | 0.623        | —            | 0.871        | 0.294        | 0.392        | 0.551        | 0.316        | 0.672        |
| movies       | NL-SSMBA   | <b>0.984</b> | <b>0.976</b> | <b>0.969</b> | 0.981        | —            | <b>0.962</b> | <b>0.983</b> | <b>0.958</b> | 0.961        | 0.971        |
|              | NL-EDA     | 0.974        | 0.975        | 0.947        | <b>0.985</b> | —            | 0.928        | 0.977        | 0.950        | <b>0.972</b> | 0.969        |
|              | NL-BT      | 0.971        | 0.952        | 0.965        | 0.965        | —            | 0.951        | 0.970        | 0.957        | 0.950        | <b>0.977</b> |
|              | NL-RandRep | 0.981        | 0.969        | 0.954        | 0.971        | —            | 0.958        | 0.969        | 0.957        | 0.958        | 0.962        |
|              | ATC-NE     | 0.948        | 0.864        | 0.902        | 0.949        | —            | 0.746        | 0.908        | 0.813        | 0.817        | 0.917        |
|              | ATC-MC     | 0.949        | 0.893        | 0.928        | 0.948        | —            | 0.802        | 0.929        | 0.861        | 0.859        | 0.926        |
| pets         | NL-SSMBA   | 0.943        | 0.974        | 0.981        | 0.945        | 0.942        | —            | 0.979        | <b>0.982</b> | 0.978        | 0.969        |
|              | NL-EDA     | <b>0.965</b> | <b>0.980</b> | <b>0.985</b> | <b>0.958</b> | <b>0.952</b> | —            | <b>0.983</b> | 0.975        | <b>0.976</b> | <b>0.983</b> |
|              | NL-BT      | 0.936        | 0.971        | 0.974        | 0.931        | 0.926        | —            | 0.980        | 0.963        | 0.979        | 0.977        |
|              | NL-RandRep | 0.941        | 0.976        | 0.971        | 0.925        | 0.931        | —            | 0.976        | 0.968        | 0.976        | 0.979        |
|              | ATC-NE     | 0.594        | 0.967        | 0.900        | 0.577        | 0.862        | —            | 0.947        | 0.959        | 0.929        | 0.882        |
|              | ATC-MC     | 0.611        | 0.968        | 0.909        | 0.589        | 0.851        | —            | 0.955        | 0.959        | 0.939        | 0.894        |
| sports       | NL-SSMBA   | 0.979        | 0.982        | <b>0.994</b> | <b>0.980</b> | 0.979        | 0.970        | —            | 0.982        | 0.984        | 0.978        |
|              | NL-EDA     | 0.979        | <b>0.987</b> | 0.986        | 0.960        | <b>0.984</b> | <b>0.985</b> | —            | <b>0.985</b> | <b>0.986</b> | 0.987        |
|              | NL-BT      | 0.966        | 0.977        | 0.989        | 0.949        | 0.969        | 0.979        | —            | <b>0.986</b> | <b>0.984</b> | <b>0.988</b> |
|              | NL-RandRep | <b>0.981</b> | 0.979        | <b>0.994</b> | 0.978        | 0.980        | 0.978        | —            | <b>0.985</b> | <b>0.985</b> | 0.984        |
|              | ATC-NE     | 0.655        | 0.945        | 0.963        | 0.688        | 0.850        | 0.890        | —            | 0.938        | 0.969        | 0.957        |
|              | ATC-MC     | 0.705        | 0.951        | 0.965        | 0.722        | 0.882        | 0.905        | —            | 0.944        | 0.972        | 0.953        |
| tech         | NL-SSMBA   | <b>0.944</b> | 0.979        | 0.977        | <b>0.947</b> | <b>0.967</b> | 0.958        | 0.975        | —            | <b>0.990</b> | 0.981        |
|              | NL-EDA     | 0.947        | 0.969        | 0.971        | 0.925        | 0.953        | 0.937        | 0.967        | —            | 0.963        | 0.972        |
|              | NL-BT      | 0.964        | <b>0.982</b> | <b>0.987</b> | 0.924        | 0.937        | 0.933        | <b>0.978</b> | —            | 0.987        | 0.972        |
|              | NL-RandRep | 0.929        | 0.959        | 0.977        | 0.918        | 0.952        | 0.944        | 0.963        | —            | 0.984        | 0.971        |
|              | ATC-NE     | 0.937        | 0.939        | 0.983        | 0.933        | 0.946        | <b>0.961</b> | 0.961        | —            | 0.974        | <b>0.981</b> |
|              | ATC-MC     | 0.941        | 0.941        | 0.980        | 0.928        | 0.955        | 0.958        | 0.963        | —            | 0.971        | <b>0.982</b> |
| tools        | NL-SSMBA   | <b>0.977</b> | <b>0.988</b> | 0.975        | <b>0.967</b> | <b>0.971</b> | 0.986        | <b>0.985</b> | <b>0.978</b> | —            | 0.983        |
|              | NL-EDA     | 0.975        | 0.985        | <b>0.978</b> | 0.969        | <b>0.980</b> | 0.986        | <b>0.988</b> | 0.975        | —            | 0.982        |
|              | NL-BT      | 0.940        | 0.976        | 0.970        | 0.941        | 0.944        | 0.969        | 0.978        | 0.948        | —            | <b>0.975</b> |
|              | NL-RandRep | 0.963        | 0.985        | <b>0.978</b> | 0.954        | 0.963        | <b>0.988</b> | 0.984        | 0.977        | —            | <b>0.987</b> |
|              | ATC-NE     | 0.857        | 0.945        | 0.964        | 0.883        | 0.865        | 0.925        | 0.964        | 0.963        | —            | 0.922        |
|              | ATC-MC     | 0.868        | 0.941        | 0.965        | 0.887        | 0.876        | 0.936        | 0.967        | 0.971        | —            | 0.919        |
| toys         | NL-SSMBA   | 0.955        | <b>0.971</b> | <b>0.985</b> | <b>0.980</b> | <b>0.961</b> | 0.966        | <b>0.988</b> | <b>0.978</b> | <b>0.971</b> | —            |
|              | NL-EDA     | <b>0.965</b> | 0.968        | 0.984        | 0.970        | <b>0.985</b> | <b>0.973</b> | 0.981        | 0.975        | <b>0.983</b> | —            |
|              | NL-BT      | 0.890        | 0.946        | 0.959        | 0.944        | 0.925        | 0.935        | 0.965        | 0.943        | 0.934        | —            |
|              | NL-RandRep | 0.959        | 0.970        | 0.979        | 0.960        | 0.965        | 0.967        | 0.977        | <b>0.978</b> | 0.974        | —            |
|              | ATC-NE     | 0.883        | 0.878        | 0.885        | 0.763        | 0.906        | 0.798        | 0.938        | 0.815        | 0.899        | —            |
|              | ATC-MC     | 0.889        | 0.897        | 0.895        | 0.754        | 0.920        | 0.818        | 0.947        | 0.833        | 0.914        | —            |

Table 13: Full  $R^2$  metrics for all pairs of training and test domains for BERT models trained on AWS.

| Train Domain | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|              |            | slate        | verbatim     | facetoface   | oup          | nineteen     | fiction      | telephone    | travel       | letters      | government   |
| slate        | NI-SSMBA   | —            | 0.457        | 0.760        | 0.544        | 0.642        | 0.706        | 0.727        | 0.585        | 0.640        | 0.714        |
|              | NI-EDA     | —            | 0.574        | 0.723        | 0.565        | 0.620        | 0.708        | 0.731        | 0.549        | 0.595        | 0.706        |
|              | NI-BT      | —            | <b>0.862</b> | <b>0.920</b> | <b>0.942</b> | <b>0.930</b> | <b>0.913</b> | <b>0.878</b> | <b>0.936</b> | <b>0.949</b> | <b>0.940</b> |
|              | NI-RandRep | —            | 0.242        | 0.336        | 0.501        | 0.415        | 0.121        | 0.170        | 0.301        | 0.652        | 0.470        |
|              | ATC-NE     | —            | 0.681        | 0.677        | 0.594        | 0.599        | 0.741        | 0.669        | 0.664        | 0.661        | 0.773        |
|              | ATC-MC     | —            | 0.682        | 0.675        | 0.590        | 0.600        | 0.737        | 0.670        | 0.665        | 0.661        | 0.771        |
| fiction      | NI-SSMBA   | 0.581        | 0.492        | 0.765        | 0.451        | 0.530        | —            | 0.683        | 0.499        | 0.502        | 0.614        |
|              | NI-EDA     | 0.657        | 0.601        | 0.728        | 0.620        | 0.530        | —            | 0.676        | 0.527        | 0.508        | 0.642        |
|              | NI-BT      | <b>0.889</b> | <b>0.887</b> | <b>0.917</b> | <b>0.950</b> | <b>0.921</b> | —            | <b>0.863</b> | <b>0.928</b> | <b>0.935</b> | <b>0.941</b> |
|              | NI-RandRep | 0.360        | 0.414        | 0.459        | 0.666        | 0.531        | —            | 0.188        | 0.464        | 0.680        | 0.631        |
|              | ATC-NE     | 0.530        | 0.425        | 0.718        | 0.323        | 0.536        | —            | 0.556        | 0.425        | 0.597        | 0.522        |
|              | ATC-MC     | 0.525        | 0.418        | 0.719        | 0.326        | 0.534        | —            | 0.555        | 0.422        | 0.592        | 0.521        |
| telephone    | NI-SSMBA   | 0.539        | 0.453        | 0.826        | 0.429        | 0.480        | 0.647        | —            | 0.436        | 0.559        | 0.439        |
|              | NI-EDA     | 0.677        | 0.636        | 0.804        | 0.492        | 0.639        | 0.819        | —            | 0.557        | 0.653        | 0.568        |
|              | NI-BT      | <b>0.882</b> | <b>0.901</b> | <b>0.928</b> | <b>0.936</b> | <b>0.927</b> | <b>0.912</b> | —            | <b>0.937</b> | <b>0.916</b> | <b>0.919</b> |
|              | NI-RandRep | 0.332        | 0.498        | 0.488        | 0.584        | 0.538        | 0.310        | —            | 0.586        | 0.677        | 0.576        |
|              | ATC-NE     | 0.557        | 0.518        | 0.762        | 0.527        | 0.467        | 0.695        | —            | 0.368        | 0.554        | 0.468        |
|              | ATC-MC     | 0.561        | 0.518        | 0.762        | 0.524        | 0.467        | 0.698        | —            | 0.372        | 0.560        | 0.470        |
| travel       | NI-SSMBA   | 0.503        | 0.506        | 0.588        | 0.419        | 0.501        | 0.543        | 0.597        | —            | 0.670        | 0.507        |
|              | NI-EDA     | 0.444        | 0.429        | 0.502        | 0.388        | 0.342        | 0.532        | 0.478        | —            | 0.438        | 0.377        |
|              | NI-BT      | <b>0.806</b> | <b>0.799</b> | <b>0.861</b> | <b>0.916</b> | <b>0.889</b> | <b>0.828</b> | <b>0.802</b> | —            | <b>0.923</b> | <b>0.907</b> |
|              | NI-RandRep | 0.315        | 0.333        | 0.385        | 0.643        | 0.516        | 0.182        | 0.224        | —            | 0.690        | 0.626        |
|              | ATC-NE     | 0.561        | 0.516        | 0.498        | 0.686        | 0.546        | 0.581        | 0.606        | —            | 0.585        | 0.598        |
|              | ATC-MC     | 0.563        | 0.517        | 0.503        | 0.690        | 0.544        | 0.588        | 0.611        | —            | 0.582        | 0.607        |
| government   | NI-SSMBA   | 0.592        | 0.497        | 0.618        | 0.572        | 0.596        | 0.648        | 0.593        | 0.539        | 0.676        | —            |
|              | NI-EDA     | 0.577        | 0.574        | 0.550        | 0.476        | 0.618        | 0.600        | 0.491        | 0.518        | 0.526        | —            |
|              | NI-BT      | <b>0.818</b> | <b>0.812</b> | <b>0.859</b> | <b>0.909</b> | <b>0.893</b> | <b>0.813</b> | <b>0.809</b> | <b>0.909</b> | <b>0.917</b> | —            |
|              | NI-RandRep | 0.010        | 0.083        | 0.207        | 0.306        | 0.279        | 0.011        | 0.027        | 0.123        | 0.360        | —            |
|              | ATC-NE     | 0.663        | 0.405        | 0.564        | 0.722        | 0.337        | 0.634        | 0.630        | 0.509        | 0.653        | —            |
|              | ATC-MC     | 0.666        | 0.405        | 0.574        | 0.721        | 0.341        | 0.633        | 0.636        | 0.509        | 0.654        | —            |

Table 14: Full  $R^2$  metrics for all pairs of training and test domains for CNN models trained on MNLI.

| Train Domain | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|              |            | slate        | verbatim     | facetoface   | oup          | nineteen     | fiction      | telephone    | travel       | letters      | government   |
| slate        | NI-SSMBA   | —            | <b>0.920</b> | 0.905        | 0.943        | 0.916        | 0.929        | <b>0.930</b> | <b>0.957</b> | 0.931        | <b>0.970</b> |
|              | NI-EDA     | —            | 0.701        | 0.754        | 0.816        | 0.784        | 0.790        | 0.708        | 0.787        | 0.820        | 0.837        |
|              | NI-BT      | —            | 0.862        | <b>0.920</b> | 0.942        | <b>0.930</b> | 0.913        | 0.878        | 0.936        | <b>0.949</b> | 0.940        |
|              | NI-RandRep | —            | 0.242        | 0.336        | 0.501        | 0.415        | 0.121        | 0.170        | 0.301        | 0.652        | 0.470        |
| fiction      | ATC-NE     | —            | 0.878        | 0.899        | <b>0.954</b> | 0.911        | 0.925        | 0.911        | 0.904        | 0.890        | 0.951        |
|              | ATC-MC     | —            | 0.871        | 0.884        | 0.953        | 0.908        | <b>0.932</b> | 0.914        | 0.911        | 0.887        | 0.949        |
|              | NI-SSMBA   | <b>0.954</b> | <b>0.928</b> | 0.895        | <b>0.952</b> | <b>0.946</b> | —            | <b>0.952</b> | <b>0.975</b> | <b>0.946</b> | <b>0.954</b> |
|              | NI-EDA     | 0.661        | 0.687        | 0.699        | 0.840        | 0.686        | —            | 0.683        | 0.764        | 0.799        | 0.797        |
| telephone    | NI-BT      | 0.889        | 0.887        | <b>0.917</b> | 0.950        | 0.921        | —            | 0.863        | 0.928        | 0.935        | 0.941        |
|              | NI-RandRep | 0.360        | 0.414        | 0.459        | 0.666        | 0.531        | —            | 0.188        | 0.464        | 0.680        | 0.631        |
|              | ATC-NE     | 0.808        | 0.585        | 0.881        | 0.726        | 0.939        | —            | 0.836        | 0.826        | 0.850        | 0.890        |
|              | ATC-MC     | 0.817        | 0.652        | 0.889        | 0.732        | 0.936        | —            | 0.841        | 0.830        | 0.861        | 0.892        |
| travel       | NI-SSMBA   | <b>0.943</b> | <b>0.962</b> | <b>0.948</b> | <b>0.953</b> | 0.917        | <b>0.954</b> | —            | 0.921        | <b>0.934</b> | <b>0.956</b> |
|              | NI-EDA     | 0.776        | 0.787        | 0.790        | 0.852        | 0.823        | 0.847        | —            | 0.853        | 0.877        | 0.852        |
|              | NI-BT      | 0.882        | 0.901        | 0.928        | 0.936        | <b>0.927</b> | 0.912        | —            | <b>0.937</b> | 0.916        | 0.919        |
|              | NI-RandRep | 0.332        | 0.498        | 0.584        | 0.538        | 0.538        | 0.310        | —            | 0.586        | 0.677        | 0.576        |
| government   | ATC-NE     | 0.662        | 0.576        | 0.884        | 0.904        | 0.762        | 0.665        | —            | 0.771        | 0.856        | 0.889        |
|              | ATC-MC     | 0.714        | 0.668        | 0.902        | 0.924        | 0.807        | 0.732        | —            | 0.795        | 0.857        | 0.899        |
|              | NI-SSMBA   | <b>0.938</b> | <b>0.935</b> | <b>0.949</b> | <b>0.924</b> | <b>0.948</b> | <b>0.946</b> | <b>0.958</b> | —            | <b>0.962</b> | <b>0.953</b> |
|              | NI-EDA     | 0.538        | 0.553        | 0.471        | 0.711        | 0.602        | 0.638        | 0.533        | —            | 0.755        | 0.699        |
| government   | NI-BT      | 0.806        | 0.799        | 0.861        | 0.916        | 0.889        | 0.828        | 0.802        | —            | 0.923        | 0.907        |
|              | NI-RandRep | 0.315        | 0.333        | 0.385        | 0.643        | 0.516        | 0.182        | 0.224        | —            | 0.690        | 0.626        |
|              | ATC-NE     | 0.319        | 0.417        | 0.500        | 0.889        | 0.768        | 0.396        | 0.494        | —            | 0.928        | 0.911        |
|              | ATC-MC     | 0.412        | 0.506        | 0.591        | 0.906        | 0.789        | 0.525        | 0.572        | —            | 0.921        | 0.917        |
| government   | NI-SSMBA   | <b>0.916</b> | <b>0.960</b> | 0.781        | <b>0.919</b> | 0.816        | <b>0.882</b> | <b>0.914</b> | <b>0.950</b> | <b>0.920</b> | —            |
|              | NI-EDA     | 0.484        | 0.572        | 0.544        | 0.728        | 0.715        | 0.655        | 0.566        | 0.644        | 0.681        | —            |
|              | NI-BT      | 0.818        | 0.812        | <b>0.859</b> | 0.909        | <b>0.893</b> | 0.813        | 0.809        | 0.909        | 0.917        | —            |
|              | NI-RandRep | 0.010        | 0.083        | 0.207        | 0.306        | 0.279        | 0.011        | 0.027        | 0.123        | 0.360        | —            |
| government   | ATC-NE     | 0.544        | 0.385        | 0.439        | 0.914        | 0.731        | 0.157        | 0.449        | 0.538        | 0.842        | —            |
|              | ATC-MC     | 0.624        | 0.481        | 0.478        | <b>0.919</b> | 0.797        | 0.243        | 0.505        | 0.576        | 0.856        | —            |

Table 15: Full  $R^2$  metrics for all pairs of training and test domains for BERT models trained on MNLI.

| Measure      | Test Domain  |                 |              |              |              |              |              |
|--------------|--------------|-----------------|--------------|--------------|--------------|--------------|--------------|
|              | ImageNetV2   | Imagenet-Sketch | ObjectNet    | ImageNet-Vid | YTBb         | ImageNet-R   | ImageNet-A   |
| NI-RandAug   | 0.816        | 0.677           | 0.646        | <b>0.790</b> | <b>0.692</b> | <b>0.724</b> | 0.586        |
| NI-Translate | 0.785        | 0.515           | 0.516        | 0.685        | 0.544        | 0.581        | 0.439        |
| NI-Erase     | 0.779        | 0.322           | 0.52         | 0.706        | 0.560        | 0.444        | 0.517        |
| NI-FC        | 0.842        | 0.552           | 0.645        | 0.780        | 0.649        | 0.679        | <b>0.589</b> |
| ATC-MC       | 0.891        | 0.708           | <b>0.756</b> | 0.761        | 0.509        | 0.521        | 0.190        |
| ATC-NE       | <b>0.926</b> | <b>0.717</b>    | 0.671        | 0.651        | 0.494        | 0.569        | 0.248        |

Table 16: Full macro  $\tau$  metrics for all ImageNet test domains. We average values across models.

| Model | Train Domain | Measure      | Test Domain |               |              |
|-------|--------------|--------------|-------------|---------------|--------------|
|       |              |              | SVHN        | Colored MNIST | MNIST        |
| NiN   | SVHN         | NI-RandAug   | —           | 0.668         | 0.616        |
|       |              | NI-Translate | —           | <b>0.699</b>  | 0.635        |
|       |              | NI-Erase     | —           | -0.102        | -0.168       |
|       |              | NI-FC        | —           | -0.233        | -0.398       |
|       |              | ATC-NE       | —           | 0.577         | 0.695        |
|       |              | ATC-MC       | —           | 0.646         | <b>0.716</b> |

Table 17: Full macro  $\tau$  metrics for all test domains for image classification models on SVHN, Colored MNIST, and MNIST.

| Model  | Train Domain | Measure      | Test Domain  |              |              |              |
|--------|--------------|--------------|--------------|--------------|--------------|--------------|
|        |              |              | CIFAR10      | CINIC10      | CIFAR10.1    | CIFAR10.2    |
| NIN    | CIFAR10      | Nl-RandAug   | —            | <b>0.785</b> | <b>0.830</b> | <b>0.783</b> |
|        |              | Nl-Translate | —            | 0.621        | 0.739        | 0.644        |
|        |              | Nl-Erase     | —            | 0.385        | 0.485        | 0.499        |
|        |              | Nl-FC        | —            | 0.194        | 0.077        | -0.037       |
|        | ATC-NE       | —            | 0.250        | 0.543        | 0.484        |              |
|        | ATC-MC       | —            | 0.319        | 0.514        | 0.446        |              |
| ResNet | CIFAR10      | Nl-RandAug   | —            | 0.706        | 0.722        | <b>0.742</b> |
|        |              | Nl-Translate | —            | <b>0.800</b> | <b>0.796</b> | 0.741        |
|        |              | Nl-Erase     | —            | 0.723        | 0.727        | 0.699        |
|        |              | Nl-FC        | —            | 0.647        | 0.585        | 0.607        |
|        | ATC-NE       | —            | 0.644        | 0.687        | 0.630        |              |
|        | ATC-MC       | —            | 0.631        | 0.672        | 0.624        |              |
| VGG    | CIFAR10      | Nl-RandAug   | —            | <b>0.868</b> | <b>0.831</b> | <b>0.772</b> |
|        |              | Nl-Translate | —            | 0.700        | 0.679        | 0.632        |
|        |              | Nl-Erase     | —            | -0.153       | -0.196       | -0.215       |
|        |              | Nl-FC        | —            | 0.597        | 0.569        | 0.512        |
|        | ATC-NE       | —            | 0.531        | 0.656        | 0.599        |              |
|        | ATC-MC       | —            | 0.533        | 0.648        | 0.586        |              |
| CNN    | CINIC10      | Nl-RandAug   | <b>0.756</b> | —            | <b>0.698</b> | <b>0.695</b> |
|        |              | Nl-Translate | 0.347        | —            | 0.318        | 0.267        |
|        |              | Nl-Erase     | 0.619        | —            | 0.539        | 0.577        |
|        |              | Nl-FC        | 0.249        | —            | 0.286        | 0.252        |
|        | ATC-NE       | 0.607        | —            | 0.510        | 0.447        |              |
|        | ATC-MC       | 0.611        | —            | 0.500        | 0.440        |              |

Table 18: Full macro  $\tau$  metrics for all test domains for image classification models on CIFAR10, CINIC10, CIFAR10.1, and CIFAR10.2.

| Train Domain | Measure    | Test Domain |          |       |        |        |       |        |       |       |       |
|--------------|------------|-------------|----------|-------|--------|--------|-------|--------|-------|-------|-------|
|              |            | books       | clothing | home  | kindle | movies | pets  | sports | tech  | tools | toys  |
| books        | Ni-SSMBA   | —           | 0.708    | 0.741 | 0.687  | 0.678  | 0.754 | 0.738  | 0.738 | 0.721 | 0.669 |
|              | Ni-EDA     | —           | 0.722    | 0.738 | 0.640  | 0.685  | 0.756 | 0.688  | 0.709 | 0.719 | 0.725 |
|              | Ni-BT      | —           | 0.615    | 0.631 | 0.549  | 0.596  | 0.656 | 0.602  | 0.644 | 0.584 | 0.621 |
|              | Ni-RandRep | —           | 0.591    | 0.586 | 0.499  | 0.587  | 0.587 | 0.530  | 0.530 | 0.546 | 0.589 |
|              | ATC-NE     | —           | 0.605    | 0.642 | 0.534  | 0.496  | 0.629 | 0.605  | 0.538 | 0.618 | 0.673 |
|              | ATC-MC     | —           | 0.604    | 0.639 | 0.534  | 0.489  | 0.630 | 0.606  | 0.544 | 0.616 | 0.670 |
| clothing     | Ni-SSMBA   | 0.769       | —        | 0.774 | 0.773  | 0.793  | 0.736 | 0.653  | 0.716 | 0.692 | 0.709 |
|              | Ni-EDA     | 0.681       | —        | 0.670 | 0.787  | 0.721  | 0.696 | 0.675  | 0.694 | 0.690 | 0.784 |
|              | Ni-BT      | 0.574       | —        | 0.542 | 0.654  | 0.595  | 0.481 | 0.506  | 0.501 | 0.503 | 0.679 |
|              | Ni-RandRep | 0.623       | —        | 0.591 | 0.603  | 0.612  | 0.557 | 0.496  | 0.507 | 0.549 | 0.640 |
|              | ATC-NE     | 0.420       | —        | 0.366 | 0.408  | 0.450  | 0.308 | 0.501  | 0.242 | 0.560 | 0.563 |
|              | ATC-MC     | 0.421       | —        | 0.363 | 0.409  | 0.451  | 0.315 | 0.498  | 0.239 | 0.558 | 0.563 |
| home         | Ni-SSMBA   | 0.786       | 0.552    | —     | 0.731  | 0.731  | 0.769 | 0.790  | 0.789 | 0.775 | 0.614 |
|              | Ni-EDA     | 0.734       | 0.659    | —     | 0.767  | 0.649  | 0.757 | 0.707  | 0.723 | 0.696 | 0.748 |
|              | Ni-BT      | 0.670       | 0.498    | —     | 0.641  | 0.639  | 0.634 | 0.706  | 0.688 | 0.662 | 0.639 |
|              | Ni-RandRep | 0.700       | 0.549    | —     | 0.665  | 0.705  | 0.686 | 0.663  | 0.664 | 0.632 | 0.651 |
|              | ATC-NE     | 0.450       | 0.480    | —     | 0.459  | 0.447  | 0.455 | 0.559  | 0.497 | 0.558 | 0.445 |
|              | ATC-MC     | 0.459       | 0.480    | —     | 0.460  | 0.452  | 0.458 | 0.557  | 0.495 | 0.557 | 0.448 |
| kindle       | Ni-SSMBA   | 0.546       | 0.679    | 0.648 | —      | 0.557  | 0.699 | 0.656  | 0.608 | 0.607 | 0.630 |
|              | Ni-EDA     | 0.663       | 0.644    | 0.717 | —      | 0.637  | 0.743 | 0.699  | 0.676 | 0.643 | 0.696 |
|              | Ni-BT      | 0.556       | 0.648    | 0.652 | —      | 0.562  | 0.686 | 0.683  | 0.668 | 0.643 | 0.674 |
|              | Ni-RandRep | 0.454       | 0.519    | 0.406 | —      | 0.428  | 0.555 | 0.434  | 0.415 | 0.415 | 0.535 |
|              | ATC-NE     | 0.370       | 0.517    | 0.468 | —      | 0.338  | 0.583 | 0.540  | 0.383 | 0.472 | 0.545 |
|              | ATC-MC     | 0.382       | 0.512    | 0.469 | —      | 0.342  | 0.578 | 0.538  | 0.384 | 0.467 | 0.541 |
| movies       | Ni-SSMBA   | 0.572       | 0.689    | 0.717 | 0.660  | —      | 0.732 | 0.741  | 0.675 | 0.714 | 0.749 |
|              | Ni-EDA     | 0.500       | 0.717    | 0.711 | 0.629  | —      | 0.733 | 0.725  | 0.660 | 0.719 | 0.748 |
|              | Ni-BT      | 0.619       | 0.681    | 0.687 | 0.595  | —      | 0.738 | 0.729  | 0.717 | 0.717 | 0.717 |
|              | Ni-RandRep | 0.435       | 0.491    | 0.523 | 0.554  | —      | 0.460 | 0.528  | 0.436 | 0.517 | 0.570 |
|              | ATC-NE     | 0.436       | 0.605    | 0.623 | 0.398  | —      | 0.571 | 0.622  | 0.502 | 0.659 | 0.701 |
|              | ATC-MC     | 0.449       | 0.612    | 0.631 | 0.403  | —      | 0.586 | 0.628  | 0.520 | 0.661 | 0.704 |
| pets         | Ni-SSMBA   | 0.747       | 0.560    | 0.653 | 0.766  | 0.769  | —     | 0.719  | 0.696 | 0.697 | 0.731 |
|              | Ni-EDA     | 0.762       | 0.684    | 0.682 | 0.794  | 0.773  | —     | 0.709  | 0.666 | 0.689 | 0.745 |
|              | Ni-BT      | 0.741       | 0.550    | 0.625 | 0.751  | 0.776  | —     | 0.704  | 0.676 | 0.702 | 0.654 |
|              | Ni-RandRep | 0.619       | 0.575    | 0.572 | 0.696  | 0.662  | —     | 0.580  | 0.532 | 0.590 | 0.710 |
|              | ATC-NE     | 0.570       | 0.545    | 0.462 | 0.533  | 0.543  | —     | 0.511  | 0.333 | 0.503 | 0.634 |
|              | ATC-MC     | 0.575       | 0.541    | 0.461 | 0.535  | 0.546  | —     | 0.510  | 0.332 | 0.502 | 0.632 |
| sports       | Ni-SSMBA   | 0.774       | 0.561    | 0.657 | 0.812  | 0.764  | 0.683 | —      | 0.689 | 0.746 | 0.651 |
|              | Ni-EDA     | 0.770       | 0.633    | 0.659 | 0.791  | 0.710  | 0.712 | —      | 0.701 | 0.732 | 0.692 |
|              | Ni-BT      | 0.670       | 0.484    | 0.598 | 0.669  | 0.672  | 0.545 | —      | 0.534 | 0.704 | 0.538 |
|              | Ni-RandRep | 0.684       | 0.472    | 0.562 | 0.704  | 0.638  | 0.589 | —      | 0.614 | 0.622 | 0.693 |
|              | ATC-NE     | 0.502       | 0.456    | 0.454 | 0.506  | 0.443  | 0.374 | —      | 0.397 | 0.565 | 0.577 |
|              | ATC-MC     | 0.513       | 0.459    | 0.459 | 0.521  | 0.453  | 0.393 | —      | 0.410 | 0.574 | 0.574 |
| tech         | Ni-SSMBA   | 0.784       | 0.674    | 0.768 | 0.793  | 0.779  | 0.755 | 0.785  | —     | 0.690 | 0.746 |
|              | Ni-EDA     | 0.718       | 0.698    | 0.760 | 0.743  | 0.752  | 0.817 | 0.772  | —     | 0.726 | 0.781 |
|              | Ni-BT      | 0.715       | 0.576    | 0.719 | 0.732  | 0.746  | 0.716 | 0.726  | —     | 0.687 | 0.683 |
|              | Ni-RandRep | 0.680       | 0.647    | 0.652 | 0.686  | 0.668  | 0.677 | 0.664  | —     | 0.553 | 0.742 |
|              | ATC-NE     | 0.547       | 0.688    | 0.681 | 0.560  | 0.640  | 0.653 | 0.666  | —     | 0.637 | 0.723 |
|              | ATC-MC     | 0.553       | 0.690    | 0.678 | 0.567  | 0.639  | 0.650 | 0.664  | —     | 0.633 | 0.721 |
| tools        | Ni-SSMBA   | 0.783       | 0.541    | 0.604 | 0.777  | 0.782  | 0.757 | 0.758  | 0.730 | —     | 0.682 |
|              | Ni-EDA     | 0.716       | 0.544    | 0.628 | 0.769  | 0.735  | 0.764 | 0.722  | 0.696 | —     | 0.745 |
|              | Ni-BT      | 0.691       | 0.342    | 0.454 | 0.681  | 0.677  | 0.567 | 0.575  | 0.636 | —     | 0.513 |
|              | Ni-RandRep | 0.661       | 0.550    | 0.581 | 0.675  | 0.685  | 0.668 | 0.623  | 0.584 | —     | 0.687 |
|              | ATC-NE     | 0.621       | 0.472    | 0.480 | 0.647  | 0.648  | 0.536 | 0.598  | 0.448 | —     | 0.619 |
|              | ATC-MC     | 0.630       | 0.465    | 0.491 | 0.649  | 0.655  | 0.538 | 0.597  | 0.456 | —     | 0.622 |
| toys         | Ni-SSMBA   | 0.760       | 0.702    | 0.758 | 0.763  | 0.731  | 0.775 | 0.782  | 0.750 | 0.753 | —     |
|              | Ni-EDA     | 0.689       | 0.678    | 0.661 | 0.757  | 0.646  | 0.736 | 0.711  | 0.661 | 0.700 | —     |
|              | Ni-BT      | 0.582       | 0.604    | 0.683 | 0.626  | 0.610  | 0.695 | 0.723  | 0.660 | 0.658 | —     |
|              | Ni-RandRep | 0.629       | 0.660    | 0.575 | 0.661  | 0.609  | 0.605 | 0.599  | 0.543 | 0.582 | —     |
|              | ATC-NE     | 0.401       | 0.617    | 0.309 | 0.479  | 0.370  | 0.323 | 0.485  | 0.143 | 0.526 | —     |
|              | ATC-MC     | 0.408       | 0.614    | 0.315 | 0.476  | 0.368  | 0.329 | 0.488  | 0.146 | 0.535 | —     |

Table 19: Full macro  $\tau$  metrics for all pairs of training and test domains for CNN models trained on AWS.

| Train Domain | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|              |            | books        | clothing     | home         | kindle       | movies       | pets         | sports       | tech         | tools        | toys         |
| books        | NL-SSMBA   | —            | <b>0.886</b> | <b>0.906</b> | 0.893        | 0.901        | <b>0.900</b> | 0.925        | <b>0.918</b> | <b>0.931</b> | <b>0.857</b> |
|              | NL-EDA     | —            | 0.883        | —            | 0.890        | 0.852        | 0.863        | 0.854        | 0.855        | 0.895        | 0.844        |
|              | NL-BT      | —            | 0.830        | 0.887        | <b>0.893</b> | 0.883        | 0.884        | 0.871        | 0.878        | 0.869        | 0.824        |
|              | NL-RandRep | —            | 0.882        | 0.871        | <b>0.908</b> | <b>0.906</b> | 0.897        | 0.894        | 0.900        | 0.913        | 0.846        |
|              | ATC-NE     | —            | 0.803        | 0.814        | 0.851        | 0.821        | 0.885        | 0.756        | 0.843        | 0.774        | 0.832        |
|              | ATC-MC     | —            | 0.789        | 0.806        | 0.850        | 0.794        | 0.874        | 0.760        | 0.845        | 0.768        | 0.824        |
| clothing     | NL-SSMBA   | 0.797        | —            | 0.882        | 0.714        | 0.773        | 0.826        | <b>0.881</b> | 0.826        | <b>0.883</b> | <b>0.824</b> |
|              | NL-EDA     | <b>0.828</b> | —            | 0.848        | <b>0.813</b> | 0.787        | <b>0.862</b> | 0.741        | <b>0.839</b> | 0.848        | 0.777        |
|              | NL-BT      | 0.817        | —            | <b>0.903</b> | 0.686        | <b>0.837</b> | 0.832        | 0.849        | 0.835        | 0.823        | 0.778        |
|              | NL-RandRep | 0.762        | —            | 0.858        | 0.691        | 0.826        | 0.832        | 0.872        | 0.822        | 0.868        | 0.786        |
|              | ATC-NE     | 0.729        | —            | 0.805        | 0.672        | 0.681        | 0.817        | 0.728        | 0.698        | 0.792        | 0.791        |
|              | ATC-MC     | 0.735        | —            | 0.794        | 0.683        | 0.701        | 0.858        | 0.744        | 0.736        | 0.808        | 0.791        |
| home         | NL-SSMBA   | 0.736        | 0.800        | —            | <b>0.786</b> | 0.780        | 0.868        | 0.896        | <b>0.876</b> | 0.888        | 0.885        |
|              | NL-EDA     | 0.680        | <b>0.885</b> | —            | 0.768        | <b>0.830</b> | <b>0.880</b> | 0.879        | 0.843        | 0.877        | <b>0.930</b> |
|              | NL-BT      | <b>0.749</b> | 0.788        | —            | 0.756        | 0.778        | 0.819        | 0.871        | 0.835        | <b>0.906</b> | 0.866        |
|              | NL-RandRep | 0.729        | 0.811        | —            | 0.701        | 0.759        | 0.853        | <b>0.931</b> | 0.831        | 0.876        | 0.874        |
|              | ATC-NE     | 0.691        | 0.788        | —            | 0.750        | 0.802        | 0.791        | 0.797        | 0.739        | 0.824        | 0.824        |
|              | ATC-MC     | 0.684        | <b>0.782</b> | —            | 0.738        | 0.793        | 0.806        | 0.772        | 0.751        | 0.826        | 0.819        |
| kindle       | NL-SSMBA   | 0.735        | <b>0.850</b> | 0.885        | —            | 0.701        | <b>0.913</b> | 0.875        | 0.869        | 0.763        | <b>0.871</b> |
|              | NL-EDA     | <b>0.840</b> | 0.830        | 0.824        | —            | 0.709        | 0.879        | 0.856        | 0.865        | <b>0.854</b> | 0.790        |
|              | NL-BT      | 0.727        | 0.848        | <b>0.892</b> | —            | <b>0.763</b> | 0.856        | <b>0.926</b> | <b>0.871</b> | <b>0.879</b> | 0.861        |
|              | NL-RandRep | 0.773        | 0.848        | 0.816        | —            | 0.735        | 0.875        | 0.857        | 0.843        | 0.794        | 0.847        |
|              | ATC-NE     | 0.695        | 0.335        | 0.486        | —            | 0.644        | 0.211        | 0.274        | 0.426        | 0.238        | 0.589        |
|              | ATC-MC     | 0.729        | 0.455        | 0.558        | —            | 0.671        | 0.337        | 0.366        | 0.509        | 0.369        | 0.652        |
| movies       | NL-SSMBA   | 0.845        | <b>0.897</b> | <b>0.888</b> | 0.782        | —            | 0.850        | <b>0.931</b> | <b>0.877</b> | 0.881        | 0.896        |
|              | NL-EDA     | 0.764        | 0.881        | 0.852        | <b>0.863</b> | —            | <b>0.859</b> | 0.901        | 0.863        | <b>0.882</b> | <b>0.938</b> |
|              | NL-BT      | 0.810        | 0.894        | 0.882        | 0.719        | —            | 0.837        | 0.894        | 0.864        | 0.872        | 0.870        |
|              | NL-RandRep | <b>0.851</b> | 0.875        | 0.855        | 0.742        | —            | 0.850        | 0.905        | 0.864        | 0.861        | 0.874        |
|              | ATC-NE     | 0.692        | 0.745        | 0.807        | 0.712        | —            | 0.561        | 0.768        | 0.580        | 0.620        | 0.763        |
|              | ATC-MC     | 0.698        | 0.770        | 0.847        | 0.722        | —            | 0.622        | 0.813        | 0.637        | 0.678        | 0.784        |
| pets         | NL-SSMBA   | 0.728        | 0.737        | 0.866        | 0.737        | 0.829        | —            | <b>0.883</b> | 0.867        | 0.859        | 0.773        |
|              | NL-EDA     | 0.732        | <b>0.831</b> | <b>0.880</b> | 0.601        | 0.788        | —            | 0.853        | 0.834        | <b>0.903</b> | <b>0.841</b> |
|              | NL-BT      | 0.697        | 0.685        | 0.816        | <b>0.758</b> | 0.816        | —            | 0.867        | 0.814        | 0.865        | 0.810        |
|              | NL-RandRep | <b>0.753</b> | 0.731        | 0.813        | <b>0.735</b> | <b>0.841</b> | —            | 0.828        | <b>0.871</b> | 0.869        | 0.774        |
|              | ATC-NE     | 0.623        | 0.740        | 0.719        | 0.553        | 0.737        | —            | 0.815        | 0.788        | 0.748        | 0.781        |
|              | ATC-MC     | 0.662        | 0.753        | 0.723        | 0.566        | 0.711        | —            | 0.831        | 0.781        | 0.752        | 0.750        |
| sports       | NL-SSMBA   | 0.715        | 0.743        | <b>0.879</b> | <b>0.764</b> | <b>0.837</b> | <b>0.863</b> | —            | <b>0.879</b> | <b>0.860</b> | <b>0.870</b> |
|              | NL-EDA     | <b>0.757</b> | 0.731        | 0.807        | 0.673        | 0.798        | 0.786        | —            | 0.815        | 0.827        | 0.833        |
|              | NL-BT      | 0.728        | <b>0.753</b> | 0.766        | 0.664        | 0.826        | 0.832        | —            | 0.783        | 0.824        | 0.865        |
|              | NL-RandRep | 0.735        | 0.702        | 0.816        | 0.728        | 0.808        | 0.802        | —            | 0.830        | 0.849        | 0.836        |
|              | ATC-NE     | 0.489        | 0.734        | 0.738        | 0.567        | 0.730        | 0.736        | —            | 0.726        | 0.735        | 0.862        |
|              | ATC-MC     | 0.484        | 0.728        | 0.766        | 0.562        | 0.754        | 0.727        | —            | 0.725        | 0.741        | 0.860        |
| tech         | NL-SSMBA   | 0.784        | 0.763        | 0.845        | 0.710        | 0.769        | 0.853        | 0.755        | —            | <b>0.886</b> | <b>0.877</b> |
|              | NL-EDA     | 0.771        | <b>0.867</b> | <b>0.881</b> | <b>0.763</b> | 0.801        | 0.855        | <b>0.886</b> | —            | 0.904        | 0.869        |
|              | NL-BT      | 0.753        | 0.750        | 0.867        | 0.719        | 0.805        | 0.869        | 0.787        | —            | 0.842        | 0.835        |
|              | NL-RandRep | <b>0.819</b> | 0.721        | 0.843        | 0.711        | <b>0.854</b> | <b>0.889</b> | 0.787        | —            | 0.854        | 0.874        |
|              | ATC-NE     | 0.737        | 0.744        | 0.787        | 0.751        | 0.732        | 0.795        | 0.727        | —            | 0.830        | 0.863        |
|              | ATC-MC     | 0.771        | 0.730        | 0.764        | 0.749        | 0.704        | 0.809        | 0.771        | —            | 0.789        | 0.843        |
| tools        | NL-SSMBA   | 0.792        | <b>0.854</b> | 0.786        | 0.707        | <b>0.746</b> | 0.875        | <b>0.888</b> | 0.792        | —            | 0.866        |
|              | NL-EDA     | <b>0.859</b> | 0.846        | <b>0.834</b> | <b>0.847</b> | <b>0.790</b> | 0.837        | 0.886        | <b>0.819</b> | —            | <b>0.894</b> |
|              | NL-BT      | 0.769        | 0.802        | 0.763        | 0.709        | 0.750        | 0.832        | 0.824        | 0.744        | —            | 0.856        |
|              | NL-RandRep | 0.792        | 0.825        | 0.816        | 0.717        | 0.724        | <b>0.885</b> | 0.877        | 0.801        | —            | 0.845        |
|              | ATC-NE     | 0.745        | 0.789        | 0.775        | 0.744        | 0.632        | 0.698        | 0.856        | 0.737        | —            | 0.799        |
|              | ATC-MC     | 0.749        | 0.798        | 0.772        | 0.745        | 0.634        | 0.737        | 0.876        | 0.777        | —            | 0.793        |
| toys         | NL-SSMBA   | 0.779        | <b>0.830</b> | <b>0.813</b> | 0.766        | <b>0.695</b> | 0.808        | <b>0.872</b> | 0.872        | <b>0.891</b> | —            |
|              | NL-EDA     | 0.696        | 0.790        | 0.805        | <b>0.805</b> | <b>0.790</b> | 0.837        | 0.834        | 0.860        | 0.840        | —            |
|              | NL-BT      | 0.748        | 0.762        | 0.746        | 0.755        | 0.707        | <b>0.840</b> | 0.809        | 0.773        | 0.811        | —            |
|              | NL-RandRep | <b>0.813</b> | 0.788        | 0.783        | 0.770        | 0.660        | <b>0.788</b> | 0.838        | <b>0.873</b> | <b>0.891</b> | —            |
|              | ATC-NE     | 0.656        | 0.469        | 0.743        | 0.550        | 0.628        | 0.515        | 0.792        | 0.703        | 0.771        | —            |
|              | ATC-MC     | 0.694        | 0.482        | 0.726        | 0.539        | 0.651        | 0.571        | 0.805        | 0.730        | 0.777        | —            |

Table 20: Full macro  $\tau$  metrics for all pairs of training and test domains for BERT models trained on AWS.

| Train Domain | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|              |            | slate        | verbatim     | facetoface   | oup          | nineteleven  | fiction      | telephone    | travel       | letters      | government   |
| slate        | NL-SSMBA   | —            | 0.483        | 0.675        | 0.553        | 0.608        | 0.650        | 0.661        | 0.570        | 0.622        | 0.641        |
|              | NL-EDA     | —            | 0.564        | 0.675        | 0.563        | 0.599        | 0.658        | 0.672        | 0.557        | 0.599        | 0.649        |
|              | NL-BT      | —            | <b>0.669</b> | <b>0.742</b> | <b>0.736</b> | <b>0.756</b> | <b>0.727</b> | <b>0.754</b> | <b>0.751</b> | <b>0.738</b> | <b>0.791</b> |
|              | NL-RandRep | —            | 0.393        | 0.526        | 0.529        | 0.480        | 0.398        | 0.438        | 0.445        | 0.640        | 0.531        |
|              | ATC-NE     | —            | 0.650        | 0.632        | 0.577        | 0.594        | 0.683        | 0.625        | 0.602        | 0.621        | 0.699        |
|              | ATC-MC     | —            | 0.652        | 0.632        | 0.578        | 0.594        | 0.681        | 0.625        | 0.603        | 0.622        | 0.697        |
| fiction      | NL-SSMBA   | 0.577        | 0.537        | 0.692        | 0.482        | 0.539        | —            | 0.629        | 0.520        | 0.546        | 0.598        |
|              | NL-EDA     | 0.632        | 0.584        | 0.691        | 0.608        | 0.557        | —            | 0.631        | 0.529        | 0.540        | 0.608        |
|              | NL-BT      | <b>0.783</b> | <b>0.704</b> | <b>0.771</b> | <b>0.693</b> | <b>0.671</b> | —            | <b>0.776</b> | <b>0.680</b> | <b>0.707</b> | <b>0.693</b> |
|              | NL-RandRep | 0.495        | 0.446        | 0.522        | 0.531        | 0.437        | —            | 0.434        | 0.464        | 0.531        | 0.493        |
|              | ATC-NE     | 0.538        | 0.461        | 0.630        | 0.428        | 0.513        | —            | 0.542        | 0.469        | 0.581        | 0.536        |
|              | ATC-MC     | 0.530        | 0.450        | 0.631        | 0.427        | 0.513        | —            | 0.543        | 0.463        | 0.576        | 0.534        |
| telephone    | NL-SSMBA   | 0.567        | 0.503        | <b>0.736</b> | 0.493        | 0.507        | <b>0.608</b> | —            | 0.492        | 0.572        | 0.496        |
|              | NL-EDA     | 0.637        | 0.633        | 0.726        | 0.537        | 0.615        | <b>0.735</b> | —            | 0.579        | <b>0.642</b> | 0.580        |
|              | NL-BT      | <b>0.657</b> | <b>0.668</b> | 0.722        | <b>0.637</b> | <b>0.663</b> | 0.639        | —            | <b>0.621</b> | 0.602        | <b>0.614</b> |
|              | NL-RandRep | 0.504        | 0.470        | 0.573        | 0.470        | 0.510        | 0.480        | —            | 0.482        | 0.576        | 0.440        |
|              | ATC-NE     | 0.533        | 0.507        | 0.681        | 0.526        | 0.500        | 0.632        | —            | 0.453        | 0.564        | 0.491        |
|              | ATC-MC     | 0.534        | 0.507        | 0.681        | 0.522        | 0.501        | 0.637        | —            | 0.455        | 0.564        | 0.494        |
| travel       | NL-SSMBA   | 0.524        | 0.528        | 0.585        | 0.462        | 0.517        | 0.537        | 0.569        | —            | 0.617        | 0.521        |
|              | NL-EDA     | 0.492        | 0.484        | 0.523        | 0.461        | 0.418        | 0.550        | 0.524        | —            | 0.507        | 0.458        |
|              | NL-BT      | <b>0.619</b> | <b>0.634</b> | <b>0.630</b> | <b>0.660</b> | <b>0.681</b> | <b>0.655</b> | <b>0.645</b> | —            | <b>0.713</b> | <b>0.699</b> |
|              | NL-RandRep | 0.465        | 0.476        | 0.467        | 0.514        | 0.492        | 0.472        | 0.526        | —            | 0.569        | 0.497        |
|              | ATC-NE     | 0.579        | 0.528        | 0.533        | 0.643        | 0.541        | 0.571        | 0.591        | —            | 0.572        | 0.597        |
|              | ATC-MC     | 0.576        | 0.528        | 0.534        | 0.642        | 0.536        | 0.573        | 0.594        | —            | 0.570        | 0.602        |
| government   | NL-SSMBA   | 0.578        | 0.531        | 0.600        | 0.568        | 0.565        | 0.617        | 0.589        | 0.563        | 0.616        | —            |
|              | NL-EDA     | 0.581        | 0.582        | 0.565        | 0.535        | 0.594        | 0.624        | 0.519        | 0.552        | 0.566        | —            |
|              | NL-BT      | <b>0.719</b> | <b>0.693</b> | <b>0.686</b> | <b>0.687</b> | <b>0.755</b> | <b>0.701</b> | <b>0.651</b> | <b>0.712</b> | <b>0.719</b> | —            |
|              | NL-RandRep | 0.274        | 0.268        | 0.357        | 0.444        | 0.464        | 0.301        | 0.247        | 0.395        | 0.452        | —            |
|              | ATC-NE     | 0.624        | 0.464        | 0.581        | 0.666        | 0.440        | 0.601        | 0.616        | 0.524        | 0.641        | —            |
|              | ATC-MC     | 0.621        | 0.465        | 0.583        | 0.667        | 0.446        | 0.602        | 0.616        | 0.522        | 0.642        | —            |

Table 21: Full macro  $\tau$  metrics for all pairs of training and test domains for CNN models trained on MNLI.

|              |            | Test Domain  |              |              |              |              |              |              |              |              |              |  |
|--------------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--|
| Train Domain | Measure    | state        | verbatin     | facetoface   | oup          | nineleven    | fiction      | telephone    | travel       | letters      | government   |  |
| state        | NI-SSMBA   | —            | <b>0.743</b> | 0.746        | 0.709        | <b>0.767</b> | 0.742        | 0.727        | <b>0.771</b> | <b>0.782</b> | <b>0.819</b> |  |
|              | NI-EDA     | —            | 0.580        | 0.605        | 0.674        | 0.696        | 0.703        | 0.655        | 0.651        | 0.671        | 0.737        |  |
|              | NI-BT      | —            | 0.742        | 0.736        | 0.756        | 0.727        | 0.754        | 0.751        | 0.751        | 0.739        | 0.791        |  |
|              | NI-RandRep | —            | 0.393        | 0.526        | 0.529        | 0.489        | 0.398        | 0.438        | 0.445        | 0.640        | 0.531        |  |
|              | ATC-NE     | —            | 0.689        | <b>0.768</b> | 0.786        | 0.728        | 0.752        | 0.761        | 0.744        | 0.743        | 0.766        |  |
| fiction      | ATC-MC     | —            | 0.684        | <b>0.761</b> | <b>0.794</b> | 0.730        | <b>0.761</b> | <b>0.789</b> | 0.765        | 0.744        | 0.785        |  |
|              | NI-SSMBA   | 0.755        | 0.673        | 0.699        | <b>0.726</b> | <b>0.729</b> | —            | 0.769        | <b>0.806</b> | <b>0.770</b> | <b>0.727</b> |  |
|              | NI-EDA     | 0.699        | 0.565        | 0.649        | 0.532        | 0.559        | —            | 0.698        | 0.584        | 0.624        | 0.629        |  |
|              | NI-BT      | <b>0.783</b> | <b>0.704</b> | <b>0.771</b> | 0.693        | 0.671        | —            | <b>0.776</b> | 0.680        | 0.767        | 0.693        |  |
|              | NI-RandRep | 0.495        | 0.446        | 0.522        | 0.531        | 0.437        | —            | 0.434        | 0.464        | 0.531        | 0.493        |  |
| telephone    | ATC-NE     | 0.618        | 0.377        | 0.707        | 0.478        | 0.711        | —            | 0.596        | 0.558        | 0.658        | 0.711        |  |
|              | ATC-MC     | 0.631        | 0.422        | 0.702        | 0.513        | 0.707        | —            | 0.603        | 0.549        | 0.670        | 0.706        |  |
|              | NI-SSMBA   | <b>0.720</b> | <b>0.768</b> | <b>0.793</b> | <b>0.792</b> | 0.668        | 0.720        | —            | <b>0.813</b> | <b>0.758</b> | <b>0.788</b> |  |
|              | NI-EDA     | 0.670        | 0.667        | 0.706        | 0.617        | <b>0.718</b> | <b>0.798</b> | —            | 0.642        | 0.732        | 0.661        |  |
|              | NI-BT      | 0.657        | 0.668        | 0.722        | 0.637        | 0.663        | 0.639        | —            | 0.621        | 0.692        | 0.614        |  |
| travel       | NI-RandRep | 0.504        | 0.470        | 0.573        | 0.470        | 0.510        | 0.480        | —            | 0.482        | 0.576        | 0.440        |  |
|              | ATC-NE     | 0.462        | 0.486        | 0.602        | 0.575        | 0.611        | 0.542        | —            | 0.644        | 0.697        | 0.702        |  |
|              | ATC-MC     | 0.519        | 0.554        | 0.717        | 0.643        | 0.687        | 0.597        | —            | 0.665        | 0.684        | 0.708        |  |
|              | NI-SSMBA   | <b>0.700</b> | <b>0.806</b> | <b>0.737</b> | <b>0.676</b> | <b>0.744</b> | <b>0.707</b> | <b>0.848</b> | —            | <b>0.789</b> | <b>0.783</b> |  |
|              | NI-EDA     | 0.483        | 0.453        | 0.486        | 0.493        | 0.489        | 0.515        | 0.483        | —            | 0.322        | 0.495        |  |
| government   | NI-BT      | 0.619        | 0.634        | 0.630        | 0.660        | 0.661        | 0.655        | 0.645        | —            | 0.713        | 0.699        |  |
|              | NI-RandRep | 0.465        | 0.476        | 0.467        | 0.514        | 0.492        | 0.472        | 0.526        | —            | 0.569        | 0.497        |  |
|              | ATC-NE     | 0.155        | 0.320        | 0.224        | 0.559        | 0.523        | 0.146        | 0.360        | —            | 0.623        | 0.630        |  |
|              | ATC-MC     | 0.175        | 0.370        | 0.320        | 0.613        | 0.576        | 0.219        | 0.443        | —            | 0.668        | 0.646        |  |
|              | NI-SSMBA   | <b>0.769</b> | <b>0.777</b> | <b>0.727</b> | 0.652        | 0.640        | <b>0.783</b> | <b>0.784</b> | <b>0.805</b> | <b>0.754</b> | —            |  |
|              | NI-EDA     | 0.521        | 0.513        | 0.480        | 0.578        | 0.661        | 0.657        | 0.494        | 0.565        | 0.598        | —            |  |
|              | NI-BT      | 0.719        | 0.693        | 0.686        | <b>0.687</b> | <b>0.755</b> | 0.701        | 0.651        | 0.712        | 0.719        | —            |  |
|              | NI-RandRep | 0.274        | 0.268        | 0.357        | 0.444        | 0.464        | 0.301        | 0.247        | 0.395        | 0.452        | —            |  |
|              | ATC-NE     | 0.377        | 0.176        | 0.472        | 0.660        | 0.558        | 0.239        | 0.497        | 0.302        | 0.638        | —            |  |
|              | ATC-MC     | 0.427        | 0.231        | 0.487        | 0.653        | 0.600        | 0.275        | 0.496        | 0.323        | 0.629        | —            |  |

Table 22: Full macro  $\tau$  metrics for all pairs of training and test domains for BERT models trained on MNLI.

|       |            | Test Domain  |              |              |              |              |              |              |              |              |              |  |
|-------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--|
| Model | Measure    | books        | clothing     | home         | kindle       | movies       | pets         | sports       | tech         | tools        | toys         |  |
| CNN   | NI-SSMBA   | <b>0.677</b> | 0.688        | <b>0.758</b> | <b>0.641</b> | <b>0.691</b> | <b>0.736</b> | <b>0.784</b> | <b>0.758</b> | 0.722        | 0.683        |  |
|       | NI-EDA     | 0.637        | 0.672        | 0.704        | 0.612        | 0.658        | 0.746        | 0.739        | 0.680        | 0.703        | 0.711        |  |
|       | NI-BT      | 0.601        | 0.532        | 0.572        | 0.509        | 0.521        | 0.509        | 0.587        | 0.536        | 0.571        | 0.529        |  |
|       | NI-RandRep | 0.485        | 0.542        | 0.501        | 0.522        | 0.440        | 0.491        | 0.650        | 0.477        | 0.567        | 0.550        |  |
|       | ATC-NE     | 0.406        | <b>0.727</b> | 0.728        | 0.362        | 0.499        | 0.722        | 0.726        | 0.679        | <b>0.730</b> | <b>0.752</b> |  |
| BERT  | ATC-MC     | 0.410        | 0.726        | 0.729        | 0.359        | 0.500        | 0.724        | 0.727        | 0.681        | <b>0.730</b> | <b>0.752</b> |  |
|       | NI-SSMBA   | 0.790        | <b>0.850</b> | 0.868        | 0.664        | <b>0.899</b> | <b>0.864</b> | <b>0.855</b> | <b>0.874</b> | <b>0.880</b> | 0.845        |  |
|       | NI-EDA     | 0.739        | 0.848        | 0.825        | <b>0.731</b> | 0.793        | 0.836        | 0.825        | 0.828        | 0.837        | 0.841        |  |
|       | NI-BT      | 0.702        | 0.845        | 0.830        | 0.698        | 0.792        | 0.839        | 0.843        | 0.813        | 0.819        | <b>0.858</b> |  |
|       | NI-RandRep | <b>0.839</b> | 0.841        | <b>0.891</b> | <b>0.694</b> | 0.864        | 0.834        | 0.747        | 0.825        | 0.850        | 0.831        |  |
|       | ATC-NE     | 0.613        | 0.721        | 0.712        | 0.618        | 0.695        | 0.666        | 0.733        | 0.693        | 0.708        | 0.755        |  |
|       | ATC-MC     | 0.621        | 0.735        | 0.730        | 0.616        | 0.711        | 0.691        | 0.749        | 0.710        | 0.726        | 0.764        |  |

Table 23: Full micro  $\tau$  metrics for all test domains for CNN and BERT models trained on AWS.

| Model | Measure    | Test Domain  |              |              |              |              |              |              |              |              |              |
|-------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|       |            | slate        | verbatim     | facetoface   | oup          | nineteen     | fiction      | telephone    | travel       | letters      | government   |
| CNN   | NI-SSMBA   | 0.540        | 0.493        | 0.562        | 0.493        | 0.527        | 0.431        | 0.607        | 0.544        | 0.579        | 0.563        |
|       | NI-EDA     | 0.560        | 0.555        | 0.427        | 0.520        | 0.488        | 0.435        | 0.499        | 0.552        | 0.532        | 0.542        |
|       | NI-BT      | <b>0.698</b> | <b>0.684</b> | <b>0.611</b> | <b>0.713</b> | <b>0.713</b> | <b>0.595</b> | <b>0.676</b> | <b>0.714</b> | <b>0.697</b> | <b>0.722</b> |
|       | NI-RandRep | 0.448        | 0.417        | 0.427        | 0.503        | 0.472        | 0.324        | 0.395        | 0.450        | 0.550        | 0.509        |
|       | ATC-NE     | 0.461        | 0.413        | 0.549        | 0.427        | 0.350        | 0.517        | 0.531        | 0.307        | 0.497        | 0.404        |
|       | ATC-MC     | 0.460        | 0.411        | 0.550        | 0.429        | 0.351        | 0.517        | 0.532        | 0.307        | 0.496        | 0.404        |
| BERT  | NI-SSMBA   | <b>0.782</b> | <b>0.710</b> | <b>0.742</b> | 0.652        | <b>0.724</b> | <b>0.756</b> | <b>0.779</b> | 0.691        | <b>0.768</b> | 0.700        |
|       | NI-EDA     | 0.563        | 0.556        | 0.481        | 0.622        | 0.586        | 0.545        | 0.529        | 0.607        | 0.607        | 0.630        |
|       | NI-BT      | 0.698        | 0.684        | 0.611        | <b>0.713</b> | 0.713        | 0.595        | 0.676        | <b>0.714</b> | 0.697        | <b>0.722</b> |
|       | NI-RandRep | 0.448        | 0.417        | 0.427        | 0.503        | 0.472        | 0.324        | 0.395        | 0.450        | 0.550        | 0.509        |
|       | ATC-NE     | 0.375        | 0.389        | 0.660        | 0.606        | 0.593        | 0.414        | 0.523        | 0.511        | 0.655        | 0.687        |
|       | ATC-MC     | 0.405        | 0.434        | 0.676        | 0.632        | 0.614        | 0.466        | 0.554        | 0.530        | 0.663        | 0.693        |

Table 24: Full micro  $\tau$  metrics for all test domains for CNN and BERT models trained on MNLI.

| Measure      | Train Domain (Model) |               |                  |               |               |
|--------------|----------------------|---------------|------------------|---------------|---------------|
|              | SVHN (NiN)           | CIFAR10 (NiN) | CIFAR10 (ResNet) | CIFAR10 (VGG) | CINIC10 (CNN) |
| NI-RandAug   | 0.733                | <b>0.797</b>  | 0.746            | <b>0.869</b>  | <b>0.759</b>  |
| NI-Translate | <b>0.881</b>         | 0.782         | <b>0.833</b>     | 0.699         | 0.329         |
| NI-Erase     | 0.324                | 0.409         | 0.708            | -0.183        | 0.606         |
| NI-FC        | -0.033               | -0.015        | 0.568            | 0.628         | 0.289         |
| ATC-NE       | 0.859                | 0.648         | 0.757            | 0.773         | 0.548         |
| ATC-MC       | 0.843                | 0.605         | 0.756            | 0.767         | 0.653         |

Table 25: Full in-domain  $\tau$  metrics for all test domains for image classification models and datasets.

| Model | Measure    | Train Domain |              |              |              |              |              |              |              |              |              |
|-------|------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|--------------|
|       |            | books        | clothing     | home         | kindle       | movies       | pets         | sports       | tech         | tools        | toys         |
| CNN   | NI-SSMBA   | 0.646        | <b>0.613</b> | <b>0.643</b> | 0.577        | 0.597        | 0.591        | 0.703        | 0.648        | 0.643        | 0.620        |
|       | NI-EDA     | <b>0.653</b> | 0.558        | 0.615        | <b>0.660</b> | 0.447        | <b>0.666</b> | <b>0.708</b> | 0.646        | <b>0.663</b> | 0.565        |
|       | NI-BT      | 0.569        | 0.449        | 0.582        | 0.537        | <b>0.626</b> | 0.605        | 0.568        | <b>0.653</b> | 0.640        | <b>0.652</b> |
|       | NI-RandRep | 0.560        | 0.557        | 0.559        | 0.559        | 0.593        | 0.596        | 0.572        | 0.545        | 0.566        | 0.615        |
|       | ATC-NE     | 0.575        | 0.413        | 0.561        | 0.478        | 0.294        | 0.464        | 0.539        | 0.561        | 0.446        | 0.341        |
|       | ATC-MC     | 0.566        | 0.405        | 0.558        | 0.478        | 0.300        | 0.461        | 0.540        | 0.569        | 0.451        | 0.338        |
| BERT  | NI-SSMBA   | 0.735        | <b>0.874</b> | 0.868        | 0.693        | 0.780        | 0.871        | <b>0.884</b> | 0.851        | 0.851        | <b>0.884</b> |
|       | NI-EDA     | <b>0.875</b> | 0.788        | 0.814        | <b>0.700</b> | <b>0.870</b> | <b>0.894</b> | 0.818        | <b>0.858</b> | <b>0.882</b> | 0.810        |
|       | NI-BT      | 0.801        | 0.811        | <b>0.857</b> | 0.638        | 0.835        | 0.813        | 0.860        | 0.846        | 0.803        | 0.750        |
|       | NI-RandRep | 0.839        | 0.841        | <b>0.891</b> | 0.694        | 0.864        | 0.834        | 0.747        | 0.825        | 0.850        | 0.831        |
|       | ATC-NE     | 0.733        | 0.724        | 0.822        | 0.640        | 0.721        | 0.711        | 0.778        | 0.783        | 0.799        | 0.783        |
|       | ATC-MC     | 0.742        | 0.746        | 0.813        | 0.630        | 0.730        | 0.725        | 0.814        | 0.780        | 0.774        | 0.739        |

Table 26: Full in-domain  $\tau$  metrics for all test domains for CNN and BERT models trained on AWS.

| Model | Measure    | Train Domain |              |              |              |              |
|-------|------------|--------------|--------------|--------------|--------------|--------------|
|       |            | slate        | fiction      | telephone    | travel       | government   |
| CNN   | NI-SSMBA   | 0.664        | 0.655        | 0.753        | 0.692        | 0.758        |
|       | NI-EDA     | 0.629        | 0.723        | <b>0.786</b> | 0.652        | 0.755        |
|       | NI-BT      | <b>0.751</b> | <b>0.765</b> | 0.637        | <b>0.786</b> | <b>0.826</b> |
|       | NI-RandRep | 0.437        | 0.471        | 0.546        | 0.570        | 0.491        |
|       | ATC-NE     | 0.704        | 0.662        | 0.719        | 0.715        | 0.722        |
|       | ATC-MC     | 0.703        | 0.660        | 0.722        | 0.716        | 0.728        |
| BERT  | NI-SSMBA   | 0.713        | 0.754        | 0.766        | 0.785        | <b>0.839</b> |
|       | NI-EDA     | 0.608        | 0.664        | 0.781        | 0.575        | 0.726        |
|       | NI-BT      | <b>0.751</b> | 0.765        | 0.637        | <b>0.786</b> | 0.826        |
|       | NI-RandRep | 0.437        | 0.471        | 0.546        | 0.570        | 0.491        |
|       | ATC-NE     | 0.722        | <b>0.796</b> | 0.740        | 0.737        | 0.701        |
|       | ATC-MC     | 0.745        | 0.791        | <b>0.791</b> | 0.719        | 0.696        |

Table 27: Full in-domain  $\tau$  metrics for all train domains for CNN and BERT models trained on MNLI.

| Measure      | Domain Shifts |              | ImageNet-A |              |           |
|--------------|---------------|--------------|------------|--------------|-----------|
|              | $R^2$         | Macro $\tau$ | $R^2$      | Macro $\tau$ | ID $\tau$ |
| NI-RandAug   | 0.091         | 0.061        | —          | —            | —         |
| NI-Translate | 0.159         | 0.099        | —          | —            | —         |
| NI-Erase     | 0.174         | 0.153        | —          | —            | —         |
| NI-FC        | 0.139         | 0.095        | —          | —            | —         |
| ATC-NE       | 0.272         | 0.136        | —          | —            | —         |
| ATC-MC       | 0.279         | 0.135        | —          | —            | —         |

(a) Standard deviations for ImageNet scale results. No standard deviations are reported for ImageNet-A or ID  $\tau$  since we report only a single value.

| Measure      | C110  |              |           | Numbers |              |           |
|--------------|-------|--------------|-----------|---------|--------------|-----------|
|              | $R^2$ | Macro $\tau$ | ID $\tau$ | $R^2$   | Macro $\tau$ | ID $\tau$ |
| NI-RandAug   | 0.044 | 0.054        | 0.048     | 0.021   | 0.026        | —         |
| NI-Translate | 0.170 | 0.181        | 0.197     | 0.043   | 0.032        | —         |
| NI-Erase     | 0.254 | 0.348        | 0.345     | 0.065   | 0.033        | —         |
| NI-FC        | 0.239 | 0.225        | 0.255     | 0.074   | 0.083        | —         |
| ATC-NE       | 0.169 | 0.115        | 0.091     | 0.109   | 0.059        | —         |
| ATC-MC       | 0.168 | 0.101        | 0.093     | 0.077   | 0.035        | —         |

(b) Standard deviations for small scale image results. No standard deviation is reported for Numbers ID  $\tau$  since we report only a single value.

| Measure    | CNN   |              |              |           | RoBERTa |              |              |           |
|------------|-------|--------------|--------------|-----------|---------|--------------|--------------|-----------|
|            | $R^2$ | Macro $\tau$ | Micro $\tau$ | ID $\tau$ | $R^2$   | Macro $\tau$ | Micro $\tau$ | ID $\tau$ |
| NI-SSMBA   | 0.079 | 0.067        | 0.046        | 0.035     | 0.012   | 0.063        | 0.064        | 0.065     |
| NI-EDA     | 0.108 | 0.051        | 0.041        | 0.072     | 0.016   | 0.058        | 0.040        | 0.055     |
| NI-BT      | 0.161 | 0.078        | 0.032        | 0.060     | 0.021   | 0.060        | 0.061        | 0.063     |
| NI-RandRep | 0.069 | 0.080        | 0.054        | 0.021     | 0.015   | 0.060        | 0.055        | 0.055     |
| ATC-NE     | 0.126 | 0.110        | 0.143        | 0.092     | 0.168   | 0.135        | 0.044        | 0.051     |
| ATC-MC     | 0.125 | 0.109        | 0.143        | 0.091     | 0.139   | 0.112        | 0.048        | 0.050     |

(c) Standard deviations on sentiment analysis results.

| Measure    | CNN   |              |              |           | RoBERTa |              |              |           |
|------------|-------|--------------|--------------|-----------|---------|--------------|--------------|-----------|
|            | $R^2$ | Macro $\tau$ | Micro $\tau$ | ID $\tau$ | $R^2$   | Macro $\tau$ | Micro $\tau$ | ID $\tau$ |
| NI-SSMBA   | 0.098 | 0.060        | 0.048        | 0.043     | 0.035   | 0.046        | 0.040        | 0.041     |
| NI-EDA     | 0.107 | 0.068        | 0.046        | 0.060     | 0.108   | 0.087        | 0.044        | 0.075     |
| NI-BT      | 0.045 | 0.049        | 0.042        | 0.063     | 0.045   | 0.049        | 0.042        | 0.063     |
| NI-RandRep | 0.196 | 0.079        | 0.061        | 0.049     | 0.196   | 0.079        | 0.061        | 0.049     |
| ATC-NE     | 0.108 | 0.068        | 0.076        | 0.022     | 0.206   | 0.180        | 0.111        | 0.032     |
| ATC-MC     | 0.108 | 0.068        | 0.076        | 0.024     | 0.174   | 0.164        | 0.100        | 0.038     |

(d) Standard deviations on NLI results.

Table 28: Standard deviations for reported values in Table 2

### Breaking raw markdown into nicely formatted solveit chunks

In [ ]:
add_msg??


```python
@llmtool(dname=dname_doc)
@delegates(_add_msg_unsafe, but=['run'])
async def add_msg(
    content:str, # Content of the message (i.e the message prompt, code, or note text)
    **kwargs
)->str: # Message ID of newly created message
    """Add/update a message to the queue to show after code execution completes.
    **NB**: when creating multiple messages in a row, after the 1st message set `id` to the result of the last `add_msg` call,
    otherwise messages will appear in the dialog in REVERSE order.
    {dname}"""
    return await _add_msg_unsafe(content=content, run=False, **kwargs)
```

**File:** `/usr/local/lib/python3.12/site-packages/dialoghelper/core.py`

Split a markdown document into logical chunks:
- Preamble (everything before first header) → single cell
- Headers → individual cells
- Content under each header → split by paragraphs (double newlines)

In [ ]:
#| export
def split_markdown_into_cells(markdown_text):
    """Split markdown into cells: preamble, then headers + paragraphs"""
    lines = markdown_text.split('\n')
    first_header_idx = None
    for i, line in enumerate(lines):
        if line.startswith('##'):
            first_header_idx = i
            break
    cells = []
    # Everything before first header = one cell (all of the authors and affiliations)
    if first_header_idx:
        preamble = '\n'.join(lines[:first_header_idx]).strip()
        if preamble:
            cells.append(preamble)
    # For rest, separate headers into their own cells; for everything else, split by double \n\n via two passes
    current_section = []
    for line in lines[first_header_idx:] if first_header_idx else lines:
        if line.startswith('#'):
            if current_section:
                cells.append('\n'.join(current_section).strip())
                current_section = []
            cells.append(line[1:] if line.startswith('##') else line)
        else:
            current_section.append(line)
    if current_section:
        content = '\n'.join(current_section).strip()
        if content:
            cells.append(content)
    # Split non-header cells by paragraphs
    final_cells = []
    for cell in cells:
        if cell.startswith('#'):
            final_cells.append(cell)
        else:
            paragraphs = cell.split('\n\n')
            final_cells.extend([p.strip() for p in paragraphs if p.strip()])
    
    return final_cells

Processes an academic paper for reading in SolveIt, with nice handling of the references for an incremental-reading-with-AI-partner workflow. The full References section gets added as a collapsed cell to the top, while inline citations like `(Author, 2030)` get replaced with less intrusive numbered footnotes (for reading ergonomics). A cheatsheet for converting between the two is placed at the start of each section.

Ideally this means that 1) you can track down references easily by searching for the unique numbered footnote, and 2) the solveit AI, if asked inline while reading, has the citations in context. 

In the future, we might augment SolveIt with some tools for tracking down the papers cited, e.g. using the Semantic Scholar python library., so it can more effectively provide context behind a citation. On eimagines this could both improve reader understanding, and occasionally catch faulty citations -- i.e. things the author hand't actually read, and which might disagree with their thesis!

In [ ]:
#| export
import re

async def solve_markdown_paper(markdown_text, citation_patterns=None):
    """Turn a markdown academic paper into appended solveit blocks for incremental reading."""
    
    if citation_patterns is None:
        citation_patterns = [
            r'\([A-Z][^)]*,\s*\d{4}[a-z]?(?:;\s*[^)]*\d{4}[a-z]?)*\)',  # (Author, 2020)
            r'\([A-Z][^)]*\s+\d{4}[a-z]?(?:;\s*[^)]*\d{4}[a-z]?)*\)'    # (Author 2020)
        ]
    
    # Extract references section (only until next ## header)
    refs_match = re.search(r'^## References\s*\n(.*?)(?=^##|\Z)', 
                          markdown_text, re.MULTILINE | re.DOTALL)
    refs_section = refs_match.group(1).strip() if refs_match else ""
    
    # Find all unique citations
    all_citations = {}
    counter = 1
    cells = split_markdown_into_cells(markdown_text)
    
    for cell in cells:
        for pattern in citation_patterns:
            for match in re.finditer(pattern, cell):
                citation = match.group(0)
                if citation not in all_citations:
                    all_citations[citation] = counter
                    counter += 1
    
    # Add full bibliography at start (collapsed)
    if refs_section:
        await add_msg(content=f"### Full References\n\n{refs_section}", 
                placement='at_end', i_collapsed=1)
    
    # Track sections: {header_id: set_of_citations}
    section_citations = {}
    current_header_id = None
    
    for cell in cells:
        if cell.startswith('##'):
            # Add header and remember its ID
            current_header_id = await add_msg(content=cell, placement='at_end')
            section_citations[current_header_id] = set()
        else:
            # Replace citations with footnotes
            modified_cell = cell
            for pattern in citation_patterns:
                for match in re.finditer(pattern, cell):
                    cite_text = match.group(0)
                    if cite_text in all_citations:
                        if current_header_id:
                            section_citations[current_header_id].add(cite_text)
                        modified_cell = modified_cell.replace(
                            cite_text, f"[{all_citations[cite_text]}]", 1
                        )
            
            await add_msg(content=modified_cell, placement='at_end')
    
    # Now go back and insert cheatsheets after each header
    for header_id, citations in section_citations.items():
        if citations:
            cheatsheet = "### Citations in this section\n\n" + "\n".join(
                f"[{all_citations[c]}]: {c}" 
                for c in sorted(citations, key=lambda x: all_citations[x])
            )
            await add_msg(content=cheatsheet, placement='add_after', 
                    id=header_id, i_collapsed=1)

## Academic Paper Processing for SolveIt

These functions convert markdown academic papers into SolveIt notebook cells optimized for incremental reading and AI-assisted study.

### `split_markdown_into_cells(markdown_text)`
Splits a markdown document into logical chunks:
- Preamble (everything before first header) → single cell
- Headers → individual cells
- Content under each header → split by paragraphs (double newlines)

### `solve_markdown_paper(markdown_text, citation_patterns=None)`
Processes an academic paper for reading in SolveIt:

**What it does:**
1. Extracts full References section → adds as collapsed cell at top
2. Finds all inline citations like `(Author, 2020)` or `(Author 2020)`
3. Replaces them with numbered footnotes: `[1]`, `[2]`, etc.
4. For each section, creates a collapsed "Citations in this section" cheatsheet
5. Places cheatsheets right after section headers (so AI sees citations in context)

**Why this design:**
- Collapsed references at top keep them accessible to AI when discussing any part
- Section-specific cheatsheets provide local context without cluttering reading
- Numbered citations `[1]` are less distracting than full inline citations
- Unique numbers across entire paper make citations searchable

This is working wonderfully. Now let's wrap everything needed to turn an Arxiv link/link to a pdf into a solveit paper into a single function called `solve_paper(url, **kwargs)`, making use of the `download_to_temp` function above, and the`o = await pdf2md(str(f))`.   

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠🧠

```python
async def solve_paper(url, **kwargs):
    """Download a paper from URL (arXiv or PDF) and process into solveit blocks.
    
    Args:
        url: arXiv abstract URL or direct PDF link
        **kwargs: passed to solve_markdown_paper (e.g., citation_patterns)
    """
    # Download PDF to temp file
    f = download_to_temp(url)
    
    # Convert to markdown
    o = await pdf2md(str(f))
    
    # Process into solveit blocks with citations
    solve_markdown_paper(o["markdown"], **kwargs)
```

Usage would be:
```python
await solve_paper("http://arxiv.org/abs/2207.02093")
```

Want me to add any error handling or additional options (like limiting to first N sections)?

An all in one wrapper:

In [ ]:
#| export
async def paper2solveit(url, **kwargs):
    """Download a paper from URL (arXiv or PDF) and process into solveit blocks.
    
    Args:
        url: arXiv abstract URL or direct PDF link
        **kwargs: passed to solve_markdown_paper (e.g., citation_patterns)
    """
    # Download PDF to temp file
    f = download_to_temp(url)
    
    # Convert to markdown
    o = await pdf2md(str(f))
    
    # Process into solveit blocks with citations
    await solve_markdown_paper(o["markdown"], **kwargs)

Presently all images downloaded and referenced inline are strewn in the base folder. Let's put them in a separate figures/ folder within the directory the command is run.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

Update `_save_md` so it creates `./figures`, saves images there, and rewrites the returned Markdown’s inline image references. `paper2solveit` can remain unchanged because `pdf2md` returns the modified result.

```python
def _save_md(r, stem, path):
    path = Path(path)
    figures = path / "figures"
    figures.mkdir(exist_ok=True, parents=True)

    markdown = r["markdown"]

    for name, data in r["images"].items():
        (figures / name).write_bytes(b64decode(data))
        markdown = markdown.replace(
            f"]({name})",
            f"](figures/{name})",
        )

    r["markdown"] = markdown
    (path / f"{stem}.md").write_text(markdown)
```

The existing wrapper then works as written:

```python
async def paper2solveit(url, **kwargs):
    """Download and convert an arXiv paper or PDF into SolveIt messages."""
    f = download_to_temp(url)
    o = await pdf2md(str(f))
    await solve_markdown_paper(o["markdown"], **kwargs)
```

This produces:

```text
current-directory/
├── tmp....md
└── figures/
    ├── _page_1_Figure_0.jpeg
    ├── _page_3_Figure_1.jpeg
    └── ...
```

References such as:

```markdown
![](_page_1_Figure_0.jpeg)
```

become:

```markdown
![](figures/_page_1_Figure_0.jpeg)
```

Please make these changes.

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

I’ll inspect the `_save_md` code cell with hash-verified line addresses, then replace only that function.
- ⏳ `py(code="await lnhashview_msg('26c3e6ec', start_l…")` ⏳


```json {.tool}
{
  "id": "fc_05a007d475bd1f44016a80e953d69087d19b1eb7d459b35644",
  "name": "py",
  "args": {
    "code": "await lnhashview_msg('26c3e6ec', start_line=45, end_line=58)"
  },
  "result": "45|61ba|            await asyncio.sleep(delay)\n46|1cef|    raise TimeoutError(f\"Polling timed out after {max_polls * delay}s\")\n47|1959|async def poll_markers(ds, key=None, max_polls=300, delay=2, verbose=False):\n48|2d41|    \"Poll multiple Marker API requests concurrently\"\n49|f452|    return await asyncio.gather(*[poll_marker(d, key, max_polls, delay, verbose) for d in ds])\n50|6fd9|@delegates(submit_marker)\n51|ec79|async def convert_pdf(fname=None, file=None, file_url=None, key=None, max_polls=300, delay=2, verbose=False, **kwargs):\n52|005a|    \"Submit and poll until complete, return result\"\n53|4dbd|    r = await submit_marker(fname=fname, file=file, file_url=file_url, key=key, **kwargs)\n54|877b|    return await poll_marker(r, key=key, max_polls=max_polls, delay=delay, verbose=verbose)\n55|2420|@delegates(submit_markers)\n56|7819|async def convert_pdfs(files=None, fnames=None, file_urls=None, key=None, max_polls=300, delay=2, verbose=False, **kwargs):\n57|84ba|    \"Submit multiple PDFs and poll all until complete\"\n58|487a|    rs = await submit_markers(files=files, fnames=fnames, file_urls=file_urls, key=key, **kwargs)"
}
```





The requested range ended before `_save_md`; I’ll inspect the next lines to capture the function exactly before editing.
- ⏳ `py(code="await lnhashview_msg('26c3e6ec', start_l…")` ⏳


```json {.tool}
{
  "id": "fc_04f8f07e3e65a6d7016a80e95780e4819683bdaa42ef2e63ce",
  "name": "py",
  "args": {
    "code": "await lnhashview_msg('26c3e6ec', start_line=58, end_line=72)"
  },
  "result": "58|487a|    rs = await submit_markers(files=files, fnames=fnames, file_urls=file_urls, key=key, **kwargs)\n59|61ea|    return await poll_markers(rs, key=key, max_polls=max_polls, delay=delay, verbose=verbose)\n60|36fc|def _save_md(r, stem, path):\n61|ec73|    (path/f'{stem}.md').write_text(r['markdown'])\n62|3927|    for nm,dt in r['images'].items(): (path/nm).write_bytes(b64decode(dt))\n63|0000|\n64|c322|@delegates(convert_pdf)\n65|784a|async def pdf2md(fname, path='.', **kwargs):\n66|ab39|    \"Convert PDF to markdown and save with images\"\n67|b1fb|    path = Path(path)\n68|bcb7|    path.mkdir(exist_ok=True, parents=True)\n69|fc5f|    r = await convert_pdf(fname=fname, **kwargs)\n70|338d|    _save_md(r, Path(fname).stem, path)\n71|7c7b|    return r\n72|0000|"
}
```





The target is lines 60–62. I’ll replace `_save_md` with the agreed version, preserving the existing `pdf2md` interface.
- ⏳ `py(code="await msg_exhash('26c3e6ec', [(r'''60|36…")` ⏳


```json {.tool}
{
  "id": "fc_075066f2b0e2dd90016a80e95be43c81969b8f522e9f0f3b49",
  "name": "py",
  "args": {
    "code": "await msg_exhash('26c3e6ec', [(r'''60|36fc|,62|3927|''', 'c', r'''def _save_md(r, stem, path):\n    path = Path(path)\n    figures = path / \"figures\"\n    figures.mkdir(exist_ok=True, parents=True)\n\n    markdown = r[\"markdown\"]\n    for name, data in r[\"images\"].items():\n        (figures / name).write_bytes(b64decode(data))\n        markdown = markdown.replace(f\"]({name})\", f\"](figures/{name})\")\n\n    r[\"markdown\"] = markdown\n    (path / f\"{stem}.md\").write_text(markdown)''')])"
  },
  "result": "--- original\n+++ modified\n 59|61ea|    return await poll_markers(rs, key=key, max_polls=max_polls, delay=delay, verbose=verbose)\n+60|36fc|def _save_md(r, stem, path):\n+61|b1fb|    path = Path(path)\n+62|d77b|    figures = path / \"figures\"\n+63|9ac8|    figures.mkdir(exist_ok=True, parents=True)\n+64|0000|\n+65|4a40|    markdown = r[\"markdown\"]\n+66|f03e|    for name, data in r[\"images\"].items():\n+67|fe0c|        (figures / name).write_bytes(b64decode(data))\n+68|5ca6|        markdown = markdown.replace(f\"]({name})\", f\"](figures/{name})\")\n+69|0000|\n+70|27c6|    r[\"markdown\"] = markdown\n+71|5bf8|    (path / f\"{stem}.md\").write_text(markdown)\n-60|36fc|def _save_md(r, stem, path):\n-61|ec73|    (path/f'{stem}.md').write_text(r['markdown'])\n-62|3927|    for nm,dt in r['images'].items(): (path/nm).write_bytes(b64decode(dt))\n 72|0000|"
}
```

Updated `_save_md` in #_26c3e6ec.

It now:

- creates a `figures/` directory beneath `path`;
- saves extracted images there;
- rewrites Markdown image references to `figures/<filename>`;
- updates `r["markdown"]`, so `paper2solveit` uses the corrected references;
- writes the corrected Markdown to disk.

Can you write an `__all__` declaration for this, including the pdfs2md and pdf2md function Jeremy's code?

##### 🤖Reply🤖<!-- SOLVEIT_SEPARATOR_7f3a9b2c -->

```python
__all__ = [
    'pdf2md', 'pdfs2md',  # Jeremy's Marker API wrappers
    'download_to_temp',
    'split_markdown_into_cells',
    'solve_markdown_paper',
    'solve_paper'
]
```

This exports the key user-facing functions while keeping the internal helpers (like `submit_marker`, `poll_marker`, etc.) private.

In [ ]:
#| export
__all__ = [
    'pdf2md', 'pdfs2md',  # Jeremy's Marker API wrappers
    'download_to_temp',
    'split_markdown_into_cells',
    'solve_markdown_paper',
    'paper2solveit',
    'list_zotero', 'zotero_pdf', 'zotero_download', 'ZoteroListing',  # Zotero library support
]

In [ ]:
!nbdev-export

### Reading papers from your Zotero library

`paper2solveit` needs a public URL, which rules out anything you can't link to directly: paywalled journal articles, preprints you've annotated, scans a colleague emailed you. But those are exactly the papers already sitting in Zotero.

So here's the same pipeline, fed from your library instead. Two calls: list a collection to see what's in it, then convert the one you want.

```python
list_zotero('Reading List')
await zotero_pdf(3)
```

This talks to the [Zotero web API](https://www.zotero.org/support/dev/web_api/v3/start), so it works from SolveIt's servers rather than needing Zotero installed alongside. Set two env vars, both from [zotero.org/settings/keys](https://www.zotero.org/settings/keys): `ZOTERO_USER_ID` (the number shown at the top of that page) and `ZOTERO_API_KEY` (create one, read-only is plenty).

If you'd rather run against the Zotero 7 desktop app on your own machine, pass `local=True` and skip the key — though note the local API is read-only and its file endpoint is less reliable than the web one.

In [ ]:
#| export
# Zotero support: pull a paper straight from your own library instead of a public URL.
import os, re
from pathlib import Path
from tempfile import NamedTemporaryFile

def _zotero_client(library_id=None, api_key=None, library_type='user', local=False):
    "Build a pyzotero client, defaulting to the `ZOTERO_USER_ID`/`ZOTERO_API_KEY` env vars."
    try: from pyzotero.zotero import Zotero
    except ImportError:
        try: from pyzotero import Zotero
        except ImportError: raise ImportError(
            "Zotero support needs pyzotero — install it with `pip install pyzotero`.") from None
    library_id = library_id or os.environ.get('ZOTERO_USER_ID')
    if not library_id: raise ValueError(
        "No Zotero library id: set ZOTERO_USER_ID (find yours at zotero.org/settings/keys) "
        "or pass library_id=.")
    if local: return Zotero(library_id, library_type, local=True)
    api_key = api_key or os.environ.get('ZOTERO_API_KEY')
    if not api_key: raise ValueError(
        "No Zotero API key: set ZOTERO_API_KEY (create one at zotero.org/settings/keys) "
        "or pass api_key=, or use local=True to talk to a running Zotero 7 desktop app.")
    return Zotero(library_id, library_type, api_key)

A listing is only useful if you can recognise the paper you want in it, so items are labelled the way you'd cite them: title, first author, year.

In [ ]:
#| export
def _year(d):
    "First 4-digit year in an item's date field, or '' if there isn't one."
    m = re.search(r'\d{4}', d.get('date') or '')
    return m.group(0) if m else ''

def _authors(d):
    "Short author string: 'Smith', 'Smith & Jones', or 'Smith et al.'"
    cs = d.get('creators') or []
    names = [c.get('lastName') or c.get('name') or '' for c in cs if c.get('creatorType') == 'author']
    names = [n for n in (names or [c.get('lastName') or c.get('name') or '' for c in cs]) if n]
    if not names: return ''
    if len(names) == 1: return names[0]
    if len(names) == 2: return f'{names[0]} & {names[1]}'
    return f'{names[0]} et al.'

def _describe(d):
    "One-line human label for a Zotero item."
    bits = [b for b in (_authors(d), _year(d)) if b]
    title = d.get('title') or d.get('name') or d.get('filename') or '(untitled)'
    return f"{title} — {', '.join(bits)}" if bits else title

def _label(entry):
    "How an entry appears in a listing: items get authors and a year, collections a size."
    d = entry['data']
    n = (entry.get('meta') or {}).get('numItems')
    if d.get('itemType') is None and n is not None: return f"{_describe(d)} ({n})"
    return _describe(d)

`ZoteroListing` is what you actually see. It renders as a numbered markdown list in SolveIt, and indexes **from 1** so the numbers on screen are the numbers you type.

In [ ]:
#| export
class ZoteroListing:
    "A numbered list of Zotero collections or items; renders as markdown, indexes like a list."
    def __init__(self, title, entries, kind):
        self.title, self.entries, self.kind = title, entries, kind
    def __len__(self): return len(self.entries)
    def __iter__(self): return iter(self.entries)
    def __getitem__(self, i):
        "1-based lookup, matching the numbers shown in the listing."
        if not 1 <= i <= len(self.entries): raise IndexError(
            f"{i} is outside the listing (1–{len(self.entries)}).")
        return self.entries[i-1]
    def _lines(self): return [f"{i}. {_label(e)}" for i, e in enumerate(self.entries, 1)]
    def _repr_markdown_(self):
        if not self.entries: return f"**{self.title}** — nothing here."
        return '\n'.join([f"**{self.title}**", ''] + self._lines())
    def __repr__(self): return '\n'.join([self.title] + self._lines())

`list_zotero()` with no argument shows your collections; with a name it shows that collection's contents. Either listing is cached, so the numbers stay meaningful for your next call — including `list_zotero(2)` to drill into collection 2 without retyping its name.

Names match case-insensitively, and a unique substring is enough: `list_zotero('reading')` finds "Reading List" as long as nothing else matches too.

In [ ]:
#| export
_last_listing = None  # what `list_zotero` showed most recently, so `zotero_pdf(n)` can resolve n

def list_zotero(collection=None, **kwargs):
    """List your Zotero collections, or the items inside one, numbered for use with `zotero_pdf`.

    Args:
        collection: a collection name (case-insensitive, partial matches allowed), or the
            number of a collection from the previous `list_zotero()` listing. Omit it to
            list the collections themselves.
        **kwargs: passed to `_zotero_client` (library_id, api_key, library_type, local)
    """
    global _last_listing
    zot = _zotero_client(**kwargs)
    if isinstance(collection, int): collection = _resolve_number(collection, 'collection')['data']['name']
    if collection is None:
        cols = sorted(zot.everything(zot.collections()), key=lambda c: c['data']['name'].lower())
        _last_listing = ZoteroListing('Zotero collections', cols, 'collection')
    else:
        col = _match_collection(zot, collection)
        items = [i for i in zot.everything(zot.collection_items_top(col['key']))
                 if i['data'].get('itemType') != 'note']
        _last_listing = ZoteroListing(f"{col['data']['name']} ({len(items)} items)", items, 'item')
    return _last_listing

def _resolve_number(n, kind):
    "Look up entry `n` in the last listing, checking it holds the kind of thing we want."
    if _last_listing is None: raise ValueError(
        f"No listing to index into — run `list_zotero()` first.")
    if _last_listing.kind != kind: raise ValueError(
        f"The last listing was of {_last_listing.kind}s, not {kind}s. "
        f"Run `list_zotero({'' if kind == 'collection' else 'collection_name'})` first.")
    return _last_listing[n]

def _match_collection(zot, name):
    "Find a collection by name: exact (case-insensitive) first, then substring."
    cols = zot.everything(zot.collections())
    exact = [c for c in cols if c['data']['name'].lower() == name.lower()]
    if len(exact) == 1: return exact[0]
    hits = exact or [c for c in cols if name.lower() in c['data']['name'].lower()]
    if not hits: raise ValueError(
        f"No Zotero collection matching {name!r}. Run `list_zotero()` to see them all.")
    if len(hits) > 1: raise ValueError(
        f"{name!r} matches several collections ({', '.join(c['data']['name'] for c in hits)}). "
        "Use the full name, or its number from `list_zotero()`.")
    return hits[0]

Now, getting the actual PDF. Two shapes to handle: a normal Zotero item with a PDF attached underneath it, and a bare PDF dropped straight into a collection (which is what an inbox tends to be full of). Items may also carry an HTML snapshot alongside the PDF, so we filter on `contentType` rather than taking the first attachment.

One caveat worth knowing about before it bites you: Zotero only serves files it's actually storing. If an attachment is a *linked* file, or you have file sync switched off, the metadata syncs but the PDF never leaves your machine — so the download fails with a message saying as much.

In [ ]:
#| export
def _pdf_key(zot, item):
    "Item key of the PDF to convert: the item itself if it's a PDF, else its first PDF child."
    d = item['data']
    if d.get('itemType') == 'attachment':
        if d.get('contentType') == 'application/pdf': return d['key']
        raise ValueError(f"{_describe(d)!r} is a {d.get('contentType')} attachment, not a PDF.")
    for ch in zot.children(d['key']):
        cd = ch['data']
        if cd.get('itemType') == 'attachment' and cd.get('contentType') == 'application/pdf':
            return cd['key']
    raise ValueError(f"No PDF attached to {_describe(d)!r}.")

def zotero_download(item, path=None, **kwargs):
    """Download a Zotero item's PDF, returning the path it was written to.

    Args:
        item: a number from the last `list_zotero` listing, a Zotero item key, or an item dict
        path: where to write the PDF (a temp file if omitted)
        **kwargs: passed to `_zotero_client`
    """
    zot = _zotero_client(**kwargs)
    if isinstance(item, int): item = _resolve_number(item, 'item')
    if isinstance(item, str): item = zot.item(item)
    key = _pdf_key(zot, item)
    try: content = zot.file(key)
    except Exception as e: raise RuntimeError(
        f"Couldn't download the PDF for {_describe(item['data'])!r} ({e}). Zotero serves files "
        "only when they're stored in your library and synced — linked-file attachments, and "
        "libraries with file sync off, aren't retrievable through the API."
    ) from e
    if path is None:
        tmp = NamedTemporaryFile(delete=False, suffix='.pdf')
        tmp.write(content); tmp.close()
        return Path(tmp.name)
    path = Path(path)
    path.write_bytes(content)
    return path

And the wrapper, mirroring `paper2solveit`: fetch, OCR, split into cells. Any `local`/`api_key`-ish arguments go to Zotero, everything else (like `citation_patterns`) goes on to `solve_markdown_paper`.

In [ ]:
#| export
async def zotero_pdf(item, **kwargs):
    """Turn a PDF from your Zotero library into solveit blocks.

    Pair it with `list_zotero`: list a collection, then pass the number of the paper you want.

        list_zotero('Reading List')
        await zotero_pdf(3)

    Args:
        item: a number from the last `list_zotero` listing, a Zotero item key, or an item dict
        **kwargs: split between `_zotero_client` (library_id, api_key, library_type, local)
            and `solve_markdown_paper` (e.g. citation_patterns)
    """
    zkw = {k: kwargs.pop(k) for k in ('library_id', 'api_key', 'library_type', 'local')
           if k in kwargs}
    f = zotero_download(item, **zkw)
    o = await pdf2md(str(f))
    await solve_markdown_paper(o["markdown"], **kwargs)

These are the fiddly bits — 1-based numbering, picking the PDF out of a pile of attachments, keeping a stale listing from silently resolving to the wrong paper. Rather than burn API calls checking them, here's a stand-in Zotero to test against.

In [ ]:
#| hide
# Exercise the listing/attachment logic against a stubbed Zotero — no network, no API key.
import sys, types

_COLS = [{'key': 'C1', 'data': {'key': 'C1', 'name': 'Reading List'}, 'meta': {'numItems': 2}},
         {'key': 'C2', 'data': {'key': 'C2', 'name': 'Inbox'}, 'meta': {'numItems': 0}}]
_ITEMS = {'C1': [
    {'key': 'I1', 'data': {'key': 'I1', 'itemType': 'journalArticle', 'title': 'Attention Is All You Need',
        'date': '2017-06-12', 'creators': [{'creatorType': 'author', 'lastName': n}
                                           for n in ('Vaswani', 'Shazeer', 'Parmar')]}},
    {'key': 'I2', 'data': {'key': 'I2', 'itemType': 'attachment', 'contentType': 'application/pdf',
        'title': 'A Standalone Scan'}},
    {'key': 'I3', 'data': {'key': 'I3', 'itemType': 'note', 'title': 'a note'}},
    {'key': 'I4', 'data': {'key': 'I4', 'itemType': 'book', 'title': 'No PDF Here', 'date': '1999'}}], 'C2': []}
_KIDS = {'I1': [{'data': {'key': 'A1', 'itemType': 'attachment', 'contentType': 'text/html'}},
                {'data': {'key': 'A2', 'itemType': 'attachment', 'contentType': 'application/pdf'}}], 'I4': []}

class _FakeZot:
    def __init__(self, *a, **kw): pass
    def collections(self): return list(_COLS)
    def collection_items_top(self, k): return list(_ITEMS[k])
    def everything(self, x): return x
    def children(self, k): return _KIDS.get(k, [])
    def file(self, k): return b'%PDF-1.4 ' + k.encode()

_m = types.ModuleType('pyzotero.zotero'); _m.Zotero = _FakeZot
sys.modules['pyzotero'] = types.ModuleType('pyzotero'); sys.modules['pyzotero.zotero'] = _m
_stub_env = [k for k in ('ZOTERO_USER_ID', 'ZOTERO_API_KEY') if k not in os.environ]
for _k in _stub_env: os.environ[_k] = 'stub'

cols = list_zotero()
assert [c['data']['name'] for c in cols] == ['Inbox', 'Reading List'], "collections sort by name"
assert cols[1]['key'] == 'C2', "listings index from 1, matching the numbers on screen"
assert '2. Reading List (2)' in cols._repr_markdown_()

items = list_zotero('reading')          # partial, case-insensitive name match
assert [i['key'] for i in items] == ['I1', 'I2', 'I4'], "notes are not readable papers"
assert '1. Attention Is All You Need — Vaswani et al., 2017' in items._repr_markdown_()
assert 'nothing here' in list_zotero('Inbox')._repr_markdown_()

_z = _FakeZot()
assert _pdf_key(_z, _ITEMS['C1'][0]) == 'A2', "prefer the PDF over the HTML snapshot"
assert _pdf_key(_z, _ITEMS['C1'][1]) == 'I2', "a bare PDF attachment is its own PDF"
for _bad, _msg in ((_ITEMS['C1'][2], 'note'), (_ITEMS['C1'][3], 'nothing attached')):
    try: _pdf_key(_z, _bad); raise AssertionError(f"{_msg} should not yield a PDF")
    except ValueError: pass

list_zotero('Reading List')
assert zotero_download(1).read_bytes() == b'%PDF-1.4 A2'
list_zotero()                            # last listing is collections again...
try: zotero_download(1); raise AssertionError("stale listing should not resolve to a paper")
except ValueError as e: assert 'not items' in str(e)

del sys.modules['pyzotero'], sys.modules['pyzotero.zotero']
# Drop the stub credentials again, so a later real call still gets the 'set ZOTERO_API_KEY' hint.
for _k in _stub_env: del os.environ[_k]
print('zotero helpers: ok')

To use it, list a collection and then convert by number:

```python
list_zotero()                 # which collections do I have?
list_zotero('Reading List')   # what's in this one?
await zotero_pdf(3)           # convert #3 into solveit cells
```

`zotero_pdf` also takes a Zotero item key directly (`await zotero_pdf('BM8MZJBB')`) if you'd rather not depend on the last listing.

In [ ]:
await paper2solveit("https://arxiv.org/abs/2504.20997v2")

# TOWARD EFFICIENT EXPLORATION BY LARGE LANGUAGE MODEL AGENTS

**Dilip Arumugam**  
Department of Computer Science  
Princeton University  
dilip.a@cs.princeton.edu

**Thomas L. Griffiths**  
Department of Computer Science  
Department of Psychology  
Princeton University  
tomg@princeton.edu

## ABSTRACT

A burgeoning area within reinforcement learning (RL) is the design of sequential decision-making agents centered around large language models (LLMs). While autonomous decision-making agents powered by modern LLMs could facilitate numerous real-world applications, such successes demand agents that are capable of data-efficient RL. One key obstacle to achieving data efficiency in RL is exploration, a challenge that we demonstrate many recent proposals for LLM agent designs struggle to contend with. Meanwhile, classic algorithms from the RL literature known to gracefully address exploration require technical machinery that can be challenging to operationalize in purely natural language settings. In this work, rather than relying on finetuning or in-context learning to coax LLMs into implicitly imitating a RL algorithm, we illustrate how LLMs can be used to explicitly implement an existing RL algorithm (Posterior Sampling for Reinforcement Learning) whose capacity for statistically-efficient exploration is already well-studied. We offer empirical results demonstrating how our LLM-based implementation of a known, data-efficient RL algorithm can be considerably more effective in natural language tasks that demand prudent exploration.

## 1 INTRODUCTION

### Citations in this section

[1]: (Bommasani et al., 2021; Achiam et al., 2023; Touvron et al., 2023; Team et al., 2023; Hurst et al., 2024; Jaech et al., 2024)
[2]: (Silver & Sutton, 2025)
[3]: (Yao et al., 2023; Shinn et al., 2024; Monea et al., 2024; Klissarov et al., 2025)
[4]: (Sutton & Barto, 1998)
[5]: (Strens, 2000; Osband et al., 2013)

Large language models (LLMs) have rapidly permeated many areas of machine learning, demonstrating proficiency across a broad range of tasks [1]. This has inspired recent work studying how LLMs can best be used to solve sequential decision-making problems [2]. These efforts have led to the introduction of new designs for LLM agents that aim to learn optimal behavior through trial-and-error interaction within natural language environments [3]. While details vary by approach, broadly speaking these new agent designs involve one or more LLMs that interact to ultimately select actions within the environment. However, such agents still reside in the classic RL setting [4] and, consequently, must still grapple with the fundamental obstacles to data efficiency (generalization, exploration, and credit assignment) that the RL literature has studied for decades.

While composing LLMs to arrive at new agent designs is the current norm, we propose that an alternative strategy is to re-examine existing RL algorithms and consider how LLMs might implement them in otherwise inaccessible environments. An RL algorithm consists of specifying inputs and detailing a sequence of steps for determining behavior at each time period. Why should the emergence and proliferation of LLMs change the fundamental principles of agent design? Instead, as visualized in Figure 1, perhaps LLMs can be used to create new, potentially-inexact incarnations of existing RL algorithms via the subroutines needed to implement them.

In this work, we focus on data-efficient RL with LLMs and isolate the key challenge of exploration. We demonstrate how modern LLMs afford a contemporary implementation of an existing RL algorithm, Posterior Sampling for Reinforcement Learning (PSRL) [5], that is both well-studied and whose capacity for good exploration is already known to yield provably-efficient RL in a number of problem classes. We empirically find that our LLM-based

![Figure 1: Comparison of RL agent design principles. Left: Existing approach where multiple LLMs (represented by icons with infinity symbols) are orchestrated by an agent design (large arrow) to perform a sequence of steps (numbered 1, 2, 3). Right: Novel approach where an existing RL algorithm (represented by a box with steps 1, 2, 3) is explicitly implemented by outsourcing individual steps to distinct LLMs (represented by icons with infinity symbols).](9ba3dc91984c80b96f217fb1bddd5c06_img.jpg)

Figure 1: Comparison of RL agent design principles. Left: Existing approach where multiple LLMs (represented by icons with infinity symbols) are orchestrated by an agent design (large arrow) to perform a sequence of steps (numbered 1, 2, 3). Right: Novel approach where an existing RL algorithm (represented by a box with steps 1, 2, 3) is explicitly implemented by outsourcing individual steps to distinct LLMs (represented by icons with infinity symbols).

Figure 1: Abstractly, an RL algorithm is an ordered sequence of steps. Existing approaches for LLM agent design (left) orchestrate some number of LLMs to implicitly induce a RL algorithm. In contrast, this paper advocates for a novel agent design principle (right) whereby an existing RL algorithm is explicitly implemented by outsourcing individual steps to distinct LLMs.

implementation of PSRL retains the strong exploration properties that, up to this point, have not only been primarily restricted to tabular domains but also been absent in recent designs for LLM agents. We further observe that the choice of LLM underlying the PSRL implementation matters and, in an environment with stochastic transition dynamics, show that upgrading to a more capable model (GPT-4o to o1-mini) is the difference between incurring linear regret and obtaining cumulative regret on par with classic PSRL. Altogether, our work underscores the importance of addressing exploration in the design of LLM agents, illustrates the considerable value that decades of RL research have to offer data-efficient decision-making with LLMs, and establishes a key distinction between LLMs that implement a RL algorithm versus a RL algorithm that is implemented with LLMs.

## 2 PROBLEM FORMULATION

### Citations in this section

[6]: (Bellman, 1957; Puterman, 1994)

All random variables are defined on a probability space  $(\Omega, \mathcal{F}, \mathbb{P})$ . For any arbitrary set  $\mathcal{X}$ , we use  $\Delta(\mathcal{X})$  to denote the set of all probability distributions with support on  $\mathcal{X}$ . For any  $N \in \mathbb{N}$ , we denote the index set as  $[N] = \{1, 2, \dots, N\}$ .

We formulate a sequential decision-making problem as a finite-horizon, episodic Markov Decision Process (MDP) [6] defined by  $\mathcal{M} = (\mathcal{S}, \mathcal{A}, \mathcal{R}, \mathcal{T}, \beta, H)$ ,  $\mathcal{S}$  is a set of states,  $\mathcal{A}$  is a set of actions,  $\mathcal{R} : \mathcal{S} \times \mathcal{A} \to [0, 1]$  is a reward function providing evaluative feedback in the unit interval,  $\mathcal{T} : \mathcal{S} \times \mathcal{A} \to \Delta(\mathcal{S})$  is a transition function prescribing distributions over next states,  $\beta \in \Delta(\mathcal{S})$  is an initial state distribution, and  $H \in \mathbb{N}$  is the maximum episode length or horizon. Within each of  $K \in \mathbb{N}$  total episodes, the agent acts for  $H$  steps beginning with an initial state  $s_1 \sim \beta(\cdot)$  and, at each timestep  $h \in [H]$ , observes the current state  $s_h \in \mathcal{S}$ , selects an action  $a_h \in \mathcal{A}$ , enjoys a reward  $r_h = \mathcal{R}(s_h, a_h)$ , and transitions to a next state  $s_{h+1} \sim \mathcal{T}(\cdot | s_h, a_h)$ .

An agent is characterized by its non-stationary, stochastic policy  $\pi : \mathcal{S} \times [H] \to \Delta(\mathcal{A})$ , which encodes a pattern of behavior by mapping individual states and the current timestep to a probability distribution over actions. We assess the performance of a policy  $\pi$  in MDP  $\mathcal{M}$  at timestep  $h \in [H]$  when starting at state  $s \in \mathcal{S}$  and taking action  $a \in \mathcal{A}$  by its associated action-value function

$$Q_{\mathcal{M}, h}^\pi(s, a) = \mathbb{E} \left[ \sum_{h'=h}^{H} \mathcal{R}(s_{h'}, a_{h'}) \mid s_h = s, a_h = a \right]. \text{ Taking the value function as } V_{\mathcal{M}, h}^\pi(s) = \mathbb{E}_{a \sim \pi_h(\cdot | s)} \left[ Q_{\mathcal{M}, h}^\pi(s, a) \right], \text{ we define the optimal policy } \pi^* \text{ as achieving supremal value } V_{\mathcal{M}, h}^*(s) = \sup_{\pi \in \Pi} V_{\mathcal{M}, h}^\pi(s) \text{ for all } s \in \mathcal{S}, h \in [H] \text{ where } \Pi \text{ denotes the class of all non-stationary, stochastic } \pi \in \Pi.$$

policies. For any episode  $k \in [K]$ , we let  $\tau_k = (s_1^{(k)}, a_1^{(k)}, r_1^{(k)}, \dots, s_H^{(k)}, a_H^{(k)}, r_H^{(k)}, s_{H+1}^{(k)})$  denote the random trajectory experienced by the agent executing its policy in the environment. Meanwhile,  $H_k = \{\tau_1, \tau_2, \dots, \tau_{k-1}\} \in \mathcal{H}$  is the entire random history of interaction at the  $k$ th episode.

Abstractly, a RL algorithm is a sequence  $\{\pi^{(k)}\}_{k \in [K]}$  where the policy deployed at each episode  $\pi^{(k)}$  is a function of the current history  $H_k$ . We may evaluate the performance of a RL algorithm on MDP  $\mathcal{M}$  via its cumulative regret:  $\text{REGRET}(\{\pi^{(k)}\}_{k \in [K]}, \mathcal{M}) =$

$$\mathbb{E} \left[ \sum_{k=1}^{K} (V_{\mathcal{M}, 1}^*(s_1) - V_{\mathcal{M}, 1}^{\pi^{(k)}}(s_1)) \mid \mathcal{M} \right], \text{ which aggregates performance shortfall between an agent's chosen policy and the optimal policy in all episodes. Naturally, an agent designer seeks out a RL algorithm with minimal cumulative regret.}$$

## 3 LLM IMPLEMENTATION OF POSTERIOR SAMPLING FOR REINFORCEMENT LEARNING

### Citations in this section

[4]: (Sutton & Barto, 1998)
[7]: (Bellman & Kalaba, 1959; Duff, 2002; Ghavamzadeh et al., 2015)
[8]: (Bellman & Kalaba, 1959; Duff, 2002)
[9]: (Gittins, 1979)
[10]: (Duff, 2002; Arumugam & Singh, 2022)
[11]: (Lu et al., 2023)
[12]: (Der Kiureghian & Ditlevsen, 2009)

One of the major obstacles to data-efficient RL is exploration, where a learner must determine what data to collect from the environment to maximize long-term performance. While much of the early work on addressing exploration in RL (see Appendix A for a detailed review of prior work) adhered to “optimism in the face of uncertainty,” an alternative is to proceed in a Bayesian fashion.

The Bayesian RL setting [7] recognizes that the underlying MDP  $\mathcal{M}$  is entirely unknown to the agent and, therefore, a random variable. The agent is thus endowed with a prior distribution  $\mathbb{P}(\mathcal{M})$  to reflect initial uncertainty in the true MDP. While the standard RL objective [4] calls for an agent to minimize regret, another performance criterion is the Bayesian regret, which simply integrates out the randomness in  $\mathcal{M}$  with respect to an agent’s prior:  $\text{BAYESREGRET}(\{\pi^{(k)}\}_{k \in [K]}) = \mathbb{E}[\text{REGRET}(\{\pi^{(k)}\}_{k \in [K]}, \mathcal{M})]$ . We make a standard assumption that the prior is well-specified and the true MDP resides in its support.

Unfortunately, the canonical Bayes-Adaptive MDP (BAMDP) [8] that encapsulates the full Bayesian RL problem is often computationally-intractable even in the simplest classes of environments with precious few exceptions [9]. This is a direct consequence of the intractably-large BAMDP hyperstate space [10], in which traditional MDP states are folded in alongside *epistemic states* [11] that contain an agent’s beliefs and epistemic uncertainty [12] about the world. The MDP transition and reward functions are unknown to a RL agent and, with each step taken in the true environment, the resulting reward and next-state transition provide ground-truth observations by which the agent may refine posterior beliefs about the underlying MDP  $\mathcal{M}$ . Even for a simple finite MDP, the epistemic state space is exponentially-large in the problem horizon  $H$ . One might hope that the epistemic state could be lazily updated while still enabling strategic exploration by reducing epistemic uncertainty; this insight is the basis of posterior-sampling methods in RL.

### 3.1 THE CLASSIC APPROACH

### Citations in this section

[13]: (Strens, 2000)
[14]: (Thompson, 1933; Russo & Van Roy, 2014; 2016; Russo et al., 2018)
[15]: (Osband et al., 2013; Osband & Van Roy, 2014; Abbasi-Yadkori & Szepesvari, 2014; Osband & Van Roy, 2016; Agrawal & Jia, 2017; Ouyang et al., 2017; Osband & Van Roy, 2017; Lu & Van Roy, 2019; Arumugam & Van Roy, 2022; Xu et al., 2024)
[16]: (Osband et al., 2013)
[17]: (Osband et al., 2016a; Lu & Van Roy, 2017; Osband et al., 2018; O’Donoghue et al., 2018; Dwivedi et al., 2020; Osband et al., 2023; Sasso et al., 2023)
[18]: (Osband et al., 2016b; 2019)
[19]: (Mazumdar et al., 2020; Karbasi et al., 2023; Ishaq et al., 2024; Jorge et al., 2024)
[20]: (Kaiser et al., 2020)

The promise of Bayesian RL methods is to facilitate statistically-efficient exploration by reducing an agent’s epistemic uncertainty about the world. One strategy for reaping the benefits of uncertainty-based exploration in a computationally-tractable manner is through Posterior Sampling for RL (PSRL) [13], presented as Algorithm 1. Rather than updating the epistemic state at each timestep, PSRL holds it fixed during each episode and only updates the posterior at the end using the full trajectory  $\tau_k$ . To govern action selection within each episode based on current knowledge of the true underlying MDP  $\mathbb{P}(\mathcal{M} \mid H_k)$ , PSRL employs Thompson sampling (TS) [14], whereby the agent draws one posterior sample as a statistically-plausible hypothesis about the true MDP (Line 3) and proceeds to act optimally with respect to it by executing the sampled MDP optimal policy (Lines 4–5). It has been shown theoretically that, by iteratively employing TS in this manner, PSRL is able to achieve strong exploration and satisfy Bayesian regret upper bounds for statistically-efficient RL in tabular MDPs and beyond [15]. A key contribution of this work is expanding empirical support for PSRL, an algorithm that has largely been a method of theoretical study up to this point.

While PSRL enjoys nice theoretical guarantees, practical implementations extending beyond tabular MDPs [16] face significant computational hurdles. Representing and maintaining epistemic uncertainty about the underlying MDP transition and reward functions is an open challenge in high-dimensional environments. While some work has studied using neural networks to address the broader problem of uncertainty estimation for guiding exploration in RL [17], the overwhelming majority of these efforts have concentrated on a model-free analogue of PSRL that maintains a Bayesian posterior over the optimal action-value function  $Q^*$  [18] in lieu of the underlying MDP  $\mathcal{M}$ . Meanwhile, the minority of such methods that actually strive to implement PSRL have either been met with mixed results across hard-exploration problems or have been limited to evaluations in smaller-scale domains.

**Algorithm 1** Posterior Sampling for Reinforcement Learning (PSRL) [13]

```

1: Input: Prior  $\mathbb{P}(M \in \cdot)$ 
2: for  $k \in [K]$  do
3:   Sample  $M_k \sim \mathbb{P}(M \in \cdot | H_k)$ 
4:   Obtain optimal policy  $\pi^{(k)} \equiv \pi_{M_k}^*$ 
5:   Execute  $\pi^{(k)}$  and get trajectory  $\tau_k$ 
6:   Update history  $H_{k+1} \equiv H_k \cup \tau_k$ 
7:   Induce posterior  $\mathbb{P}(M \in \cdot | H_{k+1})$ 
8: end for

```

Figure 2: The PSRL algorithm with LLM sub-routines of **posterior sampling**, **optimal behavior** with respect to a sample, and **posterior updating** shown. Dotted arrows show data flow.![Figure 3: Examples of a posterior (top) and posterior sample (bottom) generated by our LLM-based PSRL in Wordle. The top panel shows a 6x6 grid of letters with some cells highlighted in green and others in red. The bottom panel shows a similar grid where the letters are more clearly defined. Two text boxes provide context: the top one explains the search for the word 'HELPER' and the bottom one explains the search for the word 'HELPER' again, showing the progression of the search.](b615ff07e8a0f467f0a6f4783c4463eb_img.jpg)

Figure 3: Examples of a posterior (top) and posterior sample (bottom) generated by our LLM-based PSRL in Wordle. The top panel shows a 6x6 grid of letters with some cells highlighted in green and others in red. The bottom panel shows a similar grid where the letters are more clearly defined. Two text boxes provide context: the top one explains the search for the word 'HELPER' and the bottom one explains the search for the word 'HELPER' again, showing the progression of the search.

Figure 3: Examples of a posterior (top) and posterior sample (bottom) generated by our LLM-based PSRL in Wordle

Among them is a line of work that leans heavily into the use of Langevin dynamics for recovering the strategic exploration of PSRL [19]; in the context of this paper, such technical machinery is incredibly challenging and nontrivial to combine or even emulate with LLM agents.

In parallel, beyond the difficulties of maintaining a PSRL agent’s posterior distribution over the true MDP, computing the optimal policy for the posterior sample drawn in each episode constitutes an additional challenge that requires solving a planning problem. While there has been progress and even notable successes in this space for deep model-based RL agents [20], it is unclear if those methods are readily applicable to the natural language tasks faced by LLM agents. In our experiments, while we report positive results for our LLM-based PSRL implementation in MDPs with both deterministic and stochastic transition functions, performance in the latter type of environment eventually deteriorates as the size of the state-action space increases and exacerbates poor LLM planning capabilities under stochastic dynamics (see Appendix D).

### 3.2 A LLM IMPLEMENTATION

### Citations in this section

[11]: (Lu et al., 2023)
[21]: (Nie et al., 2024; Krishnamurthy et al., 2024; Klissarov et al., 2025; Ke et al., 2024)
[22]: (Brown et al., 2020)
[23]: (Nie et al., 2024)
[24]: (Lu et al., 2023; Arumugam & Van Roy, 2022)
[25]: (Wei et al., 2022; Kojima et al., 2022)

The key contribution of this paper is recognizing that LLMs can be operationalized to provide basic, atomic functions from which PSRL may be implemented. This stands in stark contrast to existing strides (see Appendix A) towards efficient decision-making with LLM agents [21], which either leave a LLM to its own devices for strategizing exploration or expect in-context learning (ICL) [22] to emulate the exploration of an existing RL or bandit algorithm. While future LLMs may become sufficiently capable to accommodate the former, our experiments today suggest this is not the case for simple, natural-language tasks where efficient exploration is paramount to success; by the same token, we anticipate that our proposed LLM-based implementation of PSRL will also benefit and gracefully extend to more complex natural language tasks as the constituent LLM models become more capable at performing their requested functions. Indeed, we find this to be the case empirically when applying our approach to MDPs with stochastic transition functions. LLM agents emulating the outputs of classic RL methods [23] are also bound to the same traditional problem classes whereas LLM-based implementations of RL algorithms may broaden the footprint of those classic algorithms to include natural-language domains that would otherwise be entirely infeasible.

As shown in Algorithm 1, our proposed implementation of PSRL relies on LLMs to play three distinct roles: (1) an approximate posterior updater, (2) a posterior sampler, and (3) an optimal policy with respect to a posterior sample. PSRL requires a prior distribution over MDPs as input and, more generally in any episode, needs a current posterior that accurately reflects the agent’s current knowledge *and* uncertainty about the world. For our purposes, such an approximate “posterior”<sup>1</sup> is a textual description that summarizes both the known and uncertain aspects of the true MDP transition and reward function. More importantly, it also explicitly communicates (in some way) the amount of uncertainty an agent has about these aspects of the world. As this textual summary amounts to the PSRL agent’s epistemic state representation [11], an agent designer may exert

<sup>1</sup>For ease of exposition, we will refer to this object as a posterior throughout the remainder of the paper, but acknowledge the distinction between it and the true, statistical object that is the Bayesian posterior distribution.

strong influence over this representation through the verbiage and expression of prior knowledge; as a concrete example, specifying the next-state transition distribution of a tabular MDP in our experiments as a Dirichlet distribution (in language) naturally encourages the LLM-based implementation of PSRL to maintain visitation counts. Of course, an advantage is that agent designers may now leverage the full expressivity and fluidity of natural language for communicating prior knowledge without restriction to the few statistical distributions that afford the computational conveniences of conjugate priors.

Given a current posterior reflecting the agent’s knowledge and uncertainty about the world, PSRL must be able to draw one posterior sample from these beliefs. We implement this as a first LLM that, given the agent’s current textual posterior (initially set to be the agent designer’s input prior) is tasked with generating a plausible hypothesis for how transitions and rewards unfold. In some domains, such as tabular MDPs, it may be natural for this to be an exhaustive list of rewards and next-state transitions for each state-action pair. For more practical scenarios of interest, however, it may be beneficial to prompt this posterior sampling LLM so that it can leverage an environment proxy or lossy surrogate MDP [24] that retains only the salient details needed to determine (near-)optimal behavior. As a concrete example, one of our natural language tasks is the game of Wordle (shown in Figure 3) that, as a MDP, has a transition function and reward function defined entirely around an unknown, five-letter target word. Here, the target word serves as an environment proxy that our LLM-based PSRL agent may directly monitor uncertainty over without meticulously maintaining statistics for rewards and transitions of individual state-action pairs.

With a single posterior sample in hand, a PSRL agent must be able to select actions that would be considered optimal if the sampled MDP truly reflected reality. We implement this as a second LLM tasked with executing actions given the current state that maximize value in a way that is consistent with the natural language hypothesis generated by the posterior sampling LLM. In the simplest case, this optimal sample policy LLM need only be given the posterior sample along with the current state and asked directly to generate an action. In more challenging settings, an agent designer may architect the LLM more carefully via chain-of-thought prompting [25] to increase the chance of selecting optimal actions consistent with provided hypothesis. Even when this policy is only approximately-optimal with respect to the posterior sample in a given episode, classic PSRL still admits a Bayesian regret bound (see Section 5.4 of Osbard (2016a)) and one might hope to see an LLM-based implementation of PSRL empirically exhibit similar robustness in practice.

Upon the completion of an episode with the optimal sample policy LLM acting with respect to the hypothesis of the posterior sampling LLM, we task a third and final LLM with updating the PSRL agent’s knowledge and residual uncertainty about the world, akin to an (approximate) posterior update. Given a complete trajectory consisting of reward signals and next-state transitions for exactly  $H$  state-action pairs, this posterior LLM must reconcile the agent’s prior knowledge at the start of the episode against observed interactions from within the environment. With this last piece of functionality in place, all three LLMs can then be orchestrated to run the PSRL algorithm.

## 4 EXPERIMENTS & DISCUSSION

### Citations in this section

[26]: (Hurst et al., 2024)
[27]: (Brooks et al., 2023)
[28]: (Howard, 1960)
[29]: (Monea et al., 2024)
[30]: (Shinn et al., 2024)

The goal of our experiments is assessing the extent to which our proposed LLM-based PSRL implementation not only retains the desirable exploration properties that PSRL exhibits empirically within simpler problem domains but also expands the range of problems where these benefits can be realized. To this end, we focus our evaluation on tasks which demand prudent exploration to achieve success and where an agent is minimally encumbered by the orthogonal challenges of generalization and credit assignment. For each task, we present cumulative regret curves (lower, flatter plots indicate better performance) where any shading denotes one standard error. All agents use GPT-4o [26] for their constituent LLMs unless otherwise indicated. We let  $\kappa_{\text{sampling}}$ ,  $\kappa_{\pi^*}$ , and  $\kappa_{\text{posterior}}$  denote the temperatures of the posterior sampling, optimal sample policy, and posterior update LLMs, respectively. Due to space constraints, we defer further details of our experiments and all prompts used in each task to the Appendix.

For natural language tasks, we compare our LLM-based implementation of PSRL against three baseline LLM agents. In-Context Policy Iteration (ICPI) [27] takes classic policy iteration [28] and offers an implementation via three LLMs, using ICL to elicit a rollout

policy; transition function; and reward function respectively. Together, these models allow for policy improvement via greedy action selection  $\pi^{(k)}(s_h) = \arg\max_{a \in \mathcal{A}} Q_{\mathcal{M}}^{(k-1)}(s_h, a)$ , with ties broken randomly. In-Context RL (ICRL) [29] aims to explore via the stochasticity in LLM responses from sensitivity to the input ICL data. Which episodes are included from a replay buffer for ICL with a LLM policy at each timestep is determined by sampling independent Bernoulli( $p$ ) random variables; we study three distinct values of the keep probability  $p \in \{1, 0.5, 0.1\}$ . Finally, Reflexion [30] passes each full trajectory through a self-reflection LLM that generates verbal guidance; the total history of verbal guidance is given at each timestep to the LLM policy, along with the current state, for improving the quality of decision-making.

### 4.1 MULTI-ARMED BANDITS

#### 4.1.1 BERNOLLI BANDIT

### Citations in this section

[31]: (Coda-Forno et al., 2023; Binz & Schulz, 2023; Coda-Forno et al., 2024; Krishnamurthy et al., 2024; Nie et al., 2024)
[32]: (Lai & Robbins, 1985; Bubeck & Cesa-Bianchi, 2012; Lattimore & Szepesvári, 2020)

Following prior work studying the exploratory capabilities of LLMs [31], we begin the empirical assessment of our LLM-based PSRL with a multi-armed bandit problem [32]. Readers unfamiliar with multi-armed bandits may simply observe them as a special case of a MDP with horizon  $H = 1$ , singleton state space  $|\mathcal{S}| = 1$ , and a stochastic (rather than deterministic) reward function. Our evaluation follows that of Krishnamurthy et al. (2024) who chose the simple yet challenging case of a five-armed **Bernoulli bandit** with independent arms and an action gap of 0.2.<sup>2</sup> The version we evaluate has one randomly-selected optimal arm with rewards drawn from a Bernoulli(0.6) distribution while all other arms use a Bernoulli(0.4).

Observe that PSRL specialized to a multi-armed bandit problem mirrors classic TS where, at each timestep, the agent samples one plausible hypothesis for the reward distribution of each arm and then proceeds to select the optimal action believed to achieve highest mean reward under this hypothesis. We compare PSRL implemented with LLMs to classic TS for a Bernoulli bandit with each arm initialized with a  $\text{Beta}(1, 1)$  prior. Meanwhile, our LLM-based PSRL agent begins with a prior for each arm specified as a  $\text{Beta}(1, 1)$  in natural language. While we fix temperatures  $\kappa_{\pi^*} = \kappa_{\text{posterior}} = 1$ , we find that the posterior sampling temperature has profound impact on the performance of our LLM-based PSRL agent. Figure 4 compares TS (run for 1,000 independent trials) against PSRL with four distinct settings of  $\kappa_{\text{sampling}}$  (run for 20 independent trials).

![Figure 4: Cumulative regret curves for a 5-armed Bernoulli bandit. The plot shows Cumulative Regret (y-axis, 0 to 10) versus Time Period (x-axis, 0 to 100). The legend includes: TS (blue line), PSRL + LLMs (kappa_sampling = 0.5) (orange line), PSRL + LLMs (kappa_sampling = 1) (green line), PSRL + LLMs (kappa_sampling = 1.1) (red line), and PSRL + LLMs (kappa_sampling = 1.2) (purple line). The PSRL variants show significantly lower cumulative regret than TS, with higher kappa_sampling values performing better.](e1dda754c2c88a8ad0b968aea4fc0786_img.jpg)

Figure 4: Cumulative regret curves for a 5-armed Bernoulli bandit. The plot shows Cumulative Regret (y-axis, 0 to 10) versus Time Period (x-axis, 0 to 100). The legend includes: TS (blue line), PSRL + LLMs (kappa\_sampling = 0.5) (orange line), PSRL + LLMs (kappa\_sampling = 1) (green line), PSRL + LLMs (kappa\_sampling = 1.1) (red line), and PSRL + LLMs (kappa\_sampling = 1.2) (purple line). The PSRL variants show significantly lower cumulative regret than TS, with higher kappa\_sampling values performing better.

Figure 4: Cumulative regret curves for a 5-armed Bernoulli bandit.

![Figure 5: Cumulative regret curves for the real-world customer service bandit. The plot shows Cumulative Regret (y-axis, 0 to 16) versus Episode (x-axis, 0.0 to 20.0). The legend includes: PSRL + LLMs (ours) (orange line), PSRL + LLMs (ours; well-specified prior) (green line), Reflexion (blue line), and ICRL (p = 1.0) (red line). PSRL + LLMs (ours) shows the lowest cumulative regret, followed by Reflexion, ICRL, and PSRL + LLMs (ours; well-specified prior).](80530169c7b6298ce012a85b906caeb3_img.jpg)

Figure 5: Cumulative regret curves for the real-world customer service bandit. The plot shows Cumulative Regret (y-axis, 0 to 16) versus Episode (x-axis, 0.0 to 20.0). The legend includes: PSRL + LLMs (ours) (orange line), PSRL + LLMs (ours; well-specified prior) (green line), Reflexion (blue line), and ICRL (p = 1.0) (red line). PSRL + LLMs (ours) shows the lowest cumulative regret, followed by Reflexion, ICRL, and PSRL + LLMs (ours; well-specified prior).

Figure 5: Cumulative regret curves for the real-world customer service bandit.

We find that our LLM-based PSRL achieves a better cumulative regret curve (with  $\kappa_{\text{sampling}} = 1.2$ ) than classic TS, for the limited time horizon of  $T = 100$ . We find that supplying PSRL with an initial prior of  $\text{Beta}(1, 1)$  in language automatically encourages the posterior update LLM to update binary reward observation counts for the chosen arm in each time period. Moreover, we find that the

<sup>2</sup>The action gap is defined as the difference in expected reward between the best and second best action. Larger action gaps make it easier to identify the optimal arm with few samples whereas smaller action gaps demand greater exploration.

optimal sample policy LLM has little difficulty in examining the sequence of expected reward values for each arm generated by the posterior sampling LLM and adhering to select the perceived best action. Manipulating  $\kappa_{\text{sampling}}$  shows that even values as large as 1 lead to greedy-like exploration in many trials where the resulting posterior sample favors the action observed to yield the most successes thus far. For a limited number of trials, this error proves to be not so catastrophic for temperatures of at least 1, though we would anticipate linear regret after more time periods. We find that increasing  $\kappa_{\text{sampling}} > 1$  yields exploratory behavior more aligned with TS where optimal actions more likely to be taken in the later time periods and there is a more gradual reduction of probability mass from other actions (see Appendix B).

#### 4.1.2 NATURAL LANGUAGE BANDIT

To demonstrate one concrete instance of how our proposed LLM-based PSRL may meet the demands of a real-world decision-making problem, we adapt the **customer service task** of Tajwar et al. (2025) into a multi-armed bandit problem. In each of  $K = 20$  total time periods, the agent may either ask a question or offer a solution to address a customer issue randomly sampled from the dataset<sup>3</sup> of Tajwar et al. (2025). Similar to Tajwar et al. (2025), we use two additional LLMs to simulate the customer (who answers the agent’s questions and tries suggested solutions as a non-technical person would) and to be a judge/reward function who ultimately determines the binary reward indicating successful resolution of a customer’s issue. All models use GPT-4o as the underlying LLM.

For our LLM-based PSRL, we consider two methods for specifying the prior distribution that PSRL takes as input. In the first case, we simply ask GPT-4o to provide a prior distribution (a list of plausible underlying issues for the customer complaint as well as guessed probabilities based on how likely the model perceives the issue to be) that is given directly as input to our LLM-based PSRL agent. In preliminary experiments we found that, while this agent is capable of finding success often, it can suffer from issues of prior misspecification, where the true solution (also given in the dataset of Tajwar et al. (2025)) is not within the support of the LLM-generated input prior. To remedy this without giving away the answer, we use a second method of generating an input prior that guarantees it is well-specified; we provide the dataset solution for the sampled customer service issue to GPT-4o and indicate that it is one possible resolution but that GPT-4o must itself assign a probability to it based on how plausible it is perceived to be. We report the results of this latter agent as “well-specified” in Figure 5, where all agents were run for a total of 20 trials.

In the face of prior misspecification — something that the base PSRL algorithm does not entertain by assumption and, therefore, has no explicit mechanism to cope with — baseline LLM agent designs still cannot achieve a statistically-significant improvement over PSRL. Furthermore, once the prior misspecification is removed (without handing the solution away as the agent must still sift through other plausible sources of customer issues), PSRL is able to demonstrate strong exploration that far exceeds baseline methods on a real-world task with a tremendously-large action space.

### 4.2 TABULAR MDPs

### Citations in this section

[16]: (Osband et al., 2013)
[33]: (Strehl & Littman, 2008)

For a tabular MDP widely known as a hard exploration task, we turn our focus to a truncated variant of the **RiverSwim** environment [33]. RiverSwim is a six-state chain where the agent begins in the leftmost state. The stochastic transition function mimics a water current that allows an agent to deterministically swim to the left (downstream with the current) but only stochastically swim to the right (upstream against the current) with a 35% chance of success and a small 5% chance of being pushed back one state downstream [16]. Swimming downstream in the initial state results in a small reward of 0.005. Successfully swimming all the way upstream allows the agent to reach the rightmost state where it can collect a reward of 1. As all other rewards are zero, a RiverSwim agent must explore the full length of the river to learn optimal behavior. To keep financial costs down, we truncate the environment to a river of length 3 (one initial state, intermediate state, and terminal state) with  $H = 6$ .

We compare our LLM-based implementation of PSRL with a vanilla PSRL agent for a tabular MDP [16]. The latter models epistemic uncertainty over the transition function as a collection of  $|S||A|$  Dirichlet distributions. This epistemic state representation allows for the

<sup>3</sup>[https://github.com/tajwarfahim/paprika/blob/main/llm\\_exploration/game/game\\_configs/customer\\_service.json](https://github.com/tajwarfahim/paprika/blob/main/llm_exploration/game/game_configs/customer_service.json)

computational conveniences of Dirichlet-multinomial conjugacy. We further model unknown rewards with a discrete uniform prior over  $\{0, 0.005, 1\}$ . Cumulative regret curves shown in Figure 6 compare our LLM-based PSRL with a Dirichlet  $(0.1, 0.1, 0.1, 0.1)$  prior against vanilla PSRL (with the standard uniform Dirichlet prior initialization of  $\alpha_0 = \frac{1}{|\mathcal{S}|}$ ). We use  $\kappa_{\pi^*} = \kappa_{\text{posterior}} = \kappa_{\text{sampling}} = 1$  and all agents are run for 40 independent trials, except the vanilla PSRL agent run for 1,000. We also compare against the LLM agent baselines of Reflexion and ICRL with  $p = 1$ .

Our initial results with RiverSwim were negative (see Appendix C) as GPT-4o struggled to cope with maintaining and updating the verbose epistemic state representation describing reward information and next-state transitions across all 12 state-action pairs. Curiously, however, this negative result provided an opportunity to assess a claim of Section 3.2 that more-capable LLMs would allow our PSRL implementation to scale gracefully to more complex tasks. Indeed, by upgrading from GPT-4o to o1-mini, Figure 6 shows that our LLM-based PSRL is capable of achieving sub-linear regret on par with vanilla PSRL. Reflexion is unable to persevere past failed attempts to swim upstream before settling for the smaller downstream reward of 0.005. ICRL has just over 25% of trials where it stumbles into the optimal policy and sticks with it while, for 60% of trials, it too falls back to pursuing the downstream reward. Moreover, the same LLM upgrade has little impact on the performance of Reflexion and actually manages to worsen the performance of ICRL; for the latter, we suspect the performance degradation stems from a combination of the stochastic transition dynamics coupled with the large quantity of ICL demonstrations that perhaps mesh poorly with the reasoning steps of o1-mini. Nevertheless, we find that LLM planning issues re-emerge in our LLM-based PSRL upon scaling up to a larger instance of RiverSwim (see Appendix D).

![Figure 6: Cumulative regret curves for the RiverSwim environment with 3 states. The plot shows Cumulative Regret on the y-axis (0 to 25) versus Episode on the x-axis (0 to 35). The legend includes: PSRL (blue), PSRL + LLMs (GPT-4o) (orange), PSRL + LLMs (o1-mini) (green), Reflexion (GPT-4o) (red), Reflexion (o1-mini) (purple), ICRL, p=1.0 (GPT-4o) (brown), and ICRL, p=1.0 (o1-mini) (dark green). The PSRL + LLMs (o1-mini) curve shows the lowest cumulative regret, followed by PSRL + LLMs (GPT-4o). Reflexion and ICRL curves show significantly higher regret.](d864789b0d8384da1d22fd6a5d76bbdf_img.jpg)

Figure 6: Cumulative regret curves for the RiverSwim environment with 3 states. The plot shows Cumulative Regret on the y-axis (0 to 25) versus Episode on the x-axis (0 to 35). The legend includes: PSRL (blue), PSRL + LLMs (GPT-4o) (orange), PSRL + LLMs (o1-mini) (green), Reflexion (GPT-4o) (red), Reflexion (o1-mini) (purple), ICRL, p=1.0 (GPT-4o) (brown), and ICRL, p=1.0 (o1-mini) (dark green). The PSRL + LLMs (o1-mini) curve shows the lowest cumulative regret, followed by PSRL + LLMs (GPT-4o). Reflexion and ICRL curves show significantly higher regret.

Figure 6: Cumulative regret curves for the RiverSwim environment with 3 states. Labels show the choice of constituent LLM model (GPT-4o or o1-mini) in each LLM agent.

### 4.3 NATURAL LANGUAGE MDPs

### Citations in this section

[11]: (Lu et al., 2023)
[27]: (Brooks et al., 2023)
[34]: (Lokshтанov & Subercaseaux, 2022)
[35]: (Guo et al., 2025)

![Figure 7: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret on the y-axis (0 to 8) versus Episode on the x-axis (0 to 8). The legend includes: PSRL + LLMs (ours) (blue), Reflexion (orange), ICRL (red), ICRL (p=1.0) (green), ICRL (p=0.5) (purple), ICRL (p=0.1) (brown), and Bayes-Optimal (grey). The PSRL + LLMs (ours) curve shows the lowest cumulative regret, followed by Reflexion and ICRL.](27b22513fc27a0ff5f230b062ad3112f_img.jpg)

Figure 7: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret on the y-axis (0 to 8) versus Episode on the x-axis (0 to 8). The legend includes: PSRL + LLMs (ours) (blue), Reflexion (orange), ICRL (red), ICRL (p=1.0) (green), ICRL (p=0.5) (purple), ICRL (p=0.1) (brown), and Bayes-Optimal (grey). The PSRL + LLMs (ours) curve shows the lowest cumulative regret, followed by Reflexion and ICRL.

Figure 7: Cumulative regret curves for the combination lock environment. The vertical axis shows turns to identify the unlock code.

![Figure 8: Cumulative regret curves for the Wordle environment. The plot shows Cumulative Regret on the y-axis (0 to 6) versus Episode on the x-axis (0 to 6). The legend includes: PSRL + LLMs (ours) (blue), PSRL + LLMs (ours, GPT-4o) (orange), PSRL + LLMs (ours, R1) (green), Reflexion (R1) (red), Reflexion (GPT-4o) (purple), ICRL (p=1.0, GPT-4o) (brown), ICRL (p=1.0, R1) (dark green), ICRL (p=0.5, GPT-4o) (dark blue), ICRL (p=0.5, R1) (dark red), and ICRL (p=0.1, GPT-4o) (dark orange). The PSRL + LLMs (ours) curve shows the lowest cumulative regret, followed by PSRL + LLMs (ours, GPT-4o) and PSRL + LLMs (ours, R1).](643d86ebba41e16a88461bfcb3741de6_img.jpg)

Figure 8: Cumulative regret curves for the Wordle environment. The plot shows Cumulative Regret on the y-axis (0 to 6) versus Episode on the x-axis (0 to 6). The legend includes: PSRL + LLMs (ours) (blue), PSRL + LLMs (ours, GPT-4o) (orange), PSRL + LLMs (ours, R1) (green), Reflexion (R1) (red), Reflexion (GPT-4o) (purple), ICRL (p=1.0, GPT-4o) (brown), ICRL (p=1.0, R1) (dark green), ICRL (p=0.5, GPT-4o) (dark blue), ICRL (p=0.5, R1) (dark red), and ICRL (p=0.1, GPT-4o) (dark orange). The PSRL + LLMs (ours) curve shows the lowest cumulative regret, followed by PSRL + LLMs (ours, GPT-4o) and PSRL + LLMs (ours, R1).

Figure 8: Cumulative regret curves for the Wordle environment. Labels show the choice of constituent LLM model (GPT-4o or DeepSeek-R1) in each LLM agent.

Having verified that our LLM-based PSRL retains efficient exploration in more traditional environments, we now turn to tasks entirely inaccessible by classic PSRL. The first of these tasks is a **combination lock** environment where an agent must enter  $H = 3$  distinct digits in order to open a lock and receive a reward of +1. All other rewards are zero and the agent is provided with (verbal) state information indicating whether the most recently guessed digit is either in the correct position

for the correct code, present in the correct code but in some other position, or simply not present in the correct code at all. An agent has  $K = 8$  episodes to identify the correct combination and, with each one of 20 independent trials having an unlock code sampled uniformly at random from all 720 possible codes, exploration via uniform random code selection has below 0.14% chance of success.

The second task is the challenging web game known as **Wordle** [34], where an agent has exactly  $K = 6$  episodes to enter  $H = 5$  distinct letters (which need not be a dictionary word) that spell a correct target word and receive a reward of  $+1$ . Across 40 trials (except ICPI run for 10 trials due to its significantly higher financial cost and lengthy run times), the target word is chosen uniformly at random from a filtered corpus of English dictionary words. The agent is provided verbal feedback in each state indicating whether the most recently guessed letter is in the correct position for the target word, in the target word but at some other position, or not present in the target word at all.

Our LLM-based PSRL agent ( $\kappa_{\text{sampling}} = \kappa_{\pi^*} = \kappa_{\text{posterior}} = 1$ ) is given an uninformative prior which describes all non-repeating codes/English words with the appropriate length as being equiprobable; the unlock code/target word is an environment proxy [11] such that knowledge of the proxy is a sufficient statistic for recovering the full MDP. For the combination lock, we also compute the Bayes-optimal policy with respect to the same uninformative prior and plot its cumulative regret for comparison. To assess the efficacy of our LLM-based PSRL with another alternative choice of constituent LLM, we present Wordle results with DeepSeek-R1 [35].

The combination lock and Wordle environments represent distinct instances of an exploration problem at differing scales within a deterministic environment. Notably, the immediate per-digit/letter feedback eliminates the challenge of credit assignment entirely (as there is no ambiguity in how each decision impacts delayed rewards) and isolates exploration as the sole data efficiency obstacle. Our results (Figures 7 and 8) show that the LLM-based PSRL is able to most effectively explore the space of possible unlock codes/target words relative to the baseline methods. Crucially, none of the three constituent LLMs used by PSRL are prompted to explicitly encourage exploration. Rather, these results further illustrate how prompting these LLMs to perform atomic functions of PSRL and allowing the algorithm to prescribe how those outputs should be orchestrated in the agent design can yield an effective exploration strategy. In Wordle, we observe that DeepSeek-R1 provides a performance improvement to all LLM agents; however, we find that its enhanced reasoning capabilities applied to even our best baseline LLM agent are insufficient to yield a statistically-significant improvement over our LLM-based PSRL, even when run with a less-capable GPT-4o as the constituent LLM. We invite readers to see Appendix E for analogous results on combination lock with DeepSeek-R1.

The ICPI paper [27] includes a dataset balancing scheme for ICL, presuming the requisite data has already been collected. While reasonable for some environments, exploration is fundamentally about governing data collection to synthesize optimal behavior and, in these domains, ICPI never observes non-zero reward and collapses to a random policy. For ICRL, using all available data with  $p = 1$  is equivalent to the "LLM policy" evaluated by Klisarov et al. (2025), who also find poor performance in Wordle. While results in the combination lock domain are better, we find that decreasing the keep probability  $p$  is detrimental to the "exploratory" ICRL of Monea et al. (2024). In Reflexion, we observe that self-reflections during the early stages of learning generally encourage exploration of untested digits/letters, assuming the agent knows how to explore upon simply being instructed to do so. Only once uncertainty has largely been resolved do explorations become specific suggestions about how to explore with particular digits/letters and their ordering.

## 5 CONCLUSION

### Citations in this section

[36]: (Jiang et al., 2015; Arumugam et al., 2018; Rathnam et al., 2023)
[37]: (Russo & Van Roy, 2018)

While much of the burgeoning literature surrounding LLM agents has felt compelled to design new algorithms for solving RL problems, we here have demonstrated that an existing algorithm, PSRL, can be implemented with LLMs. The main advantage of our proposed LLM-based implementation of PSRL is allowing agent designers to leverage the strong generalization and reasoning capabilities of LLMs in natural-language environments while simultaneously capitalizing on the well-studied exploration properties of TS. Future work might extend regularization methods [36] that embrace inaccurate transition models to rectify deficiencies we observed with LLM planning in stochastic domains. Our preliminary results (see Appendix F) on recovering information-directed exploration [37] with LLMs

represent what is likely to be another very fruitful direction for future work and further reinforces the potential benefits of implementing, rather than replacing, existing RL algorithms with LLMs.

## ETHICS STATEMENT

### Citations in this section

[38]: (Thompson, 1933)
[39]: (Chapelle & Li, 2011)
[40]: (Russo et al., 2018)

The impact of LLMs in recent years has been undeniable and so immense as to extend beyond the confines of the machine learning community, drawing scrutiny from the broader public. As this paper studies mechanisms for improving the decision-making capabilities of LLMs that are becoming increasingly more capable and ubiquitously deployed, there is potential for broad impact stemming from our work. This impact is amplified by the fact that our contributions for improved exploration in LLMs center around Thompson sampling [38], an exploration strategy whose impact in real-world decision-making problems such as recommendation systems [39] and beyond [40] is already well known.

## REPRODUCIBILITY STATEMENT

For all LLM agents evaluated in our experiments, the key items needed to reproduce our results are the system prompts, user prompts, environment descriptions, environment details, and the process by which constituent LLMs are queried and have their outputs organized. All of these details can be found across Section 4 and Appendix H along with a rough (anecdotal) estimates of the associated financial cost of running these experiments in Appendix I. Details of all evaluation domains can be found in Section 4 and the associated natural language descriptions common to all LLM agents evaluated in this work can be found in the appropriate sub-sections of Appendix H. As this paper relies heavily on API access to LLMs, it is impossible to obtain granular details on how much compute was used by our experiments. Instead, we have included Appendix I with ballpark estimates of how many tokens were used by our proposed approach in each of our evaluation domains as well as a translation of those token counts to dollar costs.

## ACKNOWLEDGMENTS

This work was supported by ONR MURI N00014-24-1-2748, ONR grant N00014-23-1-2510, and Azure credits from a Microsoft AFMR grant. We gratefully acknowledge Ilia Sucholutsky for setup and debugging assistance in our experiments. We thank Ted Summers for a helpful suggestion to use XML formatting when processing trajectories with LLMs. Finally, we thank Ilia Sucholutsky and David Abel for feedback and insightful comments on an early draft of the paper.

## REFERENCES

- Yasin Abbasi-Yadkori and Csaba Szepesvari. Bayesian Optimal Control of Smoothly Parameterized Systems: The Lazy Posterior Sampling Algorithm. *arXiv preprint arXiv:1406.3926*, 2014.
- Josh Achiam, Steven Adler, Sandhini Agarwal, Lama Ahmad, Ilge Akkaya, Florencia Leon Aleman, Diogo Almeida, Janko Altenhchmidt, Sam Altman, Shyamal Anadkat, et al. GPT-4 Technical Report. *arXiv preprint arXiv:2303.08774*, 2023.
- Shipra Agrawal and Randy Jia. Optimistic Posterior Sampling for Reinforcement Learning: Worst-Case Regret Bounds. In *Advances in Neural Information Processing Systems*, pp. 1184–1194, 2017.
- Dilip Arumugam and Satinder Singh. Planning to the Information Horizon of BAMDPs via Epistemic State Abstraction. In *Advances in Neural Information Processing Systems*, volume 35, 2022.
- Dilip Arumugam and Benjamin Van Roy. Deciding What to Model: Value-Equivalent Sampling for Reinforcement Learning. *Advances in Neural Information Processing Systems*, 35:9024–9044, 2022.
- Dilip Arumugam, David Abel, Kavosh Asadi, Nakul Gopalan, Christopher Grimm, Jun Ki Lee, Lucas Lehnert, and Michael L Littman. Mitigating Planner Overfitting in Model-Based Reinforcement Learning. *arXiv preprint arXiv:1812.01129*, 2018.

- P Auer, Paul Fischer, and N Cesa-Bianchi. Finite-Time Analysis of the Multiarmed Bandit Problem. *Machine Learning*, 47(3):235–256, 2002.
- Peter Auer, Thomas Jaksch, and Ronald Ortner. Near-Optimal Regret Bounds for Reinforcement Learning. In *Advances in Neural Information Processing Systems*, pp. 89–96, 2009.
- Mohammad Gheshlaghi Azar, Ian Osband, and Rémi Munos. Minimax Regret Bounds for Reinforcement Learning. In *International Conference on Machine Learning*, pp. 263–272, 2017.
- Richard Bellman. A Markovian Decision Process. *Journal of Mathematics and Mechanics*, pp. 679–684, 1957.
- Richard Bellman and Robert Kalaba. On Adaptive Control Processes. *IRE Transactions on Automatic Control*, 4(2):1–9, 1959.
- Marcel Binz and Eric Schulz. Using Cognitive Psychology to Understand GPT-3. *Proceedings of the National Academy of Sciences*, 120(6):e2218523120, 2023.
- Rishi Bommasani, Drew A Hudson, Ehsan Adeli, Russ Altman, Simran Arora, Sydney von Arx, Michael S Bernstein, Jeannette Bohg, Antoine Bosselut, Emma Brunskill, et al. On the Opportunities and Risks of Foundation Models. *arXiv preprint arXiv:2108.07258*, 2021.
- Ronen I Brafman and Moshe Tennenholtz. R-MAX - A General Polynomial Time Algorithm for Near-Optimal Reinforcement Learning. *Journal of Machine Learning Research*, 3(Oct):213–231, 2002.
- Ethan Brooks, Logan Walls, Richard L Lewis, and Satinder Singh. Large Language Models Can Implement Policy Iteration. *Advances in Neural Information Processing Systems*, 36:30349–30366, 2023.
- Tom Brown, Benjamin Mann, Nick Ryder, Melanie Subbiah, Jared D Kaplan, Prafulla Dhariwal, Arvind Neelakantan, Pranav Shyam, Girish Sastry, Amanda Askell, et al. Language Models are Few-Shot Learners. *Advances in Neural Information Processing Systems*, 33:1877–1901, 2020.
- Sébastien Bubeck and Nicolo Cesa-Bianchi. Regret Analysis of Stochastic and Nonstochastic Multi-Armed Bandit Problems. *Foundations and Trends in Machine Learning*, 5(1):1–122, 2012.
- Olivier Chapelle and Lihong Li. An Empirical Evaluation of Thompson Sampling. In *Advances in Neural Information Processing Systems*, pp. 2249–2257, 2011.
- Julian Coda-Forno, Marcel Binz, Zeynep Akata, Matt Botvinick, Jane Wang, and Eric Schulz. Meta-In-Context Learning in Large Language Models. *Advances in Neural Information Processing Systems*, 36:65189–65201, 2023.
- Julian Coda-Forno, Marcel Binz, Jane X Wang, and Eric Schulz. CogBench: A Large Language Model Walks into a Psychology Lab. In *Forty-first International Conference on Machine Learning*, 2024.
- Thomas M Cover and Joy A Thomas. *Elements of Information Theory*. John Wiley & Sons, 2012.
- Zhenwen Dai, Federico Tomasi, and Sina Ghiaussian. In-Context Exploration-Exploitation for Reinforcement Learning. In *The Twelfth International Conference on Learning Representations*, 2024.
- Christoph Dann and Emma Brunskill. Sample Complexity of Episodic Fixed-Horizon Reinforcement Learning. In *Proceedings of the 28th International Conference on Neural Information Processing Systems-Volume 2*, pp. 2818–2826, 2015.
- Christoph Dann, Tor Lattimore, and Emma Brunskill. Unifying PAC and Regret: Uniform PAC Bounds for Episodic Reinforcement Learning. In *Proceedings of the 31st International Conference on Neural Information Processing Systems*, pp. 5717–5727, 2017.
- Armen Der Kiureghian and Ove Ditlevsen. Aleatory or Epistemic? Does it Matter? *Structural Safety*, 31(2):105–112, 2009.

- Shi Dong, Benjamin Van Roy, and Zhengyuan Zhou. Simple Agent, Complex Environment: Efficient Reinforcement Learning with Agent States. *Journal of Machine Learning Research*, 23(255):1–54, 2022.
- Miroslav Dudík, Katja Hofmann, Robert E Schapire, Aleksandrs Slivkins, and Masrour Zoghi. Contextual Dueling Bandits. In *Conference on Learning Theory*, pp. 563–587, 2015.
- Michael O’Gordon Duff. *Optimal Learning: Computational Procedures for Bayes-Adaptive Markov Decision Processes*. PhD thesis, University of Massachusetts Amherst, 2002.
- Vikranth Dwarcherla, Xiuyuan Lu, Morteza Ibrahim, Ian Osband, Zheng Wen, and Benjamin Van Roy. Hypermodels for Exploration. In *International Conference on Learning Representations*, 2020.
- Vikranth Dwarcherla, Seyed Mohammad Asghari, Botao Hao, and Benjamin Van Roy. Efficient Exploration for LLMs. In *Forty-first International Conference on Machine Learning*, 2024.
- Jan-Philipp Fränken, Sam Kwok, Peixuan Ye, Kanishk Gandhi, Dilip Arumugam, Jared Moore, Alex Tamkin, Tobias Gerstenberg, and Noah D Goodman. Social Contract AI: Aligning AI Assistants with Implicit Group Norms. *arXiv preprint arXiv:2310.17769*, 2023.
- Yoav Freund and Robert E Schapire. A Decision-Theoretic Generalization of On-Line Learning and an Application to Boosting. *Journal of Computer and System Sciences*, 55(1):119–139, 1997.
- Yarin Gal and Zoubin Ghahramani. Dropout as a Bayesian Epproximation: Representing Model Uncertainty in Deep Learning. In *International Conference on Machine Learning*, pp. 1050–1059. PMLR, 2016.
- Mohammad Ghavamzadeh, Shie Mannor, Joelle Pineau, and Aviv Tamar. Bayesian Reinforcement Learning: A Survey. *Foundations and Trends in Machine Learning*, 8(5-6):359–483, 2015.
- John Gittins. Bandit Processes and Dynamic Allocation Indices. *Journal of the Royal Statistical Society Series B: Statistical Methodology*, 41(2):148–164, 1979.
- Noah Goodman. Meta-Prompt: A Simple Self-Improving Language Agent. <https://noahgoodman.substack.com/p/meta-prompt-a-simple-self-improving>, 2023.
- Daya Guo, Dejian Yang, Haowei Zhang, Junxiao Song, Ruoyu Zhang, Runxin Xu, Qihao Zhu, Shirong Ma, Peiyi Wang, Xiao Bi, et al. DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning. *arXiv preprint arXiv:2501.12948*, 2025.
- Ronald A Howard. *Dynamic Programming and Markov Processes*. MIT Press, 1960.
- Aaron Hurst, Adam Lerer, Adam P Goucher, Adam Perelman, Aditya Ramesh, Aidan Clark, AJ Ostrow, Akila Welithinda, Alan Hayes, Alec Radford, et al. GPT-4o System Card. *arXiv preprint arXiv:2410.21276*, 2024.
- Haque Ishfaq, Qingfeng Lan, Pan Xu, A Rupam Mahmood, Doina Precup, Anima Anandkumar, and Kamyar Azizzadenesheli. Provable and Practical: Efficient Exploration in Reinforcement Learning via Langevin Monte Carlo. In *The Twelfth International Conference on Learning Representations*, 2024.
- Aaron Jaech, Adam Kalai, Adam Lerer, Adam Richardson, Ahmed El-Kishky, Aiden Low, Alec Helyar, Aleksander Madry, Alex Beutel, Alex Carney, et al. OpenAI o1 System Card. *arXiv preprint arXiv:2412.16720*, 2024.
- Thomas Jaksch, Ronald Ortner, and Peter Auer. Near-Optimal Regret Bounds for Reinforcement Learning. *Journal of Machine Learning Research*, 11(4), 2010.
- Nan Jiang, Alex Kulesza, Satinder Singh, and Richard Lewis. The Dependence of Effective Planning Horizon on Model Accuracy. In *Proceedings of the 2015 International Conference on Autonomous Agents and Multiagent Systems*, pp. 1181–1189, 2015.

- Chi Jin, Zeyuan Allen-Zhu, Sebastien Bubeck, and Michael I Jordan. Is Q-Learning Provably Efficient? *Advances in Neural Information Processing Systems*, 31, 2018.
- Emilio Jorge, Christos Dimitrakakis, and Debabrata Basu. Isoperimetry is All We Need: Langevin Posterior Sampling for RL with Sublinear Regret. *arXiv preprint arXiv:2412.20824*, 2024.
- Łukasz Kaiser, Mohammad Babaeizadeh, Piotr Miłos, Błażej Osiński, Roy H Campbell, Konrad Czechowski, Dumitru Erhan, Chelsea Finn, Piotr Kozakowski, Sergey Levine, et al. Model Based Reinforcement Learning for Atari. In *International Conference on Learning Representations*, 2020.
- Sham Machandranath Kakade. *On the Sample Complexity of Reinforcement Learning*. PhD thesis, University of London, University College London (United Kingdom), 2003.
- Amin Karbasi, Nikki Lijing Kuang, Yian Ma, and Siddharth Mitra. Langevin Thompson Sampling with Logarithmic Communication: Bandits and Reinforcement Learning. In *International Conference on Machine Learning*, pp. 15828–15860, 2023.
- Nan Rosemary Ke, Danny P Sawyer, Hubert Soyer, Martin Engelcke, David P Reichert, Drew A Hudson, John Reid, Alexander Lerchner, Danilo Jimenez Rezende, Timothy P Lillicrap, Michael Mozer, and Jane X Wang. Can Foundation Models actively Gather Information in Interactive Environments to Test Hypotheses? *arXiv preprint arXiv:2412.06438*, 2024.
- Michael Kearns and Satinder Singh. Near-Optimal Reinforcement Learning in Polynomial Time. *Machine Learning*, 49:209–232, 2002.
- Martin Klissarov, R Devon Hjelm, Alexander T Toshev, and Bogdan Mazoure. On the Modeling Capabilities of Large Language Models for Sequential Decision Making. In *The Thirteenth International Conference on Learning Representations*, 2025.
- Takeshi Kojima, Shixiang Shane Gu, Machel Reid, Yutaka Matsuo, and Yusuke Iwasawa. Large Language Models are Zero-Shot Reasoners. *Advances in Neural Information Processing Systems*, 35:22199–22213, 2022.
- Akshay Krishnamurthy, Keegan Harris, Dylan J Foster, Cyril Zhang, and Aleksandrs Slivkins. Can Large Language Models Explore In-Context? In *The Thirty-eighth Annual Conference on Neural Information Processing Systems*, 2024.
- Minae Kwon, Sang Michael Xie, Kalesha Bullard, and Dorsa Sadigh. Reward design with language models. In *The Eleventh International Conference on Learning Representations*, 2023.
- Tze Leung Lai and Herbert Robbins. Asymptotically Efficient Adaptive Allocation Rules. *Advances in Applied Mathematics*, 6(1):4–22, 1985.
- Michael Laskin, Luyu Wang, Junhyuk Oh, Emilio Parisotto, Stephen Spencer, Richie Steigerwald, DJ Strouse, Steven Hansen, Angelos Filos, Ethan Brooks, et al. In-Context Reinforcement Learning with Algorithm Distillation. *arXiv preprint arXiv:2210.14215*, 2022.
- Tor Lattimore and Csaba Szepesvári. *Bandit Algorithms*. Cambridge University Press, 2020.
- Harrison Lee, Samrat Phatale, Hassan Mansoor, Thomas Mesnard, Johan Ferret, Kellie Ren Lu, Colton Bishop, Ethan Hall, Victor Carbune, Abhinav Rastogi, et al. RLHF vs. RLHF: Scaling Reinforcement Learning from Human Feedback with AI Feedback. In *International Conference on Machine Learning*, pp. 26874–26901. PMLR, 2024a.
- Jonathan Lee, Annie Xie, Aldo Pacchiano, Yash Chandak, Chelsea Finn, Ofir Nachum, and Emma Brunskill. Supervised Pretraining can Learn In-Context Reinforcement Learning. *Advances in Neural Information Processing Systems*, 36, 2024b.
- Sergey Levine. Reinforcement Learning and Control as Probabilistic Inference: Tutorial and Review. *arXiv preprint arXiv:1805.00909*, 2018.
- Long-Ji Lin. Self-Improving Reactive Agents Based on Reinforcement learning, Planning and Teaching. *Machine Learning*, 8:293–321, 1992.

- Yueyang Liu, Adithya M Devraj, Benjamin Van Roy, and Kuang Xu. Gaussian Imagination in Bandit Learning. *arXiv preprint arXiv:2201.01902*, 2022.
- Zhihan Liu, Hao Hu, Shena Zhang, Hongyi Guo, Shuqi Ke, Boyi Liu, and Zhaoran Wang. Reason for Future, Act for Now: A Principled Framework for Autonomous LLM Agents with Provable Sample Efficiency. *arXiv preprint arXiv:2309.17382*, 2023.
- Daniel Lokshstanov and Bernardo Subercaseaux. Wordle is NP-Hard. In *11th International Conference on Fun with Algorithms*, 2022.
- Xiyuan Lu and Benjamin Van Roy. Ensemble Sampling. *Advances in Neural Information Processing Systems*, 30, 2017.
- Xiyuan Lu and Benjamin Van Roy. Information-Theoretic Confidence Bounds for Reinforcement Learning. *Advances in Neural Information Processing Systems*, 32, 2019.
- Xiyuan Lu, Benjamin Van Roy, Vikranth Dwaracherla, Morteza Ibrahimi, Ian Osband, and Zheng Wen. Reinforcement Learning, Bit by Bit. *Foundations and Trends in Machine Learning*, 16(6): 733–865, 2023.
- Eric Mazumdar, Aldo Pacchiano, Yian Ma, Michael Jordan, and Peter Bartlett. On Approximate Thompson Sampling with Langevin Algorithms. In *International Conference on Machine Learning*, pp. 6797–6807, 2020.
- David McAllester and Karl Stratos. Formal Limitations on the Measurement of Mutual Information. In *International Conference on Artificial Intelligence and Statistics*, pp. 875–884, 2020.
- R Thomas McCoy, Shunyu Yao, Dan Friedman, Matthew D Hardy, and Thomas L Griffiths. Embers of Autoregression Show how Large Language Models are Shaped by the Problem They are Trained to Solve. *Proceedings of the National Academy of Sciences*, 121(41):e2322420121, 2024.
- Giovanni Monea, Antoine Bosselut, Kianté Brantley, and Yoav Artzi. LLMs Are In-Context Reinforcement Learners. *arXiv preprint arXiv:2410.05362*, 2024.
- Allan Nie, Yi Su, Bo Chang, Jonathan N Lee, Ed H Chi, Quoc V Le, and Minmin Chen. EVOLvE: Evaluating and Optimizing LLMs For Exploration. *arXiv preprint arXiv:2410.06238*, 2024.
- Brendan O’Donoghue. Variational Bayesian Reinforcement Learning with Regret Bounds. *Advances in Neural Information Processing Systems*, 34:28208–28221, 2021.
- Brendan O’Donoghue, Ian Osband, Remi Munos, and Volodymyr Mnih. The Uncertainty Bellman Equation and Exploration. In *International Conference on Machine Learning*, pp. 3836–3845, 2018.
- Brendan O’Donoghue, Ian Osband, and Catalin Ionescu. Making Sense of Reinforcement Learning and Probabilistic Inference. In *International Conference on Learning Representations*, 2020.
- Ian Osband. *Deep Exploration via Randomized Value Functions*. PhD thesis, Stanford University, 2016a.
- Ian Osband. Risk Versus Uncertainty in Deep Learning: Bayes, Bootstrap and the dangers of Dropout. In *NIPS Workshop on Bayesian Deep Learning*, 2016b.
- Ian Osband and Benjamin Van Roy. Model-Based Reinforcement Learning and the Eluder Dimension. *Advances in Neural Information Processing Systems*, 27, 2014.
- Ian Osband and Benjamin Van Roy. Posterior Sampling for Reinforcement Learning Without Episodes. *arXiv preprint arXiv:1608.02731*, 2016.
- Ian Osband and Benjamin Van Roy. Why is Posterior Sampling Better than Optimism for Reinforcement Learning? In *International Conference on Machine Learning*, pp. 2701–2710, 2017.
- Ian Osband, Daniel Russo, and Benjamin Van Roy. (More) Efficient Reinforcement Learning via Posterior Sampling. *Advances in Neural Information Processing Systems*, 26:3003–3011, 2013.

- Ian Osband, Charles Blundell, Alexander Pritzel, and Benjamin Van Roy. Deep Exploration via Bootstrapped DQN. *Advances in Neural Information Processing Systems*, 29, 2016a.
- Ian Osband, Benjamin Van Roy, and Zheng Wen. Generalization and Exploration via Randomized Value Functions. In *International Conference on Machine Learning*, pp. 2377–2386, 2016b.
- Ian Osband, John Aslanides, and Albin Cassirer. Randomized Prior Functions for Deep Reinforcement Learning. *Advances in Neural Information Processing Systems*, 31, 2018.
- Ian Osband, Benjamin Van Roy, Daniel J Russo, and Zheng Wen. Deep Exploration via Randomized Value Functions. *Journal of Machine Learning Research*, 20(124):1–62, 2019.
- Ian Osband, Zheng Wen, Seyed Mohammad Asghari, Vikranth Dwarcherla, Morteza Ibrahimi, Xiyuan Lu, and Benjamin Van Roy. Approximate Thompson Sampling via Epistemic Neural Networks. In *Uncertainty in Artificial Intelligence*, pp. 1586–1595, 2023.
- Long Ouyang, Jeff Wu, Xu Jiang, Diogo Almeida, Carroll L Wainwright, Pamela Mishkin, Chong Zhang, Sandhini Agarwal, Katarina Slama, Alex Ray, et al. Training Language Models to Follow Instructions with Human Feedback. *arXiv preprint arXiv:2203.02155*, 2022.
- Yi Ouyang, Mukul Gagani, Ashutosh Nayyar, and Rahul Jain. Learning Unknown Markov Decision Processes: A Thompson Sampling Approach. *Advances in Neural Information Processing Systems*, 30, 2017.
- Martin L. Puterman. *Markov Decision Processes—Discrete Stochastic Dynamic Programming*. John Wiley & Sons, New York, 1994.
- Sarah Rathnam, Sonali Parbhoo, Weiwei Pan, Susan Murphy, and Finale Doshi-Velez. The Unintended Consequences of Discount Regularization: Improving Regularization in Certainty Equivalence Reinforcement Learning. In *International Conference on Machine Learning*, pp. 28746–28767. PMLR, 2023.
- Daniel Russo and Benjamin Van Roy. Learning to Optimize via Posterior Sampling. *Mathematics of Operations Research*, 39(4):1221–1243, 2014.
- Daniel Russo and Benjamin Van Roy. An Information-Theoretic Analysis of Thompson Sampling. *The Journal of Machine Learning Research*, 17(1):2442–2471, 2016.
- Daniel Russo and Benjamin Van Roy. Learning to Optimize via Information-Directed Sampling. *Operations Research*, 66(1):230–252, 2018.
- Daniel J Russo, Benjamin Van Roy, Abbas Kazerouni, Ian Osband, and Zheng Wen. A Tutorial on Thompson Sampling. *Foundations and Trends in Machine Learning*, 11(1):1–96, 2018.
- Remo Sasso, Michelangelo Conserva, and Paulo Rauber. Posterior Sampling for Deep Reinforcement Learning. In *International Conference on Machine Learning*, pp. 30042–30061, 2023.
- Noah Shinn, Federico Cassano, Ashwin Gopinath, Karthik Narasimhan, and Shunyu Yao. Reflexion: Language Agents with Verbal Reinforcement Learning. *Advances in Neural Information Processing Systems*, 36, 2024.
- David Silver and Richard S Sutton. Welcome to the Era of Experience. *Google AI*, 2025.
- Max Simchowitz, Christopher Tosh, Akshay Krishnamurthy, Daniel J Hsu, Thodoris Lykouras, Miro Dudik, and Robert E Schapire. Bayesian Decision-Making Under Misspecified Priors with Applications to Meta-Learning. *Advances in Neural Information Processing Systems*, 34: 26382–26394, 2021.
- Nisan Stiennon, Long Ouyang, Jeffrey Wu, Daniel Ziegler, Ryan Lowe, Chelsea Voss, Alec Radford, Dario Amodei, and Paul F Christiano. Learning to Summarize with Human Feedback. *Advances in Neural Information Processing Systems*, 33:3008–3021, 2020.
- Alexander L Strehl and Michael L Littman. An Analysis of Model-Based Interval Estimation for Markov Decision Processes. *Journal of Computer and System Sciences*, 74(8):1309–1331, 2008.

- Alexander L Strehl, Lihong Li, and Michael L Littman. Reinforcement Learning in Finite MDPs: PAC Analysis. *Journal of Machine Learning Research*, 10(11), 2009.
- Malcolm JA Strens. A Bayesian Framework for Reinforcement Learning. In *Proceedings of the Seventeenth International Conference on Machine Learning*, pp. 943–950, 2000.
- Richard S Sutton and Andrew G Barto. *Introduction to Reinforcement Learning*. MIT Press, 1998.
- Fahim Tajwar, Yiding Jiang, Abitha Thankaraj, Sumaita Sadia Rahman, J Zico Kolter, Jeff Schneider, and Russ Salakhutdinov. Training a Generally Curious Agent. In *Forty-Second International Conference on Machine Learning*, 2025.
- Jean Tarbouriech, Tor Lattimore, and Brendan O'Donoghue. Probabilistic Inference in Reinforcement Learning Done Right. *Advances in Neural Information Processing Systems*, 36:33687–33725, 2023.
- Gemini Team, Rohan Anil, Sebastian Borgeaud, Jean-Baptiste Alayrac, Jahui Yu, Radu Soricut, Johan Schalkwyk, Andrew M Dai, Anja Haith, Katie Millican, et al. Gemini: A Family of Highly Capable Multimodal Models. *arXiv preprint arXiv:2312.11805*, 2023.
- William R Thompson. On the Likelihood That One Unknown Probability Exceeds Another in View of the Evidence of Two Samples. *Biometrika*, 25(3/4):285–294, 1933.
- Hugo Touvron, Louis Martin, Kevin Stone, Peter Albert, Amjad Almahairi, Yasmine Babaei, Nikolay Bashlykov, Soumya Batra, Prajjwal Bhargava, Shruti Bhosale, et al. Llama 2: Open foundation and fine-tuned chat models. *arXiv preprint arXiv:2307.09288*, 2023.
- Christopher JCH Watkins and Peter Dayan. Q-Learning. *Machine Learning*, 8:279–292, 1992.
- Jason Wei, Xuezhi Wang, Dale Schuurmans, Maarten Bosma, Fei Xia, Ed Chi, Quoc V Le, Denny Zhou, et al. Chain-of-Thought Prompting Elicits Reasoning in Large Language Models. *Advances in Neural Information Processing Systems*, 35:24824–24837, 2022.
- Sang Michael Xie, Aditi Raghunathan, Percy Liang, and Tengyu Ma. An Explanation of In-context Learning as Implicit Bayesian Inference. In *International Conference on Learning Representations*, 2022.
- Wanqiao Xu, Shi Dong, Dilip Arumugam, and Benjamin Van Roy. Shattering the Agent-Environment Interface for Fine-Tuning Inclusive Language Models. *arXiv preprint arXiv:2305.11455*, 2023.
- Wanqiao Xu, Shi Dong, and Benjamin Van Roy. Posterior Sampling for Continuing Environments. In *Reinforcement Learning Conference*, 2024.
- Xue Yan, Yan Song, Xidong Feng, Mengyue Yang, Haifeng Zhang, Haitham Bou Ammar, and Jun Wang. Efficient Reinforcement Learning with Large Language Model Priors. In *The Thirteenth International Conference on Learning Representations*, 2025.
- Shunyu Yao, Jeffrey Zhao, Dian Yu, Nan Du, Izhak Shafran, Karthik R Narasimhan, and Yuan Cao. ReAct: Synergizing Reasoning and Acting in Language Models. In *The Eleventh International Conference on Learning Representations*, 2023.
- Yisong Yue, Josef Broder, Robert Kleinberg, and Thorsten Joachims. The  $k$ -Armed Dueling Bandits Problem. *Journal of Computer and System Sciences*, 78(5):1538–1556, 2012.
- Andrea Zanette and Emma Brunskill. Tighter Problem-Dependent Regret Bounds in Reinforcement Learning without Domain Knowledge Using Value Function Bounds. In *International Conference on Machine Learning*, pp. 7304–7312. PMLR, 2019.
- Yufeng Zhang, Fengzhuo Zhang, Zhuoran Yang, and Zhaoran Wang. What and How Does In-Context Learning Learn? Bayesian Model Averaging, Parameterization, and Generalization. *arXiv preprint arXiv:2305.19420*, 2023.
- Qinjing Zheng, Mikael Henaff, Amy Zhang, Aditya Grover, and Brandon Amos. Online Intrinsic Rewards for Decision Making Agents from Large Language Model Feedback. *arXiv preprint arXiv:2410.23022*, 2024.

## A RELATED WORK

### Citations in this section

[22]: (Brown et al., 2020)
[28]: (Howard, 1960)
[41]: (Kearns & Singh, 2002; Brafman & Tennenholtz, 2002; Kakade, 2003; Auer et al., 2009; Strehl et al., 2009; Jaksch et al., 2010; Dann & Brunskill, 2015; Azar et al., 2017; Dann et al., 2017; Jin et al., 2018; Zanette & Brunskill, 2019; Dong et al., 2022)
[42]: (Auer et al., 2002)
[43]: (Team et al., 2023)
[44]: (Shim et al., 2024)
[45]: (Laskin et al., 2022; Liu et al., 2023; Lee et al., 2024b; Dai et al., 2024; Yan et al., 2025)
[46]: (Xie et al., 2022; Zhang et al., 2023)
[47]: (Levine, 2018)
[48]: (O’Donoghue et al., 2020; Tarbouriech et al., 2023)
[49]: (Lin, 1992)
[50]: (Watkins & Dayan, 1992)
[51]: (Gal & Ghahramani, 2016)
[52]: (Osband, 2016b)
[53]: (Goodman, 2023)
[54]: (Fränken et al., 2023)
[55]: (Klisaravov et al., 2025; Kwon et al., 2023; Zheng et al., 2024)
[56]: (Ke et al., 2024)
[57]: (Stiennon et al., 2020; Ouyang et al., 2022)
[58]: (Yue et al., 2012; Dudík et al., 2015)
[59]: (Xu et al., 2023; Dwaraacherla et al., 2024)
[60]: (Lee et al., 2024a)

While our primary focus in this paper is on efficient exploration for LLM agents, the broader challenge of efficient exploration for RL agents is a long-studied topic. One route to achieving statistically-efficient exploration relies on the use of “optimism in the face of uncertainty,” where approaches either implicitly or explicitly maintain over-inflated value function estimates for all state-action pairs [41]. These optimistic biases are calibrated by an agent designer to incentivize agent visitation of each state-action pair sufficiently many times and eventually result in accurate value estimates that give rise to optimal behavior. Nie et al. (2024) attempt to realize such an optimistic exploration strategy with LLMs (specifically, combining UCB [42] with Gemini [43]) for multi-armed bandit problems and demonstrate the difficulty in coupling statistical machinery like confidence intervals with LLMs outright. While our proposed implementation relies on an equally (if not more) complex statistical object, the Bayesian posterior, our experiments suggest that LLMs in certain cases may maintain an approximation sufficient for guiding exploration.

Existing designs for LLM agents either do not explicitly engage with the challenge of exploration or do so with complete reliance on in-context learning (ICL) [22]. One of the most popular LLM agent designs is Reflexion [44] where the policy LLM charged with selecting actions is informed at each episode by a “self-reflection” generated from another LLM given the previous episode trajectory. While suitable for some tasks, we observe in our experiments that the self-reflection LLM often “passes the buck” and encourages exploration generically in language without providing a clear strategy for the downstream policy LLM to do so. By relying on LLMs to provide the requisite functions for implementing a prudent choice of existing RL algorithm, we encounter strategic exploration without needing to explicitly instruct any of the involved LLMs to explore.

LLM agents that rely on ICL to enable exploration follow suit with a line of work that examines Transformer-based RL agents in non-natural-language tasks [45]. These methods often rely on casting ICL as either implicit, approximate Bayesian inference [46] or within the “control as inference” framework [47]; one key challenge with the former is that such implicit posterior knowledge cannot be flexibly and explicitly leveraged to guide exploration, whereas the latter suffers from not capturing epistemic uncertainty at all [48]. Very close to the spirit of our work is the in-context policy iteration (ICPI) method of Brooks et al. (2023), who take the classic RL algorithm of policy iteration (PI) [28] and implement it with LLMs and ICL. Unfortunately, the original PI algorithm is oriented towards tabular MDPs that allow for iterating over all state-action pairs simultaneously. While the ICPI algorithm forgoes this in favor of online data collection and resampling via experience replay [49], the authors find it necessary to sample with a dataset balancing scheme to ensure the accuracy of ICL; this presumes that the “right” data is already present or easily acquired from the environment. In larger environments where data must be judiciously acquired, we find that ICPI is never able to collect the data needed for ICL to exhibit any kind of performant behavior. Monea et al. (2024) study a selective “dropout” strategy for the ICL demonstrations used by a policy LLM. However, such a strategy mirrors  $\epsilon$ -greedy exploration [50] without making a concerted effort to strategically guide decision-making, much like how classic dropout in deep RL [51] is a poor proxy for uncertainty-based exploration [52]. In contrast to ICL, the core idea studied in this work is conceptually similar to meta-prompting [53], where an agent incrementally accumulates salient environmental knowledge within its system prompt to refine behavior in each episode; while prior work has suggested that meta-prompting is an implicit approximation of posterior sampling [54], we here are exclusively concerned with the explicit implementation of PSRL.

A related line of approaches examines using classic (deep) RL methods in tandem with LLM reward functions [55]. These approaches, while interesting, largely focus on non-linguistic domains whereas our goal is to bring ideas on data-efficient RL to bear on the natural language domains where LLMs stand to have the most impact.

The posterior-sampling-based exploration strategy we consider in this work connects more broadly to initial investigations surrounding the information gathering capabilities of LLMs [56].

Lastly, we note that the Reinforcement Learning from Human Feedback (RLHF) pipeline [57] used to explicitly optimize LLMs also faces an underlying sequential decision-making problem (in the original formulation, a contextual dueling bandit [58]) and, as such, may greatly benefit from mechanisms to facilitate efficient exploration [59]. Concretely, at any point in the fine-tuning process either by RLHF or Reinforcement Learning from AI Feedback (RLAIF) [60], there will be preference data that offer very little utility or change in LLM responses and those that stand to dramatically improve response quality. By actively exploring for the latter kind of prompts and responses, one stands to arrive at a more proficient LLM with fewer iterations of RLHF or RLAIF. While such work is nascent, our results may offer a promising new pathway for LLMs to achieve the strategic exploration that could reduce these significant data burdens.

## B MULTI-ARMED BANDIT RESULTS

### B.1 BERNOLLI BANDIT

As noted by Krishnamurthy et al. (2024), the financial and temporal costs of running LLM agents can be quite significant. With only 20 trials, it would be presumptuous to make any sweeping claims about superior performance of one method relative to others. Fortunately, the goal of our multi-armed bandit experiment is aimed at a relativistic comparison in the quality of exploration with our LLM-based PSRL relative to classic TS. To this end, we borrow the surrogate statistics employed by Krishnamurthy et al. (2024) to provide deeper insight into the long-term exploratory behavior of LLM-based PSRL. Figure 9 reports the *suffix failure* frequency, where a suffix failure at time period  $t$  is a binary statistic defined as 1 if the optimal action  $A^*$  is never chosen in time periods  $[t, T]$  and 0 otherwise. Clearly, an agent experiencing a large number of suffix failures early on in learning would be unlikely to identify  $A^*$  when run for a larger number of time periods. Figure 10 reports the (scaled) *minimum action frequency*, which reports at time period  $t$  the frequency of the least-chosen action in the first  $t$  time periods:  $\frac{1}{t} \cdot \min_{a \in \mathcal{A}} |\{A_{t'} \mid t' \in [t], A_{t'} = a\}|$ . The statistic is scaled by  $|\mathcal{A}|$  to reside in  $[0, 1]$ . As an agent’s knowledge of the world accumulates, one would naturally expect an agent to gradually cease selection of some (ideally, sub-optimal) actions and incur lower minimum action frequencies. Together, these two surrogate statistics paint a picture of whether or not the exploration of a LLM bandit agent gravitates toward  $A^*$  over time.

Notably, we find that increasing the temperature  $\kappa_{\text{sampling}}$  of the posterior sampling LLM has profound impact on how well our LLM-based PSRL explores according to these metrics. In particular, we find that increasing  $\kappa_{\text{sampling}}$  leads to exploratory behavior more closely aligned with that of classic TS compared to lower temperatures values.

### B.2 CUSTOMER SERVICE BANDIT & PRIOR (MIS)SPECIFICATION

### Citations in this section

[61]: (Russo & Van Roy, 2014; Simchowitz et al., 2021; Liu et al., 2022)

To demonstrate one concrete instance of how our proposed LLM-based PSRL might meet the demands of a real-world decision-making problem, we adapt the customer service task of Tajwar et al. (2025) into a multi-armed bandit problem. In each of  $K = 20$  total time periods, the agent may either ask a question or offer a solution to address a customer issue randomly sampled from the dataset<sup>4</sup> of Tajwar et al. (2025). Similar to Tajwar et al. (2025), we use two additional LLMs to simulate the customer (who answers the agent’s questions and tries suggested solutions as a non-technical person would) and to be a judge/reward function who ultimately determines the binary reward indicating successful resolution of a customer’s issue. All models use GPT-4o as the underlying LLM.

For our LLM-based PSRL, we consider two methods for specifying the prior distribution that PSRL takes as input. In the first case, we simply ask GPT-4o to provide a prior distribution (a list of plausible underlying issues for the customer complaint as well as guessed probabilities based on how likely the model perceives the issue to be) that is given directly as input to our LLM-based

<sup>4</sup>[https://github.com/tajwarfahim/paprika/blob/main/llm\\_exploration/game/game\\_configs/customer\\_service.json](https://github.com/tajwarfahim/paprika/blob/main/llm_exploration/game/game_configs/customer_service.json)

![Figure 9: Suffix failure frequency for a 5-armed Bernoulli bandit with Delta = 0.2. The plot shows the suffix failure frequency over 100 time periods for five methods: TS, PSRL + LLMs (k_sampling = 0.5), PSRL + LLMs (k_sampling = 1), PSRL + LLMs (k_sampling = 1.1), and PSRL + LLMs (k_sampling = 1.2). The y-axis ranges from 0.0 to 0.4. The TS method (blue line) shows a very low failure frequency, while the other methods show higher failure frequencies that increase over time.](3ae74a33759ae31781f484406db4feed_img.jpg)

Figure 9: Suffix failure frequency for a 5-armed Bernoulli bandit with Delta = 0.2. The plot shows the suffix failure frequency over 100 time periods for five methods: TS, PSRL + LLMs (k\_sampling = 0.5), PSRL + LLMs (k\_sampling = 1), PSRL + LLMs (k\_sampling = 1.1), and PSRL + LLMs (k\_sampling = 1.2). The y-axis ranges from 0.0 to 0.4. The TS method (blue line) shows a very low failure frequency, while the other methods show higher failure frequencies that increase over time.

Figure 9: Suffix failure frequency for a 5-armed Bernoulli bandit with  $\Delta = 0.2$ . A suffix failure occurs at time  $t$  if  $A^*$  is never chosen in time periods  $[t, T]$ .

![Figure 10: Scaled minimum action frequency for a 5-armed Bernoulli bandit with Delta = 0.2. The plot shows the scaled minimum action frequency over 100 time periods for five methods: TS, PSRL + LLMs (k_sampling = 0.5), PSRL + LLMs (k_sampling = 1), PSRL + LLMs (k_sampling = 1.1), and PSRL + LLMs (k_sampling = 1.2). The y-axis ranges from 0.0 to 0.4. The TS method (blue line) shows a high frequency that decreases over time, while the other methods show lower frequencies that also decrease.](1c427123350e0e73e2a109b79069314b_img.jpg)

Figure 10: Scaled minimum action frequency for a 5-armed Bernoulli bandit with Delta = 0.2. The plot shows the scaled minimum action frequency over 100 time periods for five methods: TS, PSRL + LLMs (k\_sampling = 0.5), PSRL + LLMs (k\_sampling = 1), PSRL + LLMs (k\_sampling = 1.1), and PSRL + LLMs (k\_sampling = 1.2). The y-axis ranges from 0.0 to 0.4. The TS method (blue line) shows a high frequency that decreases over time, while the other methods show lower frequencies that also decrease.

Figure 10: Scaled minimum action frequency for a 5-armed Bernoulli bandit with  $\Delta = 0.2$ . At time period  $t$ , this is the average frequency of the least-chosen action in time periods  $[1, t]$ .

![Figure 11: A scatter plot of suffix failure frequency vs. minimum action frequency for Thompson sampling and our LLM-based PSRL with varying k_sampling. The x-axis is 'Suffix Failure Frequency @ T/2 (Exploration Fail)' ranging from 0.00 to 0.35. The y-axis is '|A| : Minimum Action Frequency @ T (Exploration Fail)' ranging from 0.00 to 0.30. Data points are shown for TS (blue circle), PSRL + LLMs (k_sampling = 0.5) (orange square), PSRL + LLMs (k_sampling = 1) (green triangle), PSRL + LLMs (k_sampling = 1.1) (red diamond), and PSRL + LLMs (k_sampling = 1.2) (purple hexagon).](93587f920736a2fdcefeba94b29f302a_img.jpg)

Figure 11: A scatter plot of suffix failure frequency vs. minimum action frequency for Thompson sampling and our LLM-based PSRL with varying k\_sampling. The x-axis is 'Suffix Failure Frequency @ T/2 (Exploration Fail)' ranging from 0.00 to 0.35. The y-axis is '|A| : Minimum Action Frequency @ T (Exploration Fail)' ranging from 0.00 to 0.30. Data points are shown for TS (blue circle), PSRL + LLMs (k\_sampling = 0.5) (orange square), PSRL + LLMs (k\_sampling = 1) (green triangle), PSRL + LLMs (k\_sampling = 1.1) (red diamond), and PSRL + LLMs (k\_sampling = 1.2) (purple hexagon).

Figure 11: A scatter plot of suffix failure frequency vs. minimum action frequency for Thompson sampling and our LLM-based PSRL with varying  $k_{\text{sampling}}$ .

PSRL agent. In preliminary experiments we found that, while this agent is capable of finding success often, it can suffer from issues of prior misspecification, where the true solution (also given in the dataset of Tajwar et al. (2025)) is not within the support of the LLM-generated input prior. To remedy this without giving away the solution, we use a second method of generating an input prior that guarantees it is well-specified; we provide the dataset solution for the sampled customer service issue to GPT-4o and indicate that it is one possible resolution but that GPT-4o must itself assign a probability to it based on how plausible it is perceived to be. We report the results of this latter agent as “well-specified” in Figure 5. All agents were run for a total of 20 trials.

In the face of prior misspecification, something that the base PSRL algorithm does not entertain by assumption and therefore has no explicit mechanism to cope with, baseline LLM agent designs still cannot achieve a statistically significant improvement over PSRL. While theory is not a focus of this work, we simply note in passing that prior misspecification of posterior-sampling methods is a well-studied topic in bandit learning [61], where one can provably expect a graceful degradation in performance commensurate with the degree of misspecification; colloquially, similar results are expected for the full RL setting as discussed, for instance, in the introduction of O’Donoghue (2021). Future work may greatly benefit from expanding on our results to more carefully examine how PSRL can remain robust in the face of such misspecified priors. Moreover, once the prior misspecification is removed (without handing the

![Figure 12: Cumulative regret curves for the real-world customer service bandit task. The plot shows Cumulative Regret (y-axis, 0 to 16) versus Episode (x-axis, 0 to 20). Five methods are compared: PSRL + LLMs (ours) in blue, PSRL + LLMs (ours, well-specified prior) in orange, Reflexion in green, and IORL (p = 1.0) in red. The Reflexion method (green) shows the highest cumulative regret, reaching approximately 15 by episode 20. The PSRL + LLMs (ours) method (blue) shows the lowest cumulative regret, reaching approximately 6 by episode 20. The other methods (orange and red) fall in between, with the well-specified prior version (orange) performing slightly better than the standard PSRL + LLMs (blue).](de2d3e89ee4dd60958b64426cd3a81ca_img.jpg)

Figure 12: Cumulative regret curves for the real-world customer service bandit task. The plot shows Cumulative Regret (y-axis, 0 to 16) versus Episode (x-axis, 0 to 20). Five methods are compared: PSRL + LLMs (ours) in blue, PSRL + LLMs (ours, well-specified prior) in orange, Reflexion in green, and IORL (p = 1.0) in red. The Reflexion method (green) shows the highest cumulative regret, reaching approximately 15 by episode 20. The PSRL + LLMs (ours) method (blue) shows the lowest cumulative regret, reaching approximately 6 by episode 20. The other methods (orange and red) fall in between, with the well-specified prior version (orange) performing slightly better than the standard PSRL + LLMs (blue).

Figure 12: Cumulative regret curves for the real-world customer service bandit task. All LLM agents use GPT-4o.

solution away as the agent must still sift through other plausible sources of customer issues), PSRL is able to demonstrate strong exploration that far exceeds baseline methods on a real-world task with a tremendously large action space.

## C EARLY FAILURES WITH GPT-4O IN RIVERSWIM

### Citations in this section

[33]: (Strehl & Littman, 2008)

As RiverSwim is a stochastic environment, even a limited number of states may still demand a significant episode horizon in order to provide even a chance of learning progress. To keep the financial costs of our RiverSwim experiments down with horizons as small as 6 and as large as 50, we employ a policy caching scheme that capitalizes on the underlying tabular MDP that is RiverSwim. In particular, the policy LLM of *all* LLM agents (ours and baselines) used in each episode only makes one API call per *novel* state visited and the resulting selected action is cached for that state; if a state is ever revisited within the same episode, then this cached action is automatically reused without making an additional policy LLM call. After an episode is completed, this cache is then cleared and reset for the next episode. Notably, as the optimal policy for RiverSwim is non-stationary (since, if the agent is unsuccessful in swimming upstream towards the end of the episode, it is optimal to turn around and collect the smaller downstream reward), this means that the cumulative regret curves across all agents are potentially worse than what they would have been if the agents were allowed to act in a non-stationary fashion. Nevertheless, as there are only two actions in the MDP, we anticipate that the impact of this cost-saving measure on our results is minimal and equitable across all evaluated agents.

In Section 4.2, we reported positive results in a truncated (length-3) variant of the classic RiverSwim environment [33] *upon switching* from GPT-4o to o1-mini as the underlying LLM for our PSRL implementation. For clarity, we use this section to detail the initial failures we encountered with GPT-4o in RiverSwim. Figure 13 shows the associated cumulative regret curves adhering to the same setup as outlined in Section 4.2, except we use  $\kappa_{\pi^*} = \kappa_{\text{posterior}} = 0$  and  $\kappa_{\text{sampling}} = 0.5$ . Despite achieving the best regret curve out of all presented LLM agents in RiverSwim, both of our LLM-based PSRL variants with GPT-4o incur near-linear regret while most instances of classic PSRL are able to achieve optimal behavior.

We also report both vanilla and LLM-based PSRL run with prior distributions where all deterministic RiverSwim transitions (only those where the agent swims downstream) are given as prior knowledge. We posited that supplying all deterministic transitions as prior knowledge would fare better against classic PSRL. While this does allow LLM-based PSRL to exhibit optimal behavior in many trials, far too many still fail as the optimal policy LLM struggles to select optimal actions, even when supplied with posterior samples that have high fidelity to the true environment. Reasons for this include misread transition probabilities (such as swapping numerical values of the input posterior

![Figure 13: Cumulative regret curve for the RiverSwim environment with 3 states. The plot shows Cumulative Regret (y-axis, 0 to 25) versus Episode (x-axis, 0 to 35). The legend lists seven algorithms: PSRL (alpha_0 = 1/3), PSRL (alpha_0 = 1), PSRL (Det. Transitions Prior), PSRL + LLMs, PSRL + LLMs (Det. Transitions Prior), Reflexion, and ICRL (p = 1.0). The PSRL + LLMs (Det. Transitions Prior) algorithm shows the lowest cumulative regret, followed by PSRL (Det. Transitions Prior), PSRL (alpha_0 = 1), PSRL (alpha_0 = 1/3), Reflexion, and ICRL (p = 1.0).](f8630b0582d6e5b1d81f877880ef0dda_img.jpg)

Figure 13: Cumulative regret curve for the RiverSwim environment with 3 states. The plot shows Cumulative Regret (y-axis, 0 to 25) versus Episode (x-axis, 0 to 35). The legend lists seven algorithms: PSRL (alpha\_0 = 1/3), PSRL (alpha\_0 = 1), PSRL (Det. Transitions Prior), PSRL + LLMs, PSRL + LLMs (Det. Transitions Prior), Reflexion, and ICRL (p = 1.0). The PSRL + LLMs (Det. Transitions Prior) algorithm shows the lowest cumulative regret, followed by PSRL (Det. Transitions Prior), PSRL (alpha\_0 = 1), PSRL (alpha\_0 = 1/3), Reflexion, and ICRL (p = 1.0).

Figure 13: Cumulative regret curve for the RiverSwim environment with 3 states. Algorithms with knowledge of all deterministic transitions supplied *a priori* are labeled.

sample) as well as a lack of understanding for long-term planning. Additionally, we observe a rare occurrence where posterior updates can be prone to catastrophically forgetting a single transition, thereby halting learning progress entirely should the omitted transition be essential to reaching the upstream reward.

## D LIMITATION: SCALING UP STOCHASTIC ENVIRONMENTS

### Citations in this section

[62]: (McCoy et al., 2024)
[63]: (Jaech et al., 2024; Guo et al., 2025)

While the success of our LLM-based PSRL in RiverSwim after upgrading to o1-mini from GPT-4o is encouraging, we find that the scalability of such a substitution is short-lived. Recall that our version of RiverSwim used in the preceding section is a truncated variant down to a length-3 river. Unfortunately, as seen in Figure 14, just increasing the river by one additional intermediate state to obtain a length-4 RiverSwim environment ( $H = 20$ ) causes the performance of our LLM-based PSRL to degrade into linear regret.

This negative result underscores a crucial distinction in the choice of epistemic state between agents; that is, the statistical object  $\text{Dirichlet}(0.1, 0.1, 0.1, 0.1, 0.1)$  used by classic PSRL and the natural language string  $\text{Dirichlet}(0.1, 0.1, 0.1, 0.1, 0.1)$  used in LLM-based PSRL. For deterministic transitions in RiverSwim, classic PSRL is able to see eventual concentration to a Dirac delta distribution. Meanwhile the LLM-based PSRL agent, while successful at maintaining visitation counts, is slow to achieve the same convergence and, across many posterior samples, leaves non-negligible probability mass on non-existent transitions with fictitious rewards. One plausible explanation would be that such concentration errors stem from a lack of familiarity by the LLMs, given that Dirichlet distributions with fractional parameters are encountered with less frequency [62]; however, our preliminary experiments with a  $\text{Dirichlet}(1, 1, 1, 1, 1)$  prior showed no significant improvement.

Issues with posterior concentration notwithstanding, we also find that far too many episodes fail as the optimal sample policy LLM struggles to select optimal actions, even when supplied with posterior samples that have high fidelity to the true environment. Even with chain-of-thought prompting, we find a clear lack of understanding for long-term, value-based planning; the preliminary success with length-3 RiverSwim suggests that this failure is connected to the increased verbosity of the epistemic state that, in turn, compromises the optimal sample policy LLM’s ability to account for the value of traversing the full river over collecting the small downstream reward repeatedly. Altogether, while the overall result is negative, we anticipate that these issues may resolve organically in a manner similar to our early challenges with GPT-4o in length-3 RiverSwim; that is, by leveraging a more advanced alternative LLM. Even if recent open-source reasoning models [63]

![Figure 14: Cumulative regret curves for the RiverSwim environments. The plot shows Cumulative Regret (y-axis, 0 to 120) versus Episode (x-axis, 0 to 35). Four curves are plotted: PSRL (Length 3) as a solid blue line, PSRL (Length 4) as a dashed orange line, PSRL + LLMs (Length 3) as a solid green line, and PSRL + LLMs (Length 4) as a dashed red line. The PSRL + LLMs curves show significantly higher cumulative regret compared to the standard PSRL curves.](ed75e80b1e08237f7e90b65357de84d5_img.jpg)

| Episode | PSRL (Length 3) | PSRL (Length 4) | PSRL + LLMs (Length 3) | PSRL + LLMs (Length 4) |
|---------|-----------------|-----------------|------------------------|------------------------|
| 0       | 0               | 0               | 0                      | 0                      |
| 5       | 5               | 15              | 5                      | 25                     |
| 10      | 10              | 30              | 10                     | 50                     |
| 15      | 15              | 40              | 15                     | 75                     |
| 20      | 20              | 45              | 20                     | 90                     |
| 25      | 25              | 48              | 25                     | 105                    |
| 30      | 30              | 48              | 30                     | 120                    |
| 35      | 35              | 48              | 35                     | 135                    |

Figure 14: Cumulative regret curves for the RiverSwim environments. The plot shows Cumulative Regret (y-axis, 0 to 120) versus Episode (x-axis, 0 to 35). Four curves are plotted: PSRL (Length 3) as a solid blue line, PSRL (Length 4) as a dashed orange line, PSRL + LLMs (Length 3) as a solid green line, and PSRL + LLMs (Length 4) as a dashed red line. The PSRL + LLMs curves show significantly higher cumulative regret compared to the standard PSRL curves.

Figure 14: Cumulative regret curves for the RiverSwim environments with 3 (solid lines) and 4 (dashed lines) states, respectively. o1-mini is used exclusively with our LLM-based PSRL.

prove ineffective at fulfilling this purpose, one might still naturally anticipate that such deficiencies will disappear with time assuming future LLM capabilities continue to expand.

## E ADDITIONAL DEEPSEEK-R1 RESULTS

### Citations in this section

[35]: (Guo et al., 2025)
[64]: (Freund & Schapire, 1997)

While our experiments with RiverSwim (Figure 6) confirm the benefits of reasoning models that invest additional computational effort to produce so-called “reasoning” tokens prior to emitting response tokens, models such as o1-mini can be prohibitively expensive. To reduce these financial burdens and assess the efficacy of our proposed LLM-based PSRL with an alternative choice of constituent LLM, we present results for the combination lock (Figure 15 – 20 trials) and Wordle (Figure 8 – 40 trials) environments with DeepSeek-R1 [35].

Our results aggregated across both domains yield two key observations. At the highest level, we observe that R1 provides a performance improvement to all LLM agents (both ours and baselines). Curiously, we find that this performance improvement varies by model and domain; across both environments, we see very small improvements in Reflexion. Meanwhile, performance improvements for ICRL in the combination lock task and our LLM-based PSRL in Wordle are significant. More importantly, we find that the enhanced reasoning capabilities of DeepSeek-R1 applied to our best baseline LLM agents is not sufficient to yield a statistically-significant improvement over our proposed LLM-based PSRL, even when run with a “weaker” or less-capable GPT-4o as the constituent LLM. Such a result is somewhat reminiscent of classic boosting [64], wherein an ensemble of weak learners are composed together into a strong (supervised) learner. Furthermore, these empirical results might (loosely) suggest that the strategic exploration strategy (specifically, Thompson Sampling) forged into the design and structure of the PSRL algorithm offers something beyond what a current strong reasoning model is capable of today, especially when given the freedom in action selections afforded by a LLM agent design like ICRL.

## F LIMITATION: BEYOND THOMPSON SAMPLING

### Citations in this section

[11]: (Lu et al., 2023)
[37]: (Russo & Van Roy, 2018)
[65]: (Russo & Van Roy, 2018; Lu et al., 2023)
[66]: (McAllester & Stratos, 2020)
[67]: (Cover & Thomas, 2012)

While PSRL, through the use of TS, is known to yield a strong exploration strategy, it is by no means perfect. In the bandit literature, shortcomings of TS are well-known and naturally become more salient in the full RL problem [65]. By only executing actions with some probability of being optimal, TS will never take sub-optimal actions that may yield tremendous information gain. Figure 3 already illustrates how a PSRL agent’s uncompromising

![Figure 15: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 7) versus Episode (x-axis, 0 to 8). The legend lists eight methods: PSRL + LLMs (ours; GPT-4o), PSRL + LLMs (ours; R1), Reflexion (GPT-4o), Reflexion (R1), ICRL (p = 1.0; GPT-4o), ICRL (p = 1.0; R1), Bayes-Optimal, and LLM-IDS (ours; GPT-4o). The curves show that PSRL + LLMs (ours; GPT-4o) and LLM-IDS (ours; GPT-4o) achieve the lowest cumulative regret, reaching approximately 6.5 by episode 8. Reflexion (GPT-4o) and Reflexion (R1) follow, reaching around 5.5. ICRL (p = 1.0; GPT-4o) and ICRL (p = 1.0; R1) reach around 4.5. Bayes-Optimal reaches approximately 4.0. PSRL + LLMs (ours; R1) reaches approximately 3.0. The Bayes-Optimal curve is the flattest, indicating it has reached the minimum possible regret.](e9f324c9d7f5305a55cb156855e95b95_img.jpg)

Figure 15: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 7) versus Episode (x-axis, 0 to 8). The legend lists eight methods: PSRL + LLMs (ours; GPT-4o), PSRL + LLMs (ours; R1), Reflexion (GPT-4o), Reflexion (R1), ICRL (p = 1.0; GPT-4o), ICRL (p = 1.0; R1), Bayes-Optimal, and LLM-IDS (ours; GPT-4o). The curves show that PSRL + LLMs (ours; GPT-4o) and LLM-IDS (ours; GPT-4o) achieve the lowest cumulative regret, reaching approximately 6.5 by episode 8. Reflexion (GPT-4o) and Reflexion (R1) follow, reaching around 5.5. ICRL (p = 1.0; GPT-4o) and ICRL (p = 1.0; R1) reach around 4.5. Bayes-Optimal reaches approximately 4.0. PSRL + LLMs (ours; R1) reaches approximately 3.0. The Bayes-Optimal curve is the flattest, indicating it has reached the minimum possible regret.

Figure 15: Cumulative regret curves for the combination lock environment. Labels show the choice of constituent LLM model (GPT-4o or DeepSeek-R1) in each LLM agent.

execution of potentially-optimal policies cripples exploration and solely allows for the testing of two unknown letters at a time.

One remedy is to seek out instantiations of information-directed sampling (IDS) [37]. IDS is an algorithmic design principle that advocates for using a policy which balances between performance shortfall and information gain. While supported by a rigorous corroborating theory in both bandits and RL [11], concrete and practical instantiations of IDS are difficult to come by on account of the challenges surrounding information gain estimation [66]. Moreover, the temporally-delayed consequences absent from bandits but present in RL problems pose an additional challenge as a proper IDS agent must forecast future opportunities for knowledge acquisition several steps into the future when evaluating current actions.

We present an initial design for a IDS agent with LLMs. Our proposed LLM-IDS agent is myopic in that it only takes immediate information gain about optimal behavior at the next timestep into account. Nevertheless, the feedback structure of the combination lock environment allows such an agent to be unconcerned with temporally-delayed information. For a current state  $s_h \in \mathcal{S}$ , we define two  $|\mathcal{A}|$ -dimensional vectors,  $\rho$  and  $\mathcal{I}$ , where  $\rho(a) = \mathbb{E} [V_{\mathcal{M},h}^*(s_h) - Q_{\mathcal{M},h}^*(s_h, a)]$  is the expected regret of taking action  $a \in \mathcal{A}$  in  $s_h$  under the agent’s current posterior and  $\mathcal{I}(a) = \mathbb{I}(\pi^*, R_h, S_{h+1} | A_h = a, S_h = s_h)$  is the information gained (formally, the conditional mutual information [67]) about the optimal policy by taking action  $a$  from state  $s_h$ . IDS calls for sampling an action from the distribution that minimizes the information ratio:  $\min_{\pi \in \Delta(\mathcal{A})} \frac{\mathbb{E}_{a \sim \pi} [\rho(a)]^2}{\mathbb{E}_{a \sim \pi} [\mathcal{I}(a)]}$ . Normally, computation of the  $\rho$  and  $\mathcal{I}$  vectors would be done directly with the current posterior. Instead, we recycle the same posterior update LLM from our LLM-based PSRL but incorporate two new LLMs for the provision of  $\rho$  and  $\mathcal{I}$ ; each of these LLMs is prompted on a per-action basis to assess the expected regret or information gain, respectively, from each action in the current state. With these  $2|\mathcal{A}|$  LLM-generated numerical values, the convex optimization problem of minimizing the information ratio is solved to compute the policy for action selection.

We offer two empirical evaluations to highlight the limitations of LLM-based PSRL exploration inherited from TS while also underscoring the future potential of our LLM-IDS. The first is a contrived but transparent multi-armed bandit problem given as Example 2 of Russo & Van Roy (2018). In this  $(K + 1)$ -armed informative action bandit problem, there is a unique optimal action  $A^* \in [K]$  that yields a deterministic reward of 1 while all other arms yield a reward of 0; additionally, there is an

![Figure 16: Cumulative regret curves for the 11-armed informative action bandit. The plot shows Cumulative Regret (y-axis, 0 to 5) versus Time Period (x-axis, 0 to 10). Two lines are plotted: PSRL + LLMs (blue) and LLM-IDS (orange). The blue line starts at (0,0) and increases steadily, reaching a cumulative regret of approximately 4.5 at time period 10. The orange line remains very close to the x-axis, indicating near-zero cumulative regret.](df0685d2d1176d617ed1e642de4e5425_img.jpg)

Figure 16: Cumulative regret curves for the 11-armed informative action bandit. The plot shows Cumulative Regret (y-axis, 0 to 5) versus Time Period (x-axis, 0 to 10). Two lines are plotted: PSRL + LLMs (blue) and LLM-IDS (orange). The blue line starts at (0,0) and increases steadily, reaching a cumulative regret of approximately 4.5 at time period 10. The orange line remains very close to the x-axis, indicating near-zero cumulative regret.

Figure 16: Cumulative regret curves for the 11-armed informative action bandit (Example 2) of Russo & Van Roy (2018).

![Figure 17: Episodic regret curves for the 11-armed informative action bandit. The plot shows Episodic Regret (y-axis, 0.0 to 1.0) versus Time Period (x-axis, 0 to 10). Two lines are plotted: PSRL + LLMs (blue) and LLM-IDS (orange). The blue line starts at (0,1.0) and decreases, reaching a low episodic regret of approximately 0.15 at time period 10. The orange line starts at (0,1.0) and drops sharply to near 0.0 by time period 2.](45329c7d9aa2bd1290af5b2027f08d7e_img.jpg)

Figure 17: Episodic regret curves for the 11-armed informative action bandit. The plot shows Episodic Regret (y-axis, 0.0 to 1.0) versus Time Period (x-axis, 0 to 10). Two lines are plotted: PSRL + LLMs (blue) and LLM-IDS (orange). The blue line starts at (0,1.0) and decreases, reaching a low episodic regret of approximately 0.15 at time period 10. The orange line starts at (0,1.0) and drops sharply to near 0.0 by time period 2.

Figure 17: Episodic regret curves for the 11-armed informative action bandit (Example 2) of Russo & Van Roy (2018).

![Figure 18: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 8) versus Episode (x-axis, 0 to 8). The legend includes: PSRL + LLMs (ours) (blue), Reflexion (orange), ICPI (green), ICRL (p = 1.0) (red), ICRL (p = 0.5) (purple), ICRL (p = 0.1) (pink), Bayes-Optimal (dotted green), and LLM-IDS (ours) (grey). The blue line (ours) and grey line (LLM-IDS) show the lowest cumulative regret, both staying below 3.5. The other methods show significantly higher regret, with the red line (ICRL p=1.0) reaching the highest regret of approximately 7.5.](b6bd6d8ee5821226bc79251ca5937e07_img.jpg)

Figure 18: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 8) versus Episode (x-axis, 0 to 8). The legend includes: PSRL + LLMs (ours) (blue), Reflexion (orange), ICPI (green), ICRL (p = 1.0) (red), ICRL (p = 0.5) (purple), ICRL (p = 0.1) (pink), Bayes-Optimal (dotted green), and LLM-IDS (ours) (grey). The blue line (ours) and grey line (LLM-IDS) show the lowest cumulative regret, both staying below 3.5. The other methods show significantly higher regret, with the red line (ICRL p=1.0) reaching the highest regret of approximately 7.5.

Figure 18: Cumulative regret curves for the combination lock environment including LLM-IDS.

action 0 that deterministically provides a reward equal to  $(2 \cdot A^*)^{-1}$ . Naturally, an agent willing to deliberately select sub-optimal actions to gain information would take action 0 immediately and then produce optimal behavior thereafter with the identity of  $A^*$  in hand. Figures 16 and 17 show across 10 trials that LLM-IDS succeeds in recovering this optimal exploration strategy exactly for the  $K = 10$  instance whereas LLM-based PSRL is incapable of doing so while exploring via TS. This result also highlights one simple instance of the flexibility that specifying natural-language priors to LLM-based PSRL affords as encoding prior knowledge about the informative action might prove difficult when limited to classic statistical distributions. Extending past this contrived yet transparent bandit example, Figure 18 shows that LLM-IDS is able to outperform LLM-based PSRL in the combination lock task by more quickly testing for unknown digits while remaining unencumbered by known digits already discovered.

## G TOKEN EFFICIENCY

In this section, we give a brief glimpse into the token efficiency of our proposed LLM-based PSRL agent relative to our two strongest baseline LLM agents, Reflexion and ICRL ( $p = 1.0$ ), using

GPT-4o for all constituent LLMs. Notably, our focus in this work has been exclusively on data efficiency through prudent exploration and, as such, no concerted effort has been made in either our proposed agent or baseline agents towards optimizing for token efficiency explicitly (by selecting shorter prompts as inputs to the constituent LLMs) or implicitly (by encouraging LLMs to maintain brevity in their responses). With that said, Figures 19 and 20 illustrate token efficiency of these LLM agents in the combination lock and Wordle environments by plotting cumulative regret as a function of total tokens processed (on average).

![Figure 19: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 7) versus Mean Total Tokens Processed (x-axis, 0 to 40,000). Three curves are shown: PSRL + LLMs (ours) in blue, Reflexion in orange, and ICRL (p = 1.0) in green. The blue curve is the lowest, followed by the orange curve, and then the green curve.](5500ab73cf84ccc0055eecf28889b4db_img.jpg)

Figure 19: Cumulative regret curves for the combination lock environment. The plot shows Cumulative Regret (y-axis, 0 to 7) versus Mean Total Tokens Processed (x-axis, 0 to 40,000). Three curves are shown: PSRL + LLMs (ours) in blue, Reflexion in orange, and ICRL (p = 1.0) in green. The blue curve is the lowest, followed by the orange curve, and then the green curve.

Figure 19: Cumulative regret curves for the combination lock environment as a function of total tokens processed (on average).

![Figure 20: Cumulative regret curves for the Wordle environment. The plot shows Cumulative Regret (y-axis, 0 to 6) versus Mean Total Tokens Processed (x-axis, 0 to 40,000). Three curves are shown: PSRL + LLMs (ours) in blue, Reflexion in orange, and ICRL (p = 1.0) in green. The blue curve is the lowest, followed by the orange curve, and then the green curve.](cb74fd9f5ec715dd3e2e325b864b48bc_img.jpg)

Figure 20: Cumulative regret curves for the Wordle environment. The plot shows Cumulative Regret (y-axis, 0 to 6) versus Mean Total Tokens Processed (x-axis, 0 to 40,000). Three curves are shown: PSRL + LLMs (ours) in blue, Reflexion in orange, and ICRL (p = 1.0) in green. The blue curve is the lowest, followed by the orange curve, and then the green curve.

Figure 20: Cumulative regret curves for the Wordle environment as a function of total tokens processed (on average).

In Figure 19, we see that, despite improved performance and actual convergence towards the optimal policy, our proposed PSRL-LLM consumes more tokens (on average) than Reflexion and ICRL in the combination lock environment. We suspect the primary driver behind the excess tokens comes from the tendency of GPT-4o to fully enumerate all possible correct codes in the "posterior" — something that neither baseline agent does thereby allowing them to be more economical with respect to the total number of tokens processed. Despite that, however, we see that our LLM-based PSRL does achieve better cumulative regret even if truncated to the same number of tokens processed by either baseline agent. In Figure 20, we see that LLM-based PSRL displays token efficiency that is comparable to Reflexion and superior to ICRL in Wordle, eventually able to more consistently identify target words in fewer turns than Reflexion, resulting in lower cumulative regret. Unlike in the combination lock environment, there are far too many possibilities for possible target words and GPT-4o never even attempts to enumerate these candidates, instead opting to maintain information about candidate correct letters and positions.

## H EXPERIMENT PROMPTS

In this section, we outline all LLM prompts used in our experiments. We will present all **system prompts** in orange and all **user prompts** in red. It is important to note that prompts are to LLM agents what typical hyperparameters (entropy regularization coefficient, PPO clip factor, batch size, *etc.*) are to deep RL agents. In that sense, prompt optimization/hyperparameter tuning of baselines is an important facet of evaluation. As is often the case when dealing with vast hyperparameter spaces, however, an exhaustive search for the best hyperparameter settings of each method evaluated would be far too onerous. Thus, while we include our prompts for all agents in our evaluation to foster reproducibility and encourage extensions of our work, we note that future work may find performance improvements with any of these LLM agents through simple refinements of these prompts for particular models and/or downstream applications.

Each LLM used in this work (both for our and baseline agents) was prompted to perform its designated function in the context of a broader agent design/algorithm (PSRL, Reflexion, ICRL, or ICPI). Thus, our prompt iteration process simply consisted of manually adjusting prompts until preliminary experiments showed the desired functionality being achieved. For baseline agents, especially those using ICL, this required few iterations; for some elements of PSRL that involve slightly more complicated entities than a policy; transition function; reward function; or evaluator,

additional iterations were needed to weed out edge cases and tack on further constraints into the initial prompt used in the first iteration. For any given domain, the ability to successfully realize the desired functionality in each of the three LLMs should serve as “unit tests” signaling to an agent designer whether or not it is sensible to run our proposed PSRL agent. More generally, we make no claim that these prompts are optimal in any sense (a claim that likely no LLM agent paper can make in good faith). Investigating these choices in prompt iteration and downstream LLM agent robustness are important areas of future research.

### H.1 LLM-BASED PSRL

In our experiments, depending on the particular environment, we consider two different forms of posterior LLM prompting. For sufficiently short horizons, the posterior LLM is given the entire trajectory in a single prompt and is expected to produce the updated posterior. For longer horizons or whenever concerns about context buffer length come into play, the posterior LLM is prompted with one full  $(s, a, r, s')$  experience tuple at a time and each successive posterior becomes the prior for the subsequent update. Empirically, we find that whole trajectory updates may be more likely to result in erroneous updates where certain pieces of information may be mistakenly updated or forgotten entirely. While this becomes far less likely with per-step experience updates, the associated financial costs and time spent running the PSRL agent scale unfavorably with the horizon of the problem. We use whole trajectory observations for all LLM-based PSRL posterior updates in the RiverSwim, Combination Lock, and Wordle environments. For LLM-based PSRL multi-armed bandit results and LLM-IDS, we use per-step posterior updates.

For whole trajectory posterior updates, the approximate posterior LLM uses the following system prompt and user prompt:

You are a Bayesian posterior distribution for a real-world sequential decision-making problem. Given a current prior belief about the environment and single trajectory observation, you should produce the posterior distribution that accurately reflects knowledge about possibly stochastic environment transitions and environment rewards based on the observed trajectory. A trajectory observation is a sequence of experiences, where each experience consists of a state, action, reward, and next state. Each unit of experience will be separated by XML `<EXPERIENCE>` `</EXPERIENCE>` tags. The posterior distribution must always be complete and describe all sources of uncertainty the agent has about the world. There can be uncertainty about a stochastic transition or reward. The posterior distribution should take into account all information provided in the observed trajectory to update the prior belief about the environment. Be direct and don't show your work. You cannot make any assumptions about the agent and the action selections used to generate the trajectory observation. Never try to model beliefs about the agent. Do not say anything beyond providing the posterior distribution. The agent's interactions with the environment will generate rewards and the posterior distribution should keep track of how any and all rewards are generated. Information and knowledge in the current prior belief about the environment should never be discarded from the posterior distribution. If there is knowledge in the current prior belief about the environment that is unaffected by the trajectory observation, then this knowledge should not be changed and must be repeated exactly in the posterior distribution. Do not say anything to distinguish between old knowledge that is being retained and updated knowledge. The environment was described to the agent like this: `<Environment Description>`

Your current prior is as follows: `<Input prior/LLM-generated posterior>`. A trajectory observation is a sequence of experiences, where each experience consists of a state, action, reward, and next state. Each unit of experience will be separated by XML `<EXPERIENCE>` `</EXPERIENCE>` tags. Here is an observed trajectory: `<Full trajectory>`. Remember that knowledge in the current prior must only be updated but can never be discarded, forgotten, or removed. Do not say anything about which information in the posterior is new and updated or old and remains the same from the prior.

For per-step posterior updates, the approximate posterior LLM uses the following system prompt and user prompt:

You are a Bayesian posterior distribution generator for a real-world sequential decision-making problem. A sequential decision-making problem is represented by an environment that, to each current state and action, produces a next state transition and a reward based on that transition. Transitions and rewards observed from the environment may be stochastic or may be deterministic. Given a current prior belief about the environment and single observation consisting of a next state transition and reward from the environment, you should generate the posterior distribution that accurately reflects knowledge about possibly stochastic environment transitions and environment rewards. The posterior distribution should be a complete and accurate description of all uncertainty the agent has about the world. Information from the prior belief can never be discarded, only updated to be more consistent with the given observation. The posterior distribution should take into account all information provided in the observed next state transition and reward to update the prior belief about the environment. You cannot make any assumptions about the agent and the action selections used to generate the next state transition and reward observation. Never try to model beliefs about the agent. The world may be stochastic and random such that the prior knowledge may need to be updated in the posterior distribution to be consistent with an observed transition or reward. Any knowledge in the prior belief about the environment that is not affected by the observed transition and reward should be retained in full by your posterior distribution. The environment was described to the agent like this: `<Environment Description>`

Your current prior is as follows: `<Input prior/LLM-generated posterior>`. Here is an observed environment transition and reward: `<Single next-state transition and reward>`. Do not say anything about which information in the posterior is new and updated or old and remains the same from the prior. Whenever possible you must maintain exact, numerical probabilities.

The optimal sample policy LLM simply takes the current observation as the user prompt while using the following system prompt:

`<Environment Description>`. Always select optimal actions that maximize value across all future states and all remaining timesteps according to the following hypothesis: `<LLM-generated posterior sample>`. You must select actions that are optimal for and perfectly consistent with the above hypothesis. For each action, you must consider its immediate expect reward as well as the expected value of future states that can be visited by selecting the action. Always select from one of the available actions to take in the environment. Just say the action after "Action: " and nothing else.

As generating a posterior sample requires specifying a full MDP, we find that the posterior sampling LLM in PSRL benefits from having distinct prompts that cater to salient aspects of generating an instance of each environment. We organize the associated environment descriptions as well as posterior sampling system prompts and user prompts by task in the following sub-sections. We also include a sub-section for all prompts used by LLM-IDS.

### H.2 MULTI-ARMED BANDITS

#### H.2.1 BERNOLLI BANDIT

The environment description for the Bernoulli bandit task was given as:

You are an agent interacting with a 5-armed Bernoulli bandit problem. You have exactly 5 actions available labeled as <List of randomly generated letters> and each action has an independent Bernoulli distribution. When you select an action, you will receive a binary reward sampled from the associated Bernoulli distribution.

The posterior sampling LLM system prompts and user prompts were:

You are a generator of Bernoulli bandit problems. A Bernoulli bandit problem is a collection of mean reward values, one for each available action. Knowledge about the reward of each available action will be given to you in the form of a Beta distribution representing beliefs about the mean reward of each arm. This knowledge will constrain the Bernoulli bandit problems you are allowed to generate. For each action, return one plausible hypothesis for the mean reward an agent will observe when taking that action. Each mean reward you return should be consistent with the knowledge you are given about the observed rewards of each action. Each action is independent and so each hypothesis you return for the mean reward of each action will be independent of all others. You must return real, numerical values starting with the phrase "You think " and do not say anything beyond providing the mean rewards of each action. You cannot just return the mean value of the Beta distribution as your guess for the mean reward. You must return a sample from each Beta distribution as your hypothesis. Before you return your mean reward values, describe how each one obeys all constraints and knowledge provided to you. The environment was described to the agent like this: <Environment Description>

Your current knowledge about the mean reward of each action is as follows:<Input prior/LLM-generated posterior>. You must carefully read through this information to generate a Bernoulli bandit problem consistent with this knowledge.

#### H.2.2 CUSTOMER SERVICE BANDIT

The environment description for the customer service bandit was given as:

You are going to role-play as a customer service agent and you have to help a customer resolve their issue. Your goal is to gather enough information to diagnose the problem and provide a correct solution. Your instructions are the following: 1. You may either ask the customer questions or suggest particular actions to the customer. 2. The customer may not be technically inclined, so keep your language simple and clear. 3. Avoid making assumptions — ask specific questions to determine the potential causes. You should guide the customer through basic troubleshooting steps and gather data on the situation. 4. You should try to make the customer satisfied and resolve their problem as quickly as possible. You should also keep your responses short and concise. 5. If the customer mentions a specific product they are using (for example, ABC electronics), then you are the customer support agent for that product/company, i.e., you represent that product or company and have to take appropriate actions without referring the customer to somewhere else. You will receive a reward of 1 if you succeed in resolving the customer's issue and all other rewards are 0. The specific scenario the customer faces is this:<Troubleshooting task sampled from dataset>.

The posterior sampling LLM system prompts and user prompts were:

You are a troubleshooting hypothesis generator for a customer service agent. The initial issue faced by the customer was described to the customer service agent as follows: `<Environment Description>`. The customer service agent is trying to generate hypotheses for what the customer's underlying issue really is. Given all knowledge the agent currently has currently obtained thus far about the customer's issue, you must generate a single plausible hypothesis for what the customer issue is so the agent can correctly provide the solution to the customer. Current knowledge about the customer's underlying issue will be given as a probability distribution listing possible underlying issues and the probability of those issues being accurate for the customer. The hypothesis you generate must be a sample from this distribution. While it is perfectly fine to return a sample that represents the element of the distribution with highest probability, you cannot just return the most likely hypothesis from this distribution simply because it has the highest probability. You must actually sample the distribution to generate your hypothesis. If the probabilities do not sum to 1 to form a valid probability distribution, sample a hypothesis based on what seems plausible using the knowledge available. Be as specific as possible when describing your hypothesis for the customer's issue. You cannot just vaguely state that the customer's item or some component of their item has an issue. You must be more precise than that. When you return your sample for the customer's issue, start with the phrase "You think" and do not say anything beyond providing your hypothesis.

All current knowledge about the customer's issue is as follows: `<Input prior/LLM-generated posterior>`. You must carefully read through this knowledge. Never say anything about the agent or tell the agent what to do.

Additionally, the respective system prompts used for the environment, judge/rewarad function, and prior-generator LLMs are as follows:

You are going to role-play as a customer experiencing a specific issue and interacting with a customer service agent. You must adhere to the following guidelines: 1. Only respond to the agent's questions and provide relevant information when prompted. 2. Your responses should be concise and reflect a typical, non-technical customer's knowledge level. 3. You are going to know what the solution for the specific issue is, but do not give away this information at any point. Your goal is to observe if the customer service agent can ask relevant questions and arrive at the specific issue. 4. When the agent asks you about a potential solution, you should determine what the outcome would be immediately (no asking the agent to wait or hold on) based on your knowledge about the true underlying problem, and let the agent know the result. For example, if your car is not starting because of lack of fuel, and the agent suggests you to check the fuel level in your car, it should fix your issue. Use your reasoning ability to decide whether the suggested solution works, given that the solution to your problem is: `<Solution to sampled dataset issue>`. 5. Reply with 'Goal reached' (and nothing more) if the agent has reached or solved your particular issue. If your issue is solved, you must reply with 'Goal reached'. Your role-play starts now. The particular problem you, as the customer, are facing is: `<Customer service issue sampled from dataset>`, and the solution to this problem is: `<Solution to sampled dataset issue>`. Note that you should never give away or tell the agent what will solve your particular problem, your goal is to see if the agent can come up with the potential solution themselves by asking relevant questions and obtaining accurate information from you.

You are the judge of a customer service agent trying to help a customer in a particular scenario. You will be given the agent's response, and you have to verify whether the agent has been able to successfully solve the problem of the customer correctly. You must adhere to the following guidelines: 1. If the agent provides a solution that is well aligned with the true solution, you must reply with <VALID>. 2. If the agent's proposed solution does not align well with the true solution, then you must reply with <NOTVALID>. 3. The agent doesn't need to match the solution word for word to be considered correct. The agent should be considered correct as long as their solution or question clearly demonstrates that the agent has correctly discovered the source of the customer's issue. 4. Prior to returning your judgement of <VALID> or <NOTVALID> think about the agent solution and true solution and provide a brief justification of why they do or do not align well. The particular scenario the customer is facing is: <Customer service issue sampled from dataset>, and the true solution to their problem is: <Solution to sampled dataset issue>.

You are the generator of a prior distribution for a Bayesian decision-making agent. The agent is faced with a customer service task described as follows: <Customer service issue sampled from dataset>. The agent will be given an broad initial prior as follows: You know that rewards are binary and you will only receive a reward of 1 once the customer's issue has been resolved. If you knew all the relevant details about the source of the customer's issue, there would be no uncertainty about what correct solution to offer and obtain a reward of 1. You think that all common, reasonable issues based on the observations the customer has given are plausible. You think that more common and more realistic issues are more likely than uncommon and less realistic issues. Your job is to provide an additional supplement to this prior that is specific to the issue the customer is facing. Give a probability distribution for the possible underlying issue a customer could be faced along with the probabilities or relative likelihood for each issue you list based on which of them are more or less likely to be the culprit. (The next line is included if the prior is designed to be well-specified.) Be aware that one possible issue could be <Solution to sampled dataset issue> and include it in your prior with a probability the appropriately reflects how plausible it is to be the issue.

#### H.2.3 INFORMATIVE ACTION BANDIT

The environment description for the informative action bandit was given as:

You are an agent interacting with a <Number of actions>-armed bandit problem. You have exactly <Number of actions> actions available labeled by number as <List of action IDs>. When you select an action, you will receive a deterministic reward associated with that selected action.

The posterior sampling LLM system prompts and user prompts were:

You are a generator of a special class of bandit problems. A bandit problem in this class only has a deterministic reward associated with each arm. There is exactly one optimal action which yields a reward of 1. For whichever index the optimal action has, the action with index 0 must produce a reward equal to 1 divided by 2 times the optimal action index or, in other words, the reciprocal of twice the optimal action index. All other actions must produce a reward equal to 0. Knowledge about the optimal action will constrain the instance of this special bandit class that you are allowed to generate. Based on the knowledge of which actions cannot be optimal, choose one of the remaining actions to be optimal uniformly at random. Then, assign deterministic rewards to all of the actions so the bandit problem you generate belongs to the special class exactly as described. You must return real, numerical values for the deterministic action of each action starting with the phrase "You think " and do not say anything beyond providing the reward of each action. Before you return all the special bandit problem reward values, describe how each one obeys all constraints and knowledge provided to you. The environment was described to the agent like this: <Environment Description>

Your current knowledge about the rewards is as follows:<Input prior/LLM-generated posterior>. You must carefully read through this information to generate a bandit problem consistent with this knowledge that must belong to the described special class.

### H.3 RIVERSWIM

The environment description for RiverSwim was given as:

You are an agent swimming in a network of three underwater caves connected by tunnels. Each cave is labeled by its number and always has two tunnels labeled A and B that you can try to swim through. Swimming through tunnels allows you to stochastically move between the caves. There is a strong current in the water which can affect how difficult it is to successfully swim through certain tunnels. Some tunnels may be easier to swim through than others. Successfully swimming through a tunnel once in any cave does not guarantee that it will always be successful. Conversely, failing to swim through a tunnel once does not mean it is impossible and you may have to try again a few times before successfully making it through and swimming into a different cave. Swimming through specific tunnels from certain caves to reach other caves may yield scalar rewards between zero and one.

The posterior sampling LLM system prompts and user prompts were:

You are a map generator for an agent navigating an environment. The environment was described to the agent as follows: <Environment Description>. A map must specify exactly two pieces of information for each possible combination of current cave, tunnel, and next cave. The first piece of information is a transition probability that represents the probability of being in a specific cave, swimming through a particular tunnel, and ending up in a specific next cave. Knowledge about next cave transitions will be provided to you as a collection of Dirichlet distributions. Sampling these distributions will allow you to generate next cave transition probabilities for each cave and tunnel combination. The second piece of information is a deterministic reward that an agent will receive when being in a specific cave, swimming through a particular tunnel, and ending up in a specific next cave. You will be given knowledge about known rewards and rewards that are still unknown and uncertain. If a reward is known, you must repeat its numerical value exactly in the map you generate. If a reward is unknown, knowledge about what it could be will be given to you as a discrete uniform distribution over possible values. You will sample this distribution for each cave, tunnel, and next cave combination and include the concrete, numerical reward value in the map you generate. The input knowledge will constrain the maps you are allowed to generate and the map you generate must be consistent with the input knowledge. Any input knowledge that is known with certainty must be repeated exactly in the map you generate without modification. All transition probabilities and all rewards must be concrete, numerical values. You must sample the distributions you are given and cannot just return the mean value of any input distribution for transition probabilities or rewards. Generate the map using complete sentences starting with the phrase "You think," and do not say anything else. Do not say anything about the input knowledge from the agent including the Dirichlet and uniform distributions.

Current knowledge about the next cave transitions and rewards is as follows: <Input prior/LLM-generated posterior>. You must carefully read through this knowledge. Never say anything about the agent or tell the agent what to do.

### H.4 COMBINATION LOCK

The environment description for CombinationLock was given as:

You are a helpful assistant trying to guess the correct code to a combination lock as quickly as possible. The combination lock requires a three-digit code. You will incrementally construct your guess for the code that unlocks the lock by selecting one digit between 0 and 9 at each timestep. The correct code that opens the lock contains no repeated numbers. For each digit you guess, you will be given feedback indicating if the guessed digit is either in the correct position for the unlocking code, in the wrong position for the unlocking code, or does not appear in the combination lock code at all. You will receive a final reward of one if your guessed code correctly unlocks the combination lock. Otherwise, rewards will always be zero. Your only available actions are the digits from 0 to 9.

The posterior sampling LLM system prompts and user prompts were:

You are a helpful assistant trying to aid an agent in guessing an unknown code that will unlock a lock. Given all knowledge the agent currently has about the correct code, you must generate a single guess at what the correct code could be. You must read through the input information provided by the agent very carefully to produce a good, accurate guess for the correct code. The agent's current knowledge about the correct code establishes specific constraints on what your guess can be. You must generate a guess for the correct code that is consistent with these constraints. Before you return your guess, provide a short justification for each individual digit of your guess that describes how the digit is consistent with the input knowledge from the agent. When you return your guess, start with the phrase "You think " and do not say anything beyond providing your guess for the correct code. The environment was described to the agent like this: <Environment Description>

The agent's current knowledge about the correct code is the following:<Input prior/LLM-generated posterior>. You must carefully read through all information the agent has provided. Never say anything about the agent or tell the agent what decisions to make.

### H.5 WORDLE

The environment description for Wordle was given as:

You are an agent playing a customized version of the game Wordle. There is a five-letter target word from the English dictionary which you must try to guess as quickly as possible. The target word does not contain any repeated letters. You will incrementally construct your guess for this target word by selecting one letter of the alphabet at each timestep. For each letter you guess, you will be given feedback indicating if the guessed letter is either in the correct position for the target word, in the wrong position for the target word, or does not appear in the target word at all. You will receive a reward of one if your guessed word correctly matches the target word. Otherwise, rewards will always be zero. Your only available actions are letters of the alphabet.

The posterior sampling LLM system prompts and user prompts were:

You are a helpful assistant trying to aid an agent in guessing an unknown target word without any repeated letters from the English dictionary. Given all knowledge the agent currently has about the target word, you must generate a single guess at what the target word could be. You must read through the input information provided by the agent very carefully to produce a realistic, plausible guess for the target word. The agent's current knowledge about the target word establishes specific constraints on what your guess can be. You must generate a guess without repeated letters from the English dictionary for the target word that is consistent with these constraints. Before you return your guess, describe how it obeys all constraints and knowledge provided by the agent. When you return your guess from the English dictionary, start with the phrase "You think " and do not say anything beyond providing your guess for the target word. The environment was described to the agent like this: <Environment Description>

The agent's current knowledge about the target word is the following:<Input prior/LLM-generated posterior>. You must carefully read through all information the agent has provided. Never say anything about the agent or tell the agent what decisions to make.

### H.6 LLM-IDS

#### H.6.1 BANDIT VERSION

As the bandit setting does not require handling of temporally delayed consequences or the provision of a current state, it is appropriate to have a separate prompting scheme for LLM-IDS.

The expected regret LLM used the following system prompt and user prompt:

You are a pessimistic expected regret estimator for helping an agent interacting with a multi-armed bandit environment. The bandit environment was described to the agent as follows: <Environment Description>. You will be give the agent's current posterior distribution over the world and will also be given a candidate action. With these two inputs, you must provide a pessimistic estimate of the expected regret an agent will incur by taking the proposed action in the bandit environment. Recall that the regret of an action is the difference in the value or expected reward of the optimal policy and the value of the policy that takes the given action. The expected regret is computed by taking an expectation over the regret using the agent's current posterior distribution. Remember that the optimal policy always selects the optimal action with probability one and so you know that the value of the optimal policy is equal to 1. You must take an expectation with respect to the agent's current posterior distribution to compute expected regret. Your estimate of the expected regret incurred by taking this action in the environment must be pessimistic, which means that it is okay if the estimate you return is larger than the true expected regret but it absolutely cannot be smaller than the true expected regret. Naturally, you are being the most helpful when the expected regret estimate you provide is as close to the true expected regret as possible without going below it. You must produce a real and concrete numerical value as your estimate and say it as a decimal (no fractions) after "Final expected regret: ". Whenever possible, show calculations with concrete numbers before you give your estimate to justify it. Say nothing after "Final expected regret: " other than your estimate.

The agent's posterior distribution reflecting knowledge and uncertainty about the world is as follows: <Input prior/LLM-generated posterior>. Please produce a pessimistic expected regret estimate for the following candidate action: <Candidate action>. If needed, round your answer to no more than three decimal places.

The information gain LLM used the following system prompt and user prompt:

You are a conservative information gain estimator for helping an agent interacting with a multi-armed bandit environment. The bandit environment was described to the agent as follows: <Environment Description>. You will be given the agent's current posterior distribution over the world and will also be given a candidate action. With these two inputs, you must provide a conservative estimate of how much information the agent will gain about the optimal action of the bandit environment by taking the proposed action. Remember that information gain is computed as mutual information or the reduction between prior and posterior entropy, which is measured in bits. Your estimate of the information gained about the optimal action by taking the input candidate action in the bandit environment must be conservative, which means that it is okay if the estimate you return is smaller than the true information gain but it absolutely cannot be larger than the true information gain. Naturally, you are being the most helpful when the information gain estimate you provide is as close to the true information gain about the optimal action as possible without going over it. You must produce a real and concrete numerical value as your estimate and say it as a decimal (no fractions) after "Final information gain: ". Whenever possible, show brief calculations with concrete numbers before you give your estimate to quickly justify it. Say nothing after "Final information gain: " other than your estimate.

The agent's posterior distribution reflecting knowledge and uncertainty about the world is as follows: <Input prior/LLM-generated posterior>. Please produce a conservative information gain estimate (measured in bits) for the following candidate action: <Candidate action>. If needed, round your answer to no more than three decimal places. Remember that sub-optimal or incorrect actions can be informative and information can be gained about the optimal action without actually selecting the optimal action. Also remember that, once the optimal action is known under the agent's posterior distribution, information gain must be equal to 0 for all actions.

#### H.6.2 MDP VERSION

### Citations in this section

[65]: (Russo & Van Roy, 2018; Lu et al., 2023)

As previously mentioned, LLM-IDS retains the approximation posterior LLM for performing posterior updates given agent interactions with the environment. Instead of having two posterior sampling and optimal sample policy LLMs, LLM-IDS employs two LLMs for computing the expected regret and the information gain about optimal behavior, respectively, of each action in a given state. The current posterior is supplied to both LLMs as input along with the current state and the candidate action being evaluation, thereby requiring a total of  $2|A|$  API calls to obtain the two  $|A|$ -dimensional vectors needed to solve the information-ratio optimization problem.

Using the fact that finding the distribution over actions which minimizes the information ratio is a convex optimization problem that places probability mass on at most two actions [65], we solve the optimization problem near-optimally by discretizing the unit interval and searching over all pairs of actions.

For the combination lock environment, we know that the value of the optimal policy is exactly 1. Consequently, we charged the expected regret LLM with simply computing the expected return  $\mathbb{E}[Q^*(s_t, a)]$  and used one minus this output value as the expected regret. The expected regret LLM used the following system prompt and user prompt:

You are a conservative expected optimal action-value function estimator for helping an agent interacting with a sequential decision-making environment. The environment was described to the agent as follows: <Environment Description>. You will be given the agent's current posterior distribution over the world and will also be given a current state and a candidate action. With all of these inputs, you must provide a conservative estimate of the expected cumulative return an agent will observe by taking the proposed action from the current state and then following the optimal policy thereafter. Recall that the optimal-value function (also denoted as  $Q^*$ ) is the value obtained from being in a particular state, taking a particular action, and following the optimal policy thereafter. So, in other words, you are meant to evaluate the expected optimal-value function for the current state and candidate action while taking an expectation with respect to the agent's current posterior distribution. Remember that you are estimating value by taking the candidate action in the current state and then having all future actions selected by the optimal policy. The optimal policy will only make future action selections at future states but will not be able to reverse or change the use of the candidate action in the current state. You must take an expectation with respect to the agent's current posterior distribution to compute the expected optimal action-value function. Your estimate of the expected optimal action-value function must be conservative, which means that it is okay if the estimate you return is smaller than the true expected optimal action-value function but it absolutely cannot be larger than the true expected optimal action-value function. Naturally, you are being the most helpful when the estimate you provide is as close to the true expected optimal action-value function as possible while still being a lower bound and not going over it. You must produce a real and concrete numerical value as your estimate and say it as a decimal (no fractions) after "Final expected optimal action-value: ". Whenever possible, show brief calculations with concrete numbers before you give your estimate to quickly justify it. Say nothing after "Final expected optimal action-value: " other than your estimate.

The agent’s posterior distribution reflecting knowledge and uncertainty about the world is as follows:<Input prior/LIM-generated posterior>. The current state is as follows:<Current state>. Please produce a conservative expected action-value function estimate for the following candidate action:<Candidate action>. If needed, round your answer to no more than three decimal places.

The information gain LLM used the following system prompt and user prompt:

You are a conservative information gain estimator for helping an agent interacting with a sequential decision-making environment. The environment was described to the agent as follows:<Environment Description>. You will be given the agent’s current posterior distribution over the world and will also be given a current state and a candidate action. With all of these inputs, you must provide a conservative estimate of how much information the agent will gain about optimal behavior in the environment by taking the proposed action from the current state. Remember that information gain is computed as mutual information or the reduction between prior and posterior entropy, which is measured in bits. Your estimate of the information gained about optimal behavior by taking this action in the environment must be conservative, which means that it is okay if the estimate you return is smaller than the true information gain but it absolutely cannot be larger than the true information gain. Naturally, you are being the most helpful when the information gain estimate you provide is as close to the true information gain as possible without going over it. You must produce a real and concrete numerical value as your estimate and say it as a decimal (no fractions) after "Final information gain: ". Whenever possible, show brief calculations with concrete numbers before you give your estimate to quickly justify it. Say nothing after "Final information gain: " other than your estimate.

The agent’s posterior distribution reflecting knowledge and uncertainty about the world is as follows:<Input prior/LIM-generated posterior>. The current state is as follows:<Current state>. Please produce a conservative information gain estimate (measured in bits) for the following candidate action:<Candidate action>. If needed, round your answer to no more than three decimal places. Remember that sub-optimal or incorrect actions can be informative and information can be gained about optimal behavior without taking an optimal action.

### H.7 BASELINE PROMPTS

#### H.7.1 IN-CONTEXT REINFORCEMENT LEARNING

The ICRL policy LLM uses the following system prompt and user prompt:

You are a useful assistant who is supposed to select actions within a sequential decision-making environment. Your goal is to maximize expected total reward obtained from the environment through your actions. When given any history of previous interactions and the current state of the world, you will provide a single action to execute in the environment. Choose actions wisely to maximize expected total reward based on your history of previous interactions with the environment. Say nothing besides your choice from the available actions. The task is described as follows: <Environment Description>

The history of interactions you should use to guide your decisions is as follows:<(Potentially sub-sampled) history of past episodes>. The current state of the world is as follows:<Current state>. Please select one of the available actions by saying it directly and without saying anything else.

#### H.7.2 REFLEXION

The Reflexion policy LLM uses the following system prompt and user prompt:

You are the policy for a real-world sequential decision-making problem. The environment representing the decision-making problem is as follows: <Environment Description>. When given a current observation you will choose an action to execute in order to maximize expected cumulative reward. Do not say anything beyond providing a single, valid action. You will also be provided with some guidance and advice which you should use to help you make good action selections.

To help you select actions, you will be given some guidance and advice. Here is your guidance:<(Potentially sub-sampled) history of past reflections>. Please select one action among the available actions to execute from the current observation. Say nothing else besides your choice of action. The current observation is: <Current state>.

The Reflexion self-reflection LLM uses the following system prompt and user prompt:

You are a helpful assistant who is tasked with providing guidance and useful advice to a decision-making agent trying to complete a task by maximizing expected cumulative reward. The environment representing the decision-making problem is described as follows: <Environment Description>. Given a trajectory representing the agent’s behavior unfolding in the environment, provide some guidance and advice to help the agent make better decisions to complete the task. Please be helpful while remaining concise and do not say anything other than the specific advice you think the agent should follow.

A trajectory observation is a sequence of encountered state, action, reward, and next state experiences. Here is an observed trajectory generated by the agent interacting with the environment in an attempt to solve the task:<Full trajectory>.

#### H.7.3 IN-CONTEXT POLICY ITERATION

The ICPI transition function LLM uses the following system prompt and user prompt:

You are the transition function for the simulator of a real-world sequential decision-making problem. The environment you are simulating is: <Environment Description>. When given a current observation and an action the agent has chosen to execute, you will provide a next observation which represents how the world has changed in response to executing the agent’s action. Do not say anything beyond providing the next observation. To help you generate the next observation accurately, you will be provided with examples of observation, action, and next-observation data sampled from the true environment. Use the examples you are given to accurately simulate the environment.

To help you accurately model the environment transition function, you will be given a sequence of observation, action, and next-observation experiences sampled from the true environment. Each unit of experience is separated by XML `<EXPERIENCE>` `</EXPERIENCE>` tags. Here are the transition function experiences: `<Sampled state, action, next-state triples>`. Please generate a next observation for the current observation and current action. The current observation is: `<Current state>`. The current action is: `<Current action>`.

The ICPI reward function LLM uses the following system prompt and user prompt:

You are the reward function for the simulator of a real-world sequential decision-making problem. The environment you are simulating is: `<Environment Description>`. When given a current observation and an action the agent has chosen to execute, you will provide a scalar reward signal conveying the agent’s progression through the task. Do not say anything beyond providing the reward signal. To help you generate the reward accurately, you will be provided with examples of observation, action, and reward data sampled from the true environment. Use the examples you are given to accurately simulate the environment.

To help you accurately model the environment reward function, you will be given a sequence of observation, action, and reward experiences sampled from the true environment. Each unit of experience is separated by XML `<EXPERIENCE>` `</EXPERIENCE>` tags. Here are the reward function experiences: `<Sample state, action, reward triples>`. Please generate a reward for the current observation and current action. The current observation is: `<Current state>`. The current action is: `<Current action>`.

The ICPI rollout policy LLM uses the following system prompt and user prompt:

You are the policy for a real-world sequential decision-making problem. The environment representing the decision-making problem is as follows: `<Environment Description>`. When given a current observation you will choose an action to execute. Do not say anything beyond providing a single, valid action. You should select actions in a manner that is consistent with provided examples of observation and action pairs sampled from the true environment. Be consistent with the examples you are given to behave in the simulated environment.

To help you select actions, you will be given a sequence of observation ad action experiences sampled from the true environment. Each unit of experience is separated by XML `<EXPERIENCE>` `</EXPERIENCE>` tags. Here are the experiences: `<Sampled state-action pairs>`. Please select one action among the available actions to execute from the current observation. The current observation is: `<Current state>`.

## I EXPERIMENT COSTS

In this section, we give *rough* estimates of the total API calls, dollar cost (according to current GPT-4o pricing), and average as well as maximum tokens used in our main evaluation domains.

Starting with API calls, we recall that we consider a finite-horizon MDP with  $K$  episodes, each with a horizon of  $H$ . At the start of each episode, our LLM-based PSRL makes one API call to draw a "posterior" sample. At each timestep of the episode, there are exactly  $H$  API calls made by the optimal sample policy LLM. Finally, at the end of the episode, there is exactly one API call made to perform the posterior update. All together, this yields a total of  $K(H + 2)$  API calls.

Under current GPT-4o pricing, the total cost of a single trial in each of our evaluation domains is as follows:

| Domain                   | Number of Episodes ( $K$ ) | Single-Trial Dollar Cost |
|--------------------------|----------------------------|--------------------------|
| 5-Armed Bernoulli Bandit | 100                        | \$1                      |
| Combination Lock         | 8                          | \$0.11                   |
| Wordle                   | 5                          | \$0.11                   |
| RiverSwim                | 35                         | \$0.90                   |

For o1-mini in RiverSwim, the single trial cost increases to \$7.50.

The average and maximum token counts per-LLM are as follows:

| Posterior Sampling LLM   |                |                |
|--------------------------|----------------|----------------|
| Domain                   | Average Tokens | Maximum Tokens |
| 5-Armed Bernoulli Bandit | 1000           | 1500           |
| Combination Lock         | 700            | 800            |
| Wordle                   | 800            | 1000           |
| RiverSwim                | 1500           | 1700           |

| Optimal Sample Policy LLM |                |                |
|---------------------------|----------------|----------------|
| Domain                    | Average Tokens | Maximum Tokens |
| 5-Armed Bernoulli Bandit  | 400            | 500            |
| Combination Lock          | 400            | 600            |
| Wordle                    | 450            | 650            |
| RiverSwim                 | 1000           | 1400           |

| Posterior Update LLM     |                |                |
|--------------------------|----------------|----------------|
| Domain                   | Average Tokens | Maximum Tokens |
| 5-Armed Bernoulli Bandit | 900            | 1100           |
| Combination Lock         | 1200           | 1400           |
| Wordle                   | 1500           | 1700           |
| RiverSwim                | 1700           | 1900           |

| Per-Episode Tokens       |              |               |              |
|--------------------------|--------------|---------------|--------------|
| Domain                   | Input Tokens | Output Tokens | Total Tokens |
| 5-Armed Bernoulli Bandit | 1500         | 800           | 2300         |
| Combination Lock         | 4000         | 1100          | 5100         |
| Wordle                   | 3700         | 850           | 4550         |
| RiverSwim                | 4700         | 1500          | 6200         |